# ComPartMENT 机制实验：DeBERTa-v2-xlarge 的组合性预测管线

**任务**：给定一个多词表达（MWE，如 `account book` / `crack down`）和一句真实语料，
输出该表达**在这条语境里**的组合性分数（0–5 连续值）。

**这份 notebook 交付什么**：一套**机制实验**的训练管线，主干是
`DeBERTa-v2-xlarge`（`microsoft/deberta-v2-xlarge`）。
**v4 的定位**：它不再是一条「基线」，而是**验证一个机制**的实验 ——
本文件验证的机制是 **软标签对比学习（表征的相似度对齐标签的相似度）**（见 Step4 / Step7 的说明）。

**跑什么**：打开 notebook 一路执行到底，它会**自动完成 5 次训练**——
四个子集各单独训练一次（`en-nn` / `de-nn` / `en-pv` / `de-pv`），
最后再把四个子集拼起来联合训练一次。
**不需要手动切换任何开关**，结果按「运行 × 评测设置」汇总成表、画成图。

**与另一个机制实验（DeBERTa / Qwen）的关系**：两份 notebook **共用同一份数据处理、
同一套超参、同一套指标、同一套绘图**，只切换 `MODEL_KEY` 一个变量。
v4 里两者**故意**在「读出方式 / 机制项」上不同（各自验证不同机制），
但所有**非机制项**逐字相同 ⇒ 「v3 → v4」的差异能干净地归因到机制本身。

---

## 五个部分

| 部分 | 对应章节 | 内容 |
|---|---|---|
| 问题与背景 | 本节 + Step1 | 三项官方指标各自的含义，指标 2 为什么是硬门槛 |
| 原理与设计 | Step3 / Step4 | 输入怎么构造、目标怎么定位、主干怎么接回归头、损失怎么写 |
| 实现代码 | Step1–Step8 | 全部可执行代码，每段都带「为什么」 |
| 结果分析 | Step9 | **评估标准**（模型在 6 个评测设置上的三项指标）与**参考对照**（随机初始化 / type-level oracle）分区呈现 |
| 结论与延伸 | 文末 | 运行前置条件、显存降级顺序、判断实验有没有做对的三个指标 |

---

## 全局数据流（谁只读、谁被更新）

```
train/{subset}-train.tsv
        │  ① 载入：删 ContextID + 物理打乱 + 去重            （Step2）
        ▼
   DataFrame
        │  ② 定位 MWE → 标记上下文 → 拼输入模板              （Step3）
        ▼
   {"text", "labels[2]", "loss_mask[2]", "group_index"}
        │  ③ 分词（动态 padding）→ 目标级互斥切分             （Step3）
        ▼
   datasets.Dataset ──Collator──► batch{input_ids, attention_mask, labels, group_index}
        │
        ▼
   AutoModel 主干（只读参数，全部更新）
        │  ④ 池化 → 线性回归头 → sigmoid×5                   （Step4）
        ▼
   logits ∈ R^{B×2}
        │  ⑤ L = Huber(掩码加权) + λ·组内排序 + μ·组内std     （Step7）
        ▼
   反向传播 → 更新主干与回归头
        │
        ▼
   评估：指标1 ρ / 指标2 ρ / 指标3 RMSE                     （Step5, Step9）
```

- **只读**：数据文件、标签列。
- **被更新**：主干全部参数 + 回归头。
- **不参与训练**：目标字符串只作为 `group_index`（分组键）与输入里的 MWE 文本，
  **不作为任何数值特征**；`ContextID` 在载入时就被删除，防止行序泄漏。

## Step1 导入包

**这一节要解决什么**：把「会随机器、随人变」的东西全部钉死在一个地方——
模型下载端点、随机种子、设备、精度、序列长度、批大小、模型 id、学习率、运行计划。
其余 8 个 Step 一律不写死任何常量，只从这里取。

**为什么必须这样**：本项目要跑 2 个模型 × 4 个子集 × 后续若干组消融。
如果 `lr=2e-5` 写在 Step6、`max_length=160` 写在 Step3，
那么「换个模型跑一遍」就要改 6 个地方，改漏一处就会得到一张**无法归因**的对照表。
对照实验的可信度 100% 取决于配置是否只有一个入口。

**先说一件本项目特有的事：模型权重从哪儿下。**

云端 GPU（Kaggle / Colab / 阿里云等海外实例）**可以直连 `huggingface.co`**，
本机直连通常不通。所以默认策略是：

1. **先探测官方源**（`https://huggingface.co`）是否真的可达——用一次真实的 HTTPS 请求，
   不是 ping，也不是 socket 探测（很多网络环境 443 端口能连但 HTTP 被重置）；
2. 官方源可达 → 用官方；不可达 → 换镜像站 `https://hf-mirror.com`；
3. **下载时再兜一层**：即便探测时通了、下载时断了，也会自动换另一个端点重试。

探测与端点切换都发生在 `import transformers` **之前**或**显式传参**，
不依赖 `HF_ENDPOINT` 全局变量的缓存时机——这是「我明明改了镜像怎么还在走官方」
这类现象最常见的来源。

In [ ]:





import os
import urllib.request





os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")




























_GPU_INDEX = int(os.environ.get("COMPARTMENT_GPU", "0"))
os.environ["CUDA_VISIBLE_DEVICES"] = str(_GPU_INDEX)
print(f"[环境] CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']} "
      f"（本进程只看见这 1 张卡 → Trainer 不会套 nn.DataParallel → 梯度显存不翻倍）")

OFFICIAL_ENDPOINT = "https://huggingface.co"
MIRROR_ENDPOINT = "https://hf-mirror.com"






HF_MODE = os.environ.get("COMPARTMENT_HF_MODE", "auto").strip().lower()


def probe_http(url, timeout=8):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "compartment-probe"})
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return 200 <= resp.status < 500
    except Exception:
        return False


def choose_endpoints(model_id):
    if HF_MODE == "cache":
        print("端点策略: cache（只用本地缓存，不联网）")
        return []
    if HF_MODE == "mirror":
        print("端点策略: mirror（强制镜像站）")
        return [MIRROR_ENDPOINT, OFFICIAL_ENDPOINT]
    if HF_MODE == "official":
        print("端点策略: official（强制官方源）")
        return [OFFICIAL_ENDPOINT, MIRROR_ENDPOINT]
    api = f"{OFFICIAL_ENDPOINT}/api/models/{model_id}"
    ok = probe_http(api)
    print(f"官方源连通性探测: {'通' if ok else '不通'}  ({api})")
    if ok:
        print("→ 使用官方 huggingface.co（云端 GPU 直连即走这条）")
        return [OFFICIAL_ENDPOINT, MIRROR_ENDPOINT]
    print("→ 官方源不可达，改用镜像站 hf-mirror.com")
    return [MIRROR_ENDPOINT, OFFICIAL_ENDPOINT]


os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

In [ ]:

import gc
import inspect
import json
import math
import platform
import re
import shutil
import time
import unicodedata
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats
from scipy.stats import ConstantInputWarning

import matplotlib.pyplot as plt
from matplotlib import font_manager

from datasets import Dataset
from transformers import (
    AutoConfig,
    AutoModel,
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.modeling_outputs import ModelOutput
from torch.utils.data import Sampler

print("torch", torch.__version__)

In [ ]:












_FONT_CANDIDATES = [
    "Helvetica Neue", "Helvetica", "Arial", "Arial Unicode MS",
    "Liberation Sans", "DejaVu Sans", "Verdana", "Tahoma",
]


def _pick_latin_font():
    try:
        installed = {f.name for f in font_manager.fontManager.ttflist}
    except Exception:
        installed = set()
    for _name in _FONT_CANDIDATES:
        if _name in installed:
            return _name
    return "DejaVu Sans"


_CHOSEN_FONT = _pick_latin_font()
plt.rcParams["font.sans-serif"] = [_CHOSEN_FONT] + [f for f in _FONT_CANDIDATES if f != _CHOSEN_FONT]
plt.rcParams["axes.unicode_minus"] = False
print("绘图字体:", _CHOSEN_FONT, "| 图注语言: en (English only)")


PLOT_LANG = "en"

_L10N = {
    "zh": {
        "gold": "人工评分", "pred": "模型预测",
        "gold_std": "人工评分的组内标准差（金标准）", "pred_std": "模型预测的组内标准差",
        "m1": "指标1_ρ", "m2": "指标2_ρ", "m3": "指标3_RMSE",
        "rho_up": "Spearman ρ（越大越好）", "rmse_down": "RMSE（越小越好）",
        "figA": "图A  预测 vs 真值", "figB": "图B  评估标准", "figC": "图C  指标2 的可视化",
        "figD": "图D  五次运行的合计指标对照", "figE": "图E  参考对照",
        "not_standard": "非评估标准", "trained": "训练后模型", "random_init": "随机初始化",
        "oracle": "type-level oracle", "single": "单子集训练", "joint": "联合训练（覆盖全部设置）",
        "joint_short": "联合\n(四子集)", "single_short": "单子集\n{short}",
        "m2rho": "指标2 ρ =", "n_targets": "目标数 =",
        "const_std": "预测 std 恒为 0 的目标数 =", "n_settings": "个评测设置",
        "gold_axis": "横轴金标准、纵轴系统输出", "eval_std": "评估标准",
        "short": {},
    },
    "en": {
        "gold": "Gold score", "pred": "Prediction",
        "gold_std": "Gold within-target std", "pred_std": "Predicted within-target std",
        "m1": "Metric1 rho", "m2": "Metric2 rho", "m3": "Metric3 RMSE",
        "rho_up": "Spearman rho (higher is better)", "rmse_down": "RMSE (lower is better)",
        "figA": "Fig A  Prediction vs Gold", "figB": "Fig B  Evaluation standard",
        "figC": "Fig C  Metric 2", "figD": "Fig D  Aggregate metrics across runs",
        "figE": "Fig E  Reference systems",
        "figF": "Fig F  Training loss curves",
        "loss": "Loss", "step": "Step", "epoch": "Epoch",
        "train_loss": "Train loss", "eval_loss": "Eval loss",
        "loss_all_runs": "Training loss of every run",
        "loss_main": "Train vs eval loss (joint run)",
        "untrained": "untrained", "overfit_gap": "train - eval gap at last epoch",
        "not_standard": "reference only", "trained": "Trained", "random_init": "Random init",
        "oracle": "type-level oracle", "single": "Single-subset", "joint": "Joint (all settings)",
        "joint_short": "Joint\n(4 subsets)", "single_short": "single\n{short}",
        "m2rho": "Metric2 rho =", "n_targets": "targets =",
        "const_std": "targets with constant pred std =", "n_settings": "settings",
        "gold_axis": "gold (x) vs system (y)", "eval_std": "Evaluation standard",
        "short": {"英·复合词": "EN-NN", "德·复合词": "DE-NN",
                  "英·小品词": "EN-PV", "德·小品词": "DE-PV"},
    },
}


def T(key, **kw):
    v = _L10N[PLOT_LANG].get(key, key)
    return v.format(**kw) if kw else v


def short_of(subset):
    zh = SUBSET_SPECS[subset]["short"]
    return _L10N[PLOT_LANG]["short"].get(zh, zh)

In [ ]:

SEED = 42








MODEL_REGISTRY = {
    "deberta": dict(
        model_id="microsoft/deberta-v2-xlarge",
        pooling="mean",
        attn_impls=("eager",),
        head_mode="regression",
        mech="softcl",
    ),
    "qwen": dict(
        model_id="Qwen/Qwen2.5-1.5B-Instruct",
        pooling="last",
        attn_impls=("sdpa", "eager"),
        head_mode="lm_head",
        mech="lm_head",
    ),
}



MODEL_KEY = "deberta"


HEAD_MODE = MODEL_REGISTRY[MODEL_KEY]["head_mode"]
MECH_NAME = MODEL_REGISTRY[MODEL_KEY]["mech"]
print(f"读出方式 head_mode = {HEAD_MODE} | 本版机制 = {MECH_NAME}")






RUN_PLAN = [
    ("single", "en-nn"),
    ("single", "de-nn"),
    ("single", "en-pv"),
    ("single", "de-pv"),
    ("joint", None),
]


SUBSET_SPECS = {
    "en-nn": dict(key="Compound",     cols=["ModAvg", "HeadAvg"], lang="en", task="nn", short="英·复合词"),
    "de-nn": dict(key="Compound",     cols=["ModAvg", "HeadAvg"], lang="de", task="nn", short="德·复合词"),
    "en-pv": dict(key="ParticleVerb", cols=["Avg"],             lang="en", task="pv", short="英·小品词"),
    "de-pv": dict(key="ParticleVerb", cols=["Avg"],             lang="de", task="pv", short="德·小品词"),
}
SUBSET_ORDER = ["en-nn", "de-nn", "en-pv", "de-pv"]





N_OUTPUTS = 2


MAX_LENGTH   = 160






























EFFECTIVE_BATCH_BY_MODEL = {"deberta": 16, "qwen": 8}
TARGET_EFFECTIVE_BATCH = EFFECTIVE_BATCH_BY_MODEL[MODEL_KEY]
BATCH_SIZE   = 8


GRAD_ACCUM   = max(1, int(round(TARGET_EFFECTIVE_BATCH / BATCH_SIZE)))
PROBE_MEMORY = True
LR           = 2e-5














EPOCHS       = 3
JOINT_EPOCHS = 3
VAL_RATIO    = 0.2
WEIGHT_DECAY = 0.01











WARMUP_RATIO = 0.1









HUBER_DELTA  = 1.0



















USE_8BIT_OPT   = "auto"
USE_GRAD_CKPT  = False


























FIT_MODE       = "t3"
FIT_REF_PARAMS = 1.55e9
FIT_SAFETY     = 0.90


















FIT_PROBE_RESERVE_GB = 0.8




















GPU_INDEX      = int(os.environ.get("COMPARTMENT_GPU", "0"))


TIME_BUDGET_H = 11.0































































LOSS_PROFILE_BY_MODEL = {

    "deberta": {


























        "en-nn": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.00, kappa=0.15, softcl=0.00, ce=0.00),
        "de-nn": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.00, kappa=0.15, softcl=0.00, ce=0.00),
        "en-pv": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.06, kappa=0.50, softcl=0.06, ce=0.00),
        "de-pv": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.06, kappa=0.50, softcl=0.06, ce=0.00),
    },

    "qwen": {














        "en-nn": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.00, kappa=0.15, softcl=0.00, ce=0.10),
        "de-nn": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.00, kappa=0.15, softcl=0.00, ce=0.10),
        "en-pv": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.06, kappa=0.50, softcl=0.00, ce=0.10),
        "de-pv": dict(order=0.08, mse=0.05, guard=0.06, xrank=0.06, kappa=0.50, softcl=0.00, ce=0.10),
    },
}

SUBSET_LOSS_PROFILE = LOSS_PROFILE_BY_MODEL[MODEL_KEY]










AUTO_BALANCE = True
ORDER_TAU    = 1.0
XRANK_MARGIN = 0.05
XRANK_TAU    = 2.0
BALANCE_EMA_BETA  = 0.99
















SOFTCL_LABEL_SPAN = 5.0
SOFTCL_PROJ_DIM  = 128




CE_SOFT_SIGMA = 0.8







READOUT_CUES = ["\nModifier: ", "\nHead: "]
DIGIT_VALUES = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]






LOSS_FINITE_CHECK_EVERY = 50



























BEST_METRIC_KEY = "合计|_综合分"














Z_M1 = 0.023
Z_M2 = 0.055
Z_M3 = 0.046


GROUPED_BATCHING = True
SMOKE = False
SMOKE_N_ROWS = 400





SAVE_FINAL_FOR = {"joint"}





if SMOKE:
    MAX_LENGTH = 64
    EPOCHS = 1
    JOINT_EPOCHS = 1
    BATCH_SIZE = 4






    RUN_PLAN = [("single", "en-nn"), ("single", "de-nn"), ("joint", None)]

print(json.dumps(dict(MODEL_KEY=MODEL_KEY, HEAD_MODE=HEAD_MODE, MECH_NAME=MECH_NAME,
                      MAX_LENGTH=MAX_LENGTH, BATCH_SIZE=BATCH_SIZE, GRAD_ACCUM=GRAD_ACCUM,
                      TARGET_EFFECTIVE_BATCH=TARGET_EFFECTIVE_BATCH,
                      LR=LR, EPOCHS=EPOCHS, JOINT_EPOCHS=JOINT_EPOCHS,
                      WARMUP_RATIO=WARMUP_RATIO, HUBER_DELTA=HUBER_DELTA,
                      SUBSET_LOSS_PROFILE=SUBSET_LOSS_PROFILE, AUTO_BALANCE=AUTO_BALANCE,
                      ORDER_TAU=ORDER_TAU, XRANK_MARGIN=XRANK_MARGIN, XRANK_TAU=XRANK_TAU,
                      BALANCE_EMA_BETA=BALANCE_EMA_BETA,
                      SOFTCL_LABEL_SPAN=SOFTCL_LABEL_SPAN,
                      SOFTCL_PROJ_DIM=SOFTCL_PROJ_DIM, CE_SOFT_SIGMA=CE_SOFT_SIGMA,
                      READOUT_CUES=READOUT_CUES,
                      Z_M1=Z_M1, Z_M2=Z_M2, Z_M3=Z_M3,
                      LOSS_FINITE_CHECK_EVERY=LOSS_FINITE_CHECK_EVERY,
                      BEST_METRIC_KEY=BEST_METRIC_KEY,
                      SMOKE=SMOKE, RUN_PLAN=[list(x) for x in RUN_PLAN]),
                 ensure_ascii=False, indent=1))

In [ ]:

def detect_device():
    if torch.cuda.is_available():
        _n_gpu = torch.cuda.device_count()
        _gi = GPU_INDEX % max(1, _n_gpu)




        torch.cuda.set_device(_gi)
        _cap = torch.cuda.get_device_capability(_gi)
        _native_bf16 = (_cap[0] >= 8) and torch.cuda.is_bf16_supported()
        _fp16 = (not _native_bf16) and (_cap[0] >= 7)
        return "cuda", _native_bf16, _fp16, True
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():

        return "mps", False, False, False
    return "cpu", False, False, False


DEVICE, USE_BF16, USE_FP16, USE_TF32 = detect_device()


HW_BF16, HW_FP16 = USE_BF16, USE_FP16




NUM_WORKERS = 0 if platform.system() == "Darwin" else max(1, (os.cpu_count() or 2) // 2)
DATALOADER_PERSISTENT = NUM_WORKERS > 0


if USE_TF32:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

set_seed(SEED)
print({
    "device": DEVICE, "bf16": USE_BF16, "fp16": USE_FP16, "tf32": USE_TF32,
    "workers": NUM_WORKERS, "torch": torch.__version__, "platform": platform.system(),
    "python": platform.python_version(),
})



if torch.cuda.is_available():
    for _i in range(torch.cuda.device_count()):
        _p = torch.cuda.get_device_properties(_i)
        print(f"GPU{_i}: {_p.name} | {_p.total_memory / 1024 ** 3:.1f} GiB | "
              f"compute capability {_p.major}.{_p.minor} | "
              f"原生bf16={_p.major >= 8} | fp16={_p.major >= 7}")
    _total_cards = sum(torch.cuda.get_device_properties(_i).total_memory
                        for _i in range(torch.cuda.device_count())) / (1024 ** 3)
    print(f"物理卡：{torch.cuda.device_count()} 张可见（合计 {_total_cards:.1f} GiB）；"
          f"本 notebook 绑定物理 GPU{GPU_INDEX}，在进程内编号为 cuda:0")
    print("→ 单卡模式：Trainer 看到 n_gpu=1 → **不会**自动套 nn.DataParallel"
          "（DataParallel 的反向传播会额外复制一整份梯度，Qwen-1.5B 要多花 5.75 GiB）")

## Step2 加载数据集

**这一节要解决什么**：用最少的假设把原始 TSV 变成规整的 DataFrame，
并且**在载入这一步就把会泄漏答案的东西扔掉**。

**为什么载入阶段就要处理反泄漏**：本项目实测发现，
同一个 MWE 的各条例句在文件里是**按 ContextID 顺序连续排列**的，
而 ContextID 又与该目标的平均分高度相关（组内 Spearman 达 0.985）。
换句话说，**「这条例句在文件里的位置」本身就带着答案**。
如果先建特征再打乱，任何从行号派生的东西（下标、位次、窗口邻域）都会变成作弊通道。
所以顺序必须是：**先删 ContextID → 再物理打乱**。

**要保留什么**：`Compound` / `ParticleVerb`（MWE 本身）、`Mod`/`Head`/`Base`/`Particle`（成分）、
`ModAvg`/`HeadAvg`/`Avg`（标签）、`Context`（句子）。

In [ ]:






CLOUD_TRAIN_DIR = "/kaggle/input/datasets/kintsugi0v0/compartment-data/train"


def resolve_data_root(start: Path) -> Path:
    cands = []
    env = os.environ.get("COMPARTMENT_TRAIN_DIR")
    if env:
        cands.append(Path(env).expanduser())
    cands.append(Path(CLOUD_TRAIN_DIR))

    root_env = os.environ.get("COMPARTMENT_ROOT")
    if root_env:
        cands.append(Path(root_env).expanduser() / "data" / "train")
        cands.append(Path(root_env).expanduser() / "train")
    p = start.resolve()
    for cand in [p] + list(p.parents):
        for rel in ("data/train", "train"):
            if (cand / rel / "en-nn-train.tsv").exists():
                cands.append(cand / rel)
                break
        else:
            continue
        break
    for c in cands:
        if (c / "en-nn-train.tsv").exists():
            return c
    raise FileNotFoundError(
        "找不到 en-nn-train.tsv。已依次尝试：\n  " + "\n  ".join(str(c) for c in cands)
        + "\n请设置环境变量 COMPARTMENT_TRAIN_DIR 指向含 4 个 *-train.tsv 的目录。")


WORK_DIR = Path.cwd()
DATA_ROOT = resolve_data_root(WORK_DIR)

OUTPUT_ROOT = Path(os.environ.get("COMPARTMENT_OUTPUT", str(WORK_DIR / "outputs")))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("WORK_DIR  =", WORK_DIR)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)

In [ ]:



_demo_name = SUBSET_ORDER[0]
_demo_spec = SUBSET_SPECS[_demo_name]
raw = pd.read_csv(DATA_ROOT / f"{_demo_name}-train.tsv", sep="\t")
print(f"{_demo_name} 原始形状: {raw.shape}")
print("列名:", list(raw.columns))
print(raw[[_demo_spec["key"]] + _demo_spec["cols"] + ["Context"]].head(3).to_string()[:700])


print("\n--- 标签分布 ---")
for c in _demo_spec["cols"]:
    s = raw[c]
    print(f"{c}: 均值 {s.mean():.3f} | 标准差 {s.std():.3f} | 最小 {s.min():.2f} | 最大 {s.max():.2f}")
wlen = raw["Context"].str.split().str.len()
print("\n--- 句子长度（词）分位 ---")
print(dict(zip(["p50", "p90", "p95", "p99", "max"],
                np.percentile(wlen, [50, 90, 95, 99, 100]).round(0).tolist())))
print("→ 句子很短，MAX_LENGTH=160 足够，不会因为截断丢掉关键信息。")

In [ ]:

def load_subset(name, seed=SEED):
    sp = SUBSET_SPECS[name]
    df = pd.read_csv(DATA_ROOT / f"{name}-train.tsv", sep="\t")
    df = df.drop_duplicates(subset=[sp["key"], "Context"]).reset_index(drop=True)
    if "ContextID" in df.columns:
        df = df.drop(columns=["ContextID"])
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    df["y0"] = df[sp["cols"][0]].astype(float)
    df["m0"] = 1.0
    if len(sp["cols"]) > 1:
        df["y1"] = df[sp["cols"][1]].astype(float)
        df["m1"] = 1.0
    else:
        df["y1"] = 0.0
        df["m1"] = 0.0
    if SMOKE and SMOKE_N_ROWS:



        keep = df[sp["key"]].drop_duplicates().head(max(1, SMOKE_N_ROWS // 5))
        df = df[df[sp["key"]].isin(set(keep))].reset_index(drop=True)
    return df


for _n in SUBSET_ORDER:
    _d = load_subset(_n)
    _sp = SUBSET_SPECS[_n]
    print(f"[{_n}] 形状 {_d.shape} | 目标数 {_d[_sp['key']].nunique()} | "
          f"每目标例句数中位 {_d.groupby(_sp['key']).size().median()} | "
          f"ContextID 是否还在: {'ContextID' in _d.columns}")

## Step3 数据预处理

这一节做四件事，每件都有明确的动机：

| 环节 | 做什么 | 为什么 |
|---|---|---|
| ① 定位 MWE | 在句子里标出 MWE 出现的那个 span | 任务问的是「这个 MWE 在**这句**里是不是被当成整体理解」。不告诉模型句中哪个 span 对应 MWE，它只能靠整句语义猜 |
| ② 拼输入模板 | `MWE: <mwe> \| Context: <ctx>` | 两个模型必须看到**逐字相同**的输入，否则「模型差异」里会混进「模板差异」 |
| ③ 目标级互斥切分 | 按 MWE 切 train/val，并复刻官方「先分箱后分层抽样」 | 按行切会把同一 MWE 的例句分到两边，模型等于「见过这个 MWE」再去预测它，指标被高估 |
| ④ 分词 + 动态 padding | 批量分词，batch 内补齐到 8 的倍数 | 定长补齐到 `max_length` 会把每条的注意力开销抬到 O(L²) 的上限，动态补齐只按本批最长序列付费 |

**定位为什么难**：本项目实测过定位率——
英语复合词 98.6%、德语复合词 95.9% 可以字面命中；
但英语小品词动词只有 33.6%（`crack down` 在句子里常写成 `cracking down`），
德语更只有 20.8%（德语可分动词写作 `abgehauen`，粒子粘在动词前，且中间插前缀 `ge-`）。
所以定位必须做**分级级联**，而不是一次精确匹配。

In [ ]:






def fold(s, ascii_mode=False):
    s = unicodedata.normalize("NFKD", str(s).lower()).replace("ß", "ss")
    s = re.sub(r"[^a-z0-9äöüß']", "", s)
    if ascii_mode:
        for a, b in [("ä", "ae"), ("ö", "oe"), ("ü", "ue")]:
            s = s.replace(a, b)
    return s


def stem_de(b):
    for suf in ["ern", "eln", "en", "er", "e", "t", "ten", "n"]:
        if b.endswith(suf) and len(b) - len(suf) >= 4:
            return b[:-len(suf)]
    return b


def stem_en(w):
    for suf in ["ing", "edly", "ed", "es", "s", "y", "e"]:
        if w.endswith(suf) and len(w) - len(suf) >= 3:
            return w[:-len(suf)]
    return w


def stem_keys(base, lang):
    ks = set()
    for v in (fold(base), fold(base, True)):
        st = stem_de(v) if lang == "de" else stem_en(v)
        for L in (5, 4, 3):
            if len(st) >= L:
                ks.add(st[:L])
    return ks


WORD_RE = re.compile(r"[a-zA-ZäöüÄÖÜß]+(?:[-'][a-zA-ZäöüÄÖÜß]+)*")
print("归一化与词干工具就绪")

In [ ]:




_demo_df = load_subset(_demo_name)
_sample_row = _demo_df.iloc[0].to_dict()
_mwe = _sample_row[_demo_spec["key"]]
_context = _sample_row["Context"]
print("MWE   :", _mwe)
print("原句  :", _context[:170])


_toks = [(m.group(0), m.start(), m.end()) for m in WORD_RE.finditer(_context.lower())]
print(f"\n句子切成 {len(_toks)} 个词:", [t[0] for t in _toks][:14])


print("\n归一化与词干:")
print("  fold(MWE) =", fold(_mwe))
if _demo_spec["task"] == "nn":
    print("  词干键(复合词) =", sorted(stem_keys(_mwe, _demo_spec["lang"])))
else:
    print("  fold(基干)   =", fold(_sample_row["Base"]))
    print("  词干键(基干) =", sorted(stem_keys(_sample_row["Base"], _demo_spec["lang"])))
    print("  粒子         =", _sample_row["Particle"])


_m = re.search(re.escape(_mwe.lower()).replace(r"\-", r"[-']?").replace(r"\ ", r"[-']?\s"),
               _context.lower())
print("\n最小版本（只做 S0 精确匹配）:", ("命中 → " + repr(_context[_m.start():_m.end()])) if _m else "未命中")
print("→ 精确匹配在复合词上够用，但在小品词动词上只能覆盖约三分之一；")
print("   所以下一格把级联封装成一个函数，逐级放宽。")

In [ ]:

def _mark(text, start, end):
    return text[:start] + "[[" + text[start:end] + "]]" + text[end:]


def locate_target(row, sp):
    ctx = str(row["Context"])
    low = ctx.lower()
    mwe = str(row[sp["key"]])
    if sp["task"] == "nn":
        base, part = mwe, ""
    else:
        base, part = str(row["Base"]), str(row["Particle"])


    m = re.search(re.escape(mwe.lower()).replace(r"\-", r"[-']?").replace(r"\ ", r"[-']?\s"), low)
    if m:
        return _mark(ctx, m.start(), m.end()), "S0 精确匹配"

    toks = [(mm.group(0), mm.start(), mm.end()) for mm in WORD_RE.finditer(low)]
    if not toks:
        return ctx, "S5 未定位"


    if fold(mwe, True) and fold(mwe, True) in fold(low, True):
        return ctx, "S1 历史转写还原"

    ks = stem_keys(base, sp["lang"])
    p = fold(part)
    pw = {fold(part), fold(part, True)}
    fw = [fold(t[0]) for t in toks]
    fa = [fold(t[0], True) for t in toks]

    if sp["lang"] == "de":

        for i2, t in enumerate(toks):
            if not fw[i2].startswith(p):
                continue
            r, ra = fw[i2][len(p):], fa[i2][len(p):]
            if any(k and (k in r or k in ra) for k in ks):
                return _mark(ctx, t[1], t[2]), "S2a 德语前缀型(屈折)"

        for i2 in range(len(toks)):
            if not any(k and (k in fw[i2] or k in fa[i2]) for k in ks):
                continue
            for j2 in range(i2 + 1, min(i2 + 4, len(toks))):
                if fw[j2] in pw or fa[j2] in pw:
                    return _mark(ctx, toks[i2][1], toks[j2][2]), "S2b 德语后置型(动词分离)"
        return ctx, "S5 未定位"


    for i2, t in enumerate(toks):
        if any(k and (k in fw[i2] or k in fa[i2]) for k in ks):
            return _mark(ctx, t[1], t[2]), "S3 英文词干前缀"

    for i2 in range(len(toks) - 1):
        two = fw[i2] + fw[i2 + 1]
        if any(k and k in two for k in ks):
            return _mark(ctx, toks[i2][1], toks[i2 + 1][2]), "S4 相邻窗口"
    return ctx, "S5 未定位"


def build_input(mwe, marked_context):
    return f"MWE: {mwe} | Context: {marked_context}"



_marked, _strat = locate_target(_sample_row, _demo_spec)
print("策略  :", _strat)
print("标记后:", _marked[:200])
print("最终输入:", build_input(_mwe, _marked)[:220])

In [ ]:





for _n in SUBSET_ORDER:
    _sp = SUBSET_SPECS[_n]
    _d = load_subset(_n)
    _cnt = Counter()
    for _row in _d.to_dict("records"):
        _, _st = locate_target(_row, _sp)
        _cnt[_st] += 1
    _total = sum(_cnt.values())
    _loc = _total - _cnt.get("S5 未定位", 0) - _cnt.get("S1 历史转写还原", 0)
    print(f"{_n} 定位覆盖（n={_total}）成功标出 span {_loc / _total:.1%} | " +
          " | ".join(f"{k} {v}" for k, v in _cnt.most_common()))

In [ ]:

def split_by_target(df, sp, val_ratio=VAL_RATIO, seed=SEED, n_bins=5):
    key = sp["key"]
    tgt_mean = df.groupby(key)[sp["cols"]].mean().mean(axis=1)
    try:
        bins = pd.qcut(tgt_mean, n_bins, labels=False, duplicates="drop").to_numpy()
    except ValueError:
        bins = np.zeros(len(tgt_mean), dtype=int)

    rng = np.random.default_rng(seed)
    val_targets = []
    for b in np.unique(bins):
        idx = tgt_mean.index[bins == b].to_numpy()
        rng.shuffle(idx)
        k = max(1, int(round(len(idx) * val_ratio)))
        val_targets.extend(idx[:k].tolist())

    is_val = df[key].isin(set(val_targets)).to_numpy()
    return df[~is_val].reset_index(drop=True), df[is_val].reset_index(drop=True)

In [ ]:




_ALLOW_PATTERNS = ["*.json", "*.txt", "*.model", "*.safetensors", "*.bin", "*.py", "*.tiktoken"]


def resolve_model_dir(repo_id):
    from huggingface_hub import snapshot_download
    endpoints = choose_endpoints(repo_id)
    if not endpoints:
        return repo_id
    last_err = None
    for ep in endpoints:
        try:
            path = snapshot_download(repo_id, endpoint=ep, allow_patterns=_ALLOW_PATTERNS)
            print(f"权重就绪: {repo_id} @ {ep}\n         -> {path}")
            return path
        except Exception as e:
            last_err = e
            print(f"从 {ep} 下载失败: {type(e).__name__}: {str(e)[:180]}")
    raise last_err



if SMOKE:
    _SMOKE_IDS = {"deberta": "hf-internal-testing/tiny-random-DebertaV2Model",
                  "qwen": "Qwen/Qwen2.5-0.5B-Instruct"}
    MODEL_SOURCE = _SMOKE_IDS[MODEL_KEY]
    print("SMOKE=True：使用微型随机主干，不下载权重 ->", MODEL_SOURCE)
else:
    MODEL_SOURCE = resolve_model_dir(MODEL_REGISTRY[MODEL_KEY]["model_id"])

In [ ]:

def load_tokenizer(model_key, source):
    tok = AutoTokenizer.from_pretrained(source)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    tok.model_max_length = 512
    return tok


def build_dataset(frame, tokenizer, max_length=MAX_LENGTH):
    records = []
    for row in frame.to_dict("records"):
        sp = SUBSET_SPECS[row["subset_name"]]
        mwe = str(row[sp["key"]])
        marked, _ = locate_target(row, sp)
        records.append({
            "text": build_input(mwe, marked),
            "labels": [float(row["y0"]), float(row["y1"])],
            "loss_mask": [float(row["m0"]), float(row["m1"])],
            "group_index": 0,
            "group_key": f'{row["subset_name"]}||{mwe}',
            "subset_index": int(row["subset_id"]),
        })
    ds = Dataset.from_list(records)


    codes = {k: i for i, k in enumerate(sorted(set(frame["group_key"].astype(str))))}
    ds = ds.map(lambda b: {"group_index": [codes[k] for k in b["group_key"]]}, batched=True)
    ds = ds.remove_columns(["group_key"])

    if HEAD_MODE == "lm_head":

        cue_ids = [tokenizer(c, add_special_tokens=False).input_ids for c in READOUT_CUES]
        assert len(cue_ids) == N_OUTPUTS, (
            f"读出锚点个数 {len(cue_ids)} 与输出列数 {N_OUTPUTS} 不一致")
        budget = max_length - max(len(c) for c in cue_ids)

        def _tok_lm(b):
            ids0, ids1 = [], []
            for t in b["text"]:
                base = tokenizer(t, truncation=True, max_length=budget).input_ids
                ids0.append(list(base) + list(cue_ids[0]))
                ids1.append(list(base) + list(cue_ids[1]))
            return {"input_ids": ids0, "input_ids2": ids1}

        ds = ds.map(_tok_lm, batched=True, remove_columns=["text"])
    else:

        ds = ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=max_length),
                    batched=True, remove_columns=["text"])
    return ds


def ds_labels(ds, n_out=None):
    n_out = n_out if n_out is not None else N_OUTPUTS
    arr = np.asarray(ds["labels"], dtype=float)
    if arr.ndim == 1:
        arr = arr.reshape(-1, n_out)
    return arr


tokenizer = load_tokenizer(MODEL_KEY, MODEL_SOURCE)
print("tokenizer:", type(tokenizer).__name__, "| pad:", tokenizer.pad_token,
      "| padding_side:", tokenizer.padding_side)

In [ ]:





def prepare_run(subset_list, tokenizer):
    frames_tr, frames_va = [], []
    for name in subset_list:
        sp = SUBSET_SPECS[name]
        df = load_subset(name)
        df["subset_name"] = name
        df["subset_id"] = SUBSET_ORDER.index(name)
        df["group_key"] = name + "||" + df[sp["key"]].astype(str)
        tr, va = split_by_target(df, sp)
        frames_tr.append(tr)
        frames_va.append(va)
    train_df = pd.concat(frames_tr, ignore_index=True)
    val_df = pd.concat(frames_va, ignore_index=True)
    train_ds = build_dataset(train_df, tokenizer)
    val_ds = build_dataset(val_df, tokenizer)

    eval_cols = [(n, c, j) for n in subset_list for j, c in enumerate(SUBSET_SPECS[n]["cols"])]
    return dict(subset_list=list(subset_list), train_df=train_df, val_df=val_df,
                train_ds=train_ds, val_ds=val_ds, eval_cols=eval_cols)



_demo_run = prepare_run(["en-nn"], tokenizer)
_tr, _va = _demo_run["train_df"], _demo_run["val_df"]
print(f"en-nn 训练集 {_tr.shape} | 验证集 {_va.shape}")
print("目标交集:", len(set(_tr["group_key"]) & set(_va["group_key"])), "（必须为 0）")
for _n, _c, _j in _demo_run["eval_cols"]:
    _col = "y0" if _j == 0 else "y1"
    print(f"  {_n}·{_c}: train 均值 {_tr[_col].mean():.3f} | val 均值 {_va[_col].mean():.3f}")
print("示例解码:", tokenizer.decode(_demo_run["train_ds"][0]["input_ids"])[:220])

In [ ]:




_lens = [len(x) for x in _demo_run["train_ds"]["input_ids"]]
_fixed_total = len(_lens) * MAX_LENGTH
_dynamic_total = 0
for _i in range(0, len(_lens), BATCH_SIZE):
    _dynamic_total += min(BATCH_SIZE, len(_lens) - _i) * max(_lens[_i:_i + BATCH_SIZE])
print(f"序列长度: 中位 {int(np.median(_lens))} | 均值 {int(np.mean(_lens))} | 最长 {max(_lens)}")
print(f"定长补齐到 {MAX_LENGTH} 的 token 总量: {_fixed_total:,}")
print(f"动态补齐的 token 总量            : {_dynamic_total:,}")
print(f"→ 动态补齐省下 {(1 - _dynamic_total / _fixed_total):.1%} 的 token，"
      f"注意力开销按平方计，实际省得更多。")

In [ ]:

class CompartmentCollator:
    def __init__(self, tokenizer, pad_to_multiple_of=8):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, features):
        model_inputs = [{k: v for k, v in f.items()
                         if k not in ("labels", "loss_mask", "group_index", "subset_index")
                         and k != "input_ids2"}
                        for f in features]
        batch = self.tokenizer.pad(model_inputs, padding=True,
                                   pad_to_multiple_of=self.pad_to_multiple_of,
                                   return_tensors="pt")





        if "input_ids2" in features[0]:
            _b2 = self.tokenizer.pad([{"input_ids": f["input_ids2"]} for f in features],
                                     padding=True, pad_to_multiple_of=self.pad_to_multiple_of,
                                     return_tensors="pt")
            batch["input_ids2"] = _b2["input_ids"]
            batch["attention_mask2"] = _b2["attention_mask"]
        batch["labels"] = torch.tensor([f["labels"] for f in features], dtype=torch.float32)


        batch["loss_mask"] = torch.tensor([f["loss_mask"] for f in features], dtype=torch.float32)
        batch["group_index"] = torch.tensor([f["group_index"] for f in features], dtype=torch.long)
        batch["subset_index"] = torch.tensor([f["subset_index"] for f in features], dtype=torch.long)
        return batch


collator = CompartmentCollator(tokenizer)
_demo_batch = collator([_demo_run["train_ds"][i] for i in range(4)])
for k, v in _demo_batch.items():
    print(f"{k:16s} shape={tuple(v.shape)} dtype={v.dtype}")
print("\nlabels 样例   :", _demo_batch["labels"].tolist())
print("loss_mask 样例:", _demo_batch["loss_mask"].tolist())
print("subset_index  :", _demo_batch["subset_index"].tolist())
print("→ labels 是 [B, 2] 的 float 张量，没有 -100 填充；loss_mask 标出哪一列真有标签。")

## Step4 创建模型

**这一节要解决什么**：把「编码器」和「解码器」两个完全不同的架构，
装配成**同一个形状的模型**：`(input_ids, attention_mask) → 两个 0–5 的连续分数`。

### ★ v4：两种「读出方式」

v1–v3 用的是**回归头**；v4 里 Qwen 改用**完全不加回归头**的读出方式，
两条基线各自验证一个机制。两者的**输入端完全一样**，只有末端不同：

```
【A】回归头（DeBERTa，head_mode = "regression"）

AutoModel 主干 ──► last_hidden_state [B, L, H]
                          │
              掩码均值池化(编码器) / 取最后一个真实 token(解码器)
                          │
                       [B, H]
                          │
                  Dropout → Linear(H, n_out)          ← 新建的回归头
                          │
                    logits [B, n_out]
                          │
                  × 5 并落在 (0,5)

【B】无回归头（Qwen，head_mode = "lm_head"）

Qwen2ForCausalLM ──► 主干 ──► last_hidden_state [B, L, H]
                                        │
                            取锚点（"\nModifier: "）最后一个 token 处
                                        │
                                    [B, H]
                                        │
                   主干**自带**的 LM 头（不新建任何层，与 embedding 共享权重）
                                        │
                              全词表 logits [B, V]
                                        │
                        只取 "0"–"5" 这 6 个 token（id 15…20）
                                        │
                             p = softmax(6 个 logit)
                                        │
                    分数 = Σ_k k·p_k  ∈ [0,5]     ← 6 个数的凸组合
```

**为什么「期望值」这个读法是对的**：标签本来就是 0–5 的**连续均值**
（实测：每个标签是 10 个整数评分的均值，粒度 0.1，唯一值 47–51 个）。
所以「标签 3.5」对应的天然分布就是「3 与 4 的概率混合」，
而不是「3 或 4」—— 模型输出一个数字 token 的分布、再取期望，
恰好与标签的生成方式**同构**。

**为什么不用 `AutoModelForSequenceClassification`**：两条理由，都很硬。

1. **任务类型不匹配**。强行用 `num_labels=6` + 交叉熵，会把「4.2 分」和「4.0 分」的差别完全抹平，
   而指标 1（组合性相关）恰恰只看**排序**——抹平排序等于直接放弃指标 1。
   （注意：lm_head 模式里出现的「6」**不是**把标签离散化成 6 类，
   而是模型本来就要生成的 6 个 token；两回事。）
2. **接口不统一**。DeBERTa 的 `DebertaV2ForSequenceClassification` 与
   Qwen 的 `Qwen2ForSequenceClassification` 在 pooling、损失、logits 形状上都不一致。
   自己写一个 20 行的头，两个模型才能**逐行同构**，
   对照实验里唯一的变量才真的是「架构」而不是「框架细节」。

**为什么回归头那条要 ×5**：标签的量纲是 0–5，模型是无约束实数输出。
用 `sigmoid × 5` 把输出**先验地**限制在合法区间，
这样即使训练早期发散，预测值也不会跑到 [-100, 100] 这种让 RMSE 直接爆掉的范围。
（lm_head 模式不需要这一步：期望值是 6 个概率的凸组合，**天然**落在 [0,5]。）

In [ ]:

def shrink_config(cfg, num_layers=2, hidden_size=64, num_heads=4, tokenizer=None):
    cfg.num_hidden_layers = num_layers
    cfg.hidden_size = hidden_size
    cfg.num_attention_heads = num_heads
    cfg.intermediate_size = hidden_size * 2

    if hasattr(cfg, "attention_head_size"):
        cfg.attention_head_size = hidden_size // num_heads
    if hasattr(cfg, "head_dim"):
        cfg.head_dim = hidden_size // num_heads
    if hasattr(cfg, "embedding_size"):
        cfg.embedding_size = hidden_size
    if hasattr(cfg, "num_key_value_heads"):
        cfg.num_key_value_heads = min(int(getattr(cfg, "num_key_value_heads", 1)), num_heads)
    if tokenizer is not None:


        cfg.vocab_size = max(cfg.vocab_size, len(tokenizer))
    return cfg






_demo_cfg = shrink_config(AutoConfig.from_pretrained(MODEL_SOURCE), tokenizer=tokenizer)
_demo_backbone = AutoModel.from_config(_demo_cfg)
_demo_backbone.eval()

_demo_ids = tokenizer("MWE: account book | Context: He read his [[account book]] last night.",
                      return_tensors="pt")
with torch.no_grad():
    _demo_out = _demo_backbone(input_ids=_demo_ids["input_ids"],
                               attention_mask=_demo_ids["attention_mask"])
_h = _demo_out.last_hidden_state
print("主干类型:", type(_demo_backbone).__name__)
print("config: hidden_size=%d num_attention_heads=%d attention_head_size=%s head_dim=%s" % (
    _demo_cfg.hidden_size, _demo_cfg.num_attention_heads,
    getattr(_demo_cfg, "attention_head_size", "-"), getattr(_demo_cfg, "head_dim", "-")))
print("last_hidden_state shape:", tuple(_h.shape))


assert _h.shape[-1] == _demo_cfg.hidden_size, (
    f"主干输出宽度 {_h.shape[-1]} ≠ hidden_size {_demo_cfg.hidden_size}；"
    "多半是 config 里有派生宽度字段（attention_head_size / head_dim / embedding_size）没同步")
print("→ [batch, 序列长度, 隐层维度]。真正的「句子向量」还需要池化，这就是下面要做的。")

In [ ]:

class CompositionalityRegressor(nn.Module):

    def __init__(self, backbone, hidden_size, n_outputs=1, pooling="mean", dropout=0.1,
                 head_mode="regression", lm_head=None, digit_ids=None, digit_values=None,
                 proj_dim=None, tied_to_embedding=False):
        super().__init__()
        self.backbone = backbone
        self.pooling = pooling
        self.dropout = nn.Dropout(dropout)
        self.head_mode = head_mode
        self.n_outputs = int(n_outputs)
        if head_mode == "regression":
            self.head = nn.Linear(hidden_size, n_outputs)
            self.lm_head = None
            self.tied_to_embedding = False
        else:












            self.head = None
            self.tied_to_embedding = bool(tied_to_embedding)
            self.lm_head = None if self.tied_to_embedding else lm_head
            assert digit_ids is not None and len(digit_ids) == 6, (
                f"lm_head 模式需要 6 个数字 token 的 id，实际收到 {digit_ids}")
            self.register_buffer("digit_ids", torch.as_tensor(list(digit_ids), dtype=torch.long))
            self.register_buffer("digit_values",
                                 torch.as_tensor(list(digit_values), dtype=torch.float32))






            self._pending_readout_g6 = None


        self.proj = nn.Linear(hidden_size, proj_dim) if proj_dim else None

    def readout_weight(self):
        if self.lm_head is not None:
            return self.lm_head.weight
        return self.backbone.embed_tokens.weight








    supports_gradient_checkpointing = True

    def gradient_checkpointing_enable(self, gradient_checkpointing_kwargs=None, **kwargs):
        if not hasattr(self.backbone, "gradient_checkpointing_enable"):
            raise RuntimeError("当前主干不支持 gradient checkpointing，无法开启")
        self.backbone.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs=gradient_checkpointing_kwargs, **kwargs)

    @property
    def is_gradient_checkpointing(self):
        return bool(getattr(self.backbone, "is_gradient_checkpointing", False))


    def _pool(self, h, attention_mask):
        if self.pooling == "mean":
            m = attention_mask.unsqueeze(-1).to(h.dtype)
            return (h * m).sum(dim=1) / m.sum(dim=1).clamp(min=1e-6)
        last_idx = attention_mask.sum(dim=1) - 1
        return h[torch.arange(h.size(0), device=h.device), last_idx]


    def flush_readout_grad(self):
        _p = getattr(self, "_pending_readout_g6", None)
        if _p is None:
            return
        W = self.readout_weight()
        with torch.no_grad():
            if W.grad is None:
                W.grad = torch.zeros_like(W)
            W.grad.index_add_(0, self.digit_ids, _p.to(W.grad.dtype))
        self._pending_readout_g6 = None

    def _digit_logits(self, pooled):
        if self.lm_head is not None:
            W = self.lm_head.weight
            b = self.lm_head.bias
        else:
            W = self.backbone.embed_tokens.weight
            b = None



        Wd = W.detach()




        if pooled.is_cuda:
            with torch.autocast(device_type="cuda", enabled=False):
                out = pooled.float() @ Wd.index_select(0, self.digit_ids).float().t()
        else:
            out = pooled.float() @ Wd.index_select(0, self.digit_ids).float().t()



        if out.requires_grad:
            if getattr(self, "_pending_readout_g6", None) is None:
                self._pending_readout_g6 = torch.zeros(
                    (self.digit_ids.numel(), W.shape[1]),
                    dtype=torch.float32, device=W.device)
            _pooled_d = pooled.detach().float()

            def _readout_hook(g, _pd=_pooled_d):
                with torch.no_grad():
                    self._pending_readout_g6 += g.float().t() @ _pd
            out.register_hook(_readout_hook)
        if b is not None:
            out = out + b.index_select(0, self.digit_ids).float()
        return out


    def _hidden(self, input_ids, attention_mask):
        _inner = getattr(self.backbone, "model", None)
        if _inner is not None:
            _out = _inner(input_ids=input_ids, attention_mask=attention_mask)
            return _out.last_hidden_state
        try:
            _out = self.backbone(input_ids=input_ids, attention_mask=attention_mask,
                                 logits_to_keep=1)
            return _out.last_hidden_state
        except TypeError:
            _out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
            return _out.last_hidden_state

    def forward(self, input_ids=None, attention_mask=None,
                input_ids2=None, attention_mask2=None, need2=None,
                labels=None, group_index=None, **kwargs):
        if self.head_mode == "regression":
            out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
            h = out.last_hidden_state
            pooled = self._pool(h, attention_mask)
            logits = self.head(self.dropout(pooled))






            logits = 5.0 * torch.sigmoid(logits)
            proj = None
            if self.proj is not None:

                proj = F.normalize(self.proj(self.dropout(pooled)), dim=-1)
            return ModelOutput(logits=logits, proj=proj)



        h = self._hidden(input_ids, attention_mask)



        pooled0 = self._pool(h, attention_mask)
        d0 = self._digit_logits(pooled0)
        p0 = d0.softmax(dim=-1)
        exp0 = (p0 * self.digit_values.to(p0.dtype)).sum(dim=-1)
        exps = [exp0]
        d_list = [d0]
        if self.n_outputs > 1:
            if input_ids2 is None:
                raise RuntimeError(
                    "lm_head 模式需要第 2 列的 token（input_ids2）。"
                    "请确认数据管线走的是 HEAD_MODE == 'lm_head' 分支。")
            B = input_ids.shape[0]
            if need2 is None:
                need2 = torch.ones(B, dtype=torch.bool, device=input_ids.device)
            idx = need2.nonzero(as_tuple=True)[0]
            e1 = torch.zeros(B, device=exp0.device, dtype=exp0.dtype)
            d1 = torch.zeros(B, d0.shape[1], device=d0.device, dtype=d0.dtype)
            if idx.numel() > 0:
                h2 = self._hidden(input_ids2[idx], attention_mask2[idx])
                pooled1 = self._pool(h2, attention_mask2[idx])
                d1v = self._digit_logits(pooled1)
                p1 = d1v.softmax(dim=-1)
                e1 = e1.index_copy(0, idx, (p1 * self.digit_values.to(p1.dtype)).sum(dim=-1))
                d1 = d1.index_copy(0, idx, d1v)
            exps.append(e1)
            d_list.append(d1)
        logits = torch.stack(exps, dim=1)
        digit_logits = torch.stack(d_list, dim=1)
        return ModelOutput(logits=logits, digit_logits=digit_logits)


def build_tiny_backbone(model_key, source, tokenizer):
    from transformers import DebertaV2Model, Qwen2Model
    cfg = shrink_config(AutoConfig.from_pretrained(source), tokenizer=tokenizer)
    if model_key == "qwen":
        cfg.use_cache = False
        return Qwen2Model(cfg)
    return DebertaV2Model(cfg)


def build_tiny_causal_lm(source, tokenizer):
    from transformers import Qwen2ForCausalLM
    cfg = shrink_config(AutoConfig.from_pretrained(source), tokenizer=tokenizer)
    cfg.use_cache = False
    return Qwen2ForCausalLM(cfg)



_AUTO_PRETRAINED_PARAMS = set(inspect.signature(AutoModel.from_pretrained).parameters)
_DTYPE_KW = "dtype" if "dtype" in _AUTO_PRETRAINED_PARAMS else "torch_dtype"
print("from_pretrained 的精度参数名:", _DTYPE_KW)


def load_backbone():
    mcfg = MODEL_REGISTRY[MODEL_KEY]
    if SMOKE:
        return build_tiny_backbone(MODEL_KEY, MODEL_SOURCE, tokenizer)

    backbone, last_err = None, None
    for impl in mcfg["attn_impls"]:
        try:
            backbone = AutoModel.from_pretrained(MODEL_SOURCE, **{_DTYPE_KW: torch.float32},
                                                 attn_implementation=impl)
            print(f"主干加载成功: {mcfg['model_id']} | attn_implementation={impl}")
            break
        except ValueError as e:
            if "attention implementation" not in str(e).lower():
                raise
            last_err = e
            print(f"attn_implementation={impl} 不支持该架构，自动回退")
    if backbone is None:
        raise last_err

    backbone.config.use_cache = False
    return backbone


def load_causal_lm():
    mcfg = MODEL_REGISTRY[MODEL_KEY]
    if SMOKE:
        return build_tiny_causal_lm(MODEL_SOURCE, tokenizer)

    clm, last_err = None, None
    for impl in mcfg["attn_impls"]:
        try:
            clm = AutoModelForCausalLM.from_pretrained(MODEL_SOURCE, **{_DTYPE_KW: torch.float32},
                                                       attn_implementation=impl)
            print(f"因果语言模型加载成功: {mcfg['model_id']} | attn_implementation={impl}")
            break
        except ValueError as e:
            if "attention implementation" not in str(e).lower():
                raise
            last_err = e
            print(f"attn_implementation={impl} 不支持该架构，自动回退")
    if clm is None:
        raise last_err
    clm.config.use_cache = False
    return clm


def resolve_digit_ids(tokenizer):
    ids = []
    for d in range(6):
        _s = str(d)
        _enc = tokenizer(_s, add_special_tokens=False).input_ids
        assert len(_enc) == 1, (
            f"数字 {_s!r} 不是单 token（编码成 {_enc}）："
            "无回归头模式的读出机制依赖「数字 = 一个 token」，否则分布读错了且不会报错。")
        ids.append(int(_enc[0]))

    for d in range(6):
        _enc = tokenizer(" " + str(d), add_special_tokens=False).input_ids
        assert len(_enc) == 2 and int(_enc[1]) == ids[d], (
            f"' {d}' 的编码 {_enc} 说明空格与数字被合并了（期望 [空格, 数字]）："
            "锚点读出的下一个 token 就不再是数字本身。")

    for c in READOUT_CUES:
        _enc = tokenizer(c, add_special_tokens=False).input_ids
        assert tokenizer.decode([_enc[-1]]) == " ", (
            f"锚点 {c!r} 的最后一个 token 解码后不是空格（是 {tokenizer.decode([_enc[-1]])!r}）："
            "锚点必须以空格结尾，模型生成的下一个 token 才是数字。")
    print(f"数字 token id = {ids} | 锚点 = {READOUT_CUES}（均以空格结尾 ✓）")
    return ids


_DIGIT_IDS_CACHE = None


def get_digit_ids(tokenizer):
    global _DIGIT_IDS_CACHE
    if _DIGIT_IDS_CACHE is None:
        _DIGIT_IDS_CACHE = resolve_digit_ids(tokenizer)
    return _DIGIT_IDS_CACHE






In [ ]:

FIT_SCOPE_FRAC = 1.0


def _find_layer_list(backbone):
    for path in ("encoder.layer", "layers", "layer", "decoder.layers", "encoder.decoder.layer"):
        obj = backbone
        for part in path.split("."):
            if not hasattr(obj, part):
                obj = None
                break
            obj = getattr(obj, part)
        if obj is not None and isinstance(obj, (nn.ModuleList, list)) and len(obj) > 0:
            return obj
    return None


def apply_fit_scope(model, scope_frac=None):
    frac = FIT_SCOPE_FRAC if scope_frac is None else scope_frac
    if frac >= 1.0:
        return model
    layers = _find_layer_list(model.backbone)
    if layers is None:
        print(f"⚠️ 找不到主干的层列表，本档位（只训顶部 {frac:.0%} 层）无法生效 → 退回全参微调")
        return model
    n_total = len(layers)
    n_keep = max(1, int(round(n_total * frac)))
    for p in model.backbone.parameters():
        p.requires_grad_(False)
    for layer in layers[n_total - n_keep:]:
        for p in layer.parameters():
            p.requires_grad_(True)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_all = sum(p.numel() for p in model.parameters())
    print(f"档位生效：只训练顶部 {n_keep}/{n_total} 层 | 可训练参数 {n_tr / 1e6:.1f}M / "
          f"{n_all / 1e6:.1f}M ({n_tr / n_all:.1%})")
    return model


def ensure_readout_trainable(model):
    if getattr(model, "lm_head", None) is None and getattr(model, "head", None) is not None:
        return model
    if getattr(model, "head", None) is not None and getattr(model, "lm_head", None) is None:
        return model
    if model.head_mode != "lm_head":
        return model
    w = model.readout_weight()
    w.requires_grad_(True)
    _n = w.numel()
    assert _n > 0, "读出权重被冻结了：读出通路学不到东西（无回归头模式会退化成随机读出）"
    print(f"读出通路已确认可训练：读出权重 {_n / 1e6:.2f}M"
          f"（{'与 embedding 共享同一张量 ⇒ embedding 同时可训' if model.tied_to_embedding else '独立的 LM 头'}）")
    return model

In [ ]:

def build_model():
    _use_proj = any(float(v.get("softcl", 0.0)) > 0 for v in SUBSET_LOSS_PROFILE.values())
    if HEAD_MODE == "lm_head":
        clm = load_causal_lm()
        _digit_ids = get_digit_ids(tokenizer)


        _tied = False
        try:
            _tied = (clm.lm_head.weight.data_ptr() == clm.model.embed_tokens.weight.data_ptr())
        except Exception:
            _tied = bool(getattr(clm.config, "tie_word_embeddings", False))
        print(f"读出权重绑定检测：lm_head.weight 与 embed_tokens.weight "
              f"{'共享同一块显存（⇒ 读出直接用 embedding 矩阵，避免 safetensors 共享张量报错）' if _tied else '不共享（⇒ 直接挂 lm_head 子模块）'}")
        m = CompositionalityRegressor(
            backbone=clm.model,
            hidden_size=clm.config.hidden_size,
            n_outputs=N_OUTPUTS,
            pooling=MODEL_REGISTRY[MODEL_KEY]["pooling"],
            head_mode="lm_head",
            lm_head=clm.lm_head,
            digit_ids=_digit_ids,
            digit_values=DIGIT_VALUES,
            proj_dim=None,
            tied_to_embedding=_tied,
        ).to(DEVICE)
    else:
        bb = load_backbone()
        m = CompositionalityRegressor(
            backbone=bb,
            hidden_size=bb.config.hidden_size,
            n_outputs=N_OUTPUTS,
            pooling=MODEL_REGISTRY[MODEL_KEY]["pooling"],
            head_mode="regression",

            proj_dim=(SOFTCL_PROJ_DIM if _use_proj else None),
        ).to(DEVICE)
    apply_fit_scope(m)
    ensure_readout_trainable(m)
    return m


model = build_model()
n_all = sum(p.numel() for p in model.parameters())
if HEAD_MODE == "regression":
    n_head = sum(p.numel() for p in model.head.parameters())
    _head_txt = f"回归头 {n_head / 1e3:.1f}K ({n_head / n_all:.4%})"
else:
    _n_lm = sum(p.numel() for p in model.readout_weight())
    _head_txt = f"**无回归头**（用主干自带的 LM 头 {_n_lm / 1e6:.1f}M" \
                + ("，与 embedding 共享权重" if model.tied_to_embedding else "）")
print(f"主干       : {MODEL_REGISTRY[MODEL_KEY]['model_id']}（SMOKE={SMOKE}）")
print(f"读出方式   : {HEAD_MODE} | {_head_txt}")
print(f"池化方式   : {MODEL_REGISTRY[MODEL_KEY]['pooling']}")
print(f"输出维度   : {N_OUTPUTS}（统一 2 列）")
print(f"总参数     : {n_all / 1e6:.1f}M | {_head_txt}")
if model.proj is not None:
    print(f"投影头     : {SOFTCL_PROJ_DIM} 维（仅供 softcl 对比学习使用）")
print(f"设备       : {DEVICE}")


model.eval()
with torch.no_grad():
    _b = collator([_demo_run["train_ds"][i] for i in range(2)])
    _out = model(input_ids=_b["input_ids"].to(DEVICE),
                 attention_mask=_b["attention_mask"].to(DEVICE),
                 input_ids2=_b["input_ids2"].to(DEVICE) if "input_ids2" in _b else None,
                 attention_mask2=_b["attention_mask2"].to(DEVICE) if "attention_mask2" in _b else None,
                 need2=(_b["loss_mask"][:, 1] > 0).to(DEVICE) if "loss_mask" in _b else None)
print("前向 logits shape:", tuple(_out.logits.shape))
print("是否含 NaN:", bool(torch.isnan(_out.logits).any().item()))
if HEAD_MODE == "lm_head":

    print("lm_head 读出样例:", _out.logits.detach().cpu().tolist())
    print("数字 token 分布 shape:", tuple(_out.digit_logits.shape),
          "（每列 6 个数字 token 的 logits）")
    assert float(_out.logits.min()) >= 0.0 and float(_out.logits.max()) <= 5.0, (
        "期望值读出必须落在 [0,5]：它是 6 个数字的概率加权平均，不可能越界")

## Step5 创建评估函数

**这一节要解决什么**：把官方的三项指标**逐字实现**出来，并让本地实现和
未来用 Trial 的 `scores.txt` 对得上。

| 指标 | 官方口径 | 一句话含义 |
|---|---|---|
| 指标 1 | 组合性相关 ρ | 全部例句上 Spearman(预测, 真值)——整体排序对不对 |
| 指标 2 | 语境方差相关 ρ | 每个目标跨例句算 std，再在**目标之间** Spearman(预测 std, 人工 std)——模型有没有真的「因句而异」 |
| 指标 3 | RMSE | 全部例句上的均方根误差——数值校准准不准 |

**指标 2 为什么是硬门槛**：如果模型对同一目标的所有例句给出同一个分数，
那它的组内 std 恒为 0，相关系数的输入数组是常数 → **相关系数未定义（nan）**。
所以指标 2 不是「加分项」，而是**及格线**：一个常数输出系统直接出局。

**指标 1 与指标 2 存在张力**：指标 1 奖励跨目标的数值校准，
指标 2 奖励目标内部的离散程度。过度压缩输出方差 → 指标 2 掉分；
过度放大方差 / 加噪声 → 指标 1 的排序被破坏、RMSE 变差。
本 notebook 不做后处理调参（那属于刷分），而是让模型真正对语境敏感。

**这一节末尾有一组断言自检**：用「完美预测 / 常数预测 / 反向预测」三种人造系统
把指标函数验一遍。指标实现一旦写错，训练完才发现就太晚了。

In [ ]:


_demo_v = _demo_run["val_df"]
_demo_key = SUBSET_SPECS["en-nn"]["key"]
_type_level_pred = _demo_v.groupby(_demo_key)["ModAvg"].transform("mean")
_gold = _demo_v["ModAvg"].to_numpy()
_g = _demo_v[_demo_key].to_numpy()

_df = pd.DataFrame({"g": _g, "p": _type_level_pred.to_numpy(), "y": _gold})
_ps = _df.groupby("g")["p"].std(ddof=1)
_gs = _df.groupby("g")["y"].std(ddof=1)
_ok = _ps.notna() & _gs.notna()
print(f"type-level 基线：指标1 Spearman = {stats.spearmanr(_type_level_pred, _gold)[0]:.3f}"
      f" | 指标3 RMSE = {np.sqrt(((_type_level_pred.to_numpy() - _gold) ** 2).mean()):.3f}")
print(f"组内预测 std 的唯一取值个数 = {_ps[_ok].nunique()}（0 表示每个目标内完全相同）")
print("→ 指标 2 的输入是常数数组，相关系数**未定义（nan）**：该系统在指标 2 上直接出局。")
print("→ 这就是为什么本任务必须「逐例句预测」，不能先聚合成目标级分数再回归。")

In [ ]:

def strip_eval_prefix(metrics):
    return {k[len("eval_"):] if k.startswith("eval_") else k: v for k, v in metrics.items()}


def _three_metrics(p, g, grp):


    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ConstantInputWarning)
        m1 = float(stats.spearmanr(p, g)[0])
        frame = pd.DataFrame({"g": grp, "p": p, "y": g})
        ps = frame.groupby("g")["p"].std(ddof=1)
        gs = frame.groupby("g")["y"].std(ddof=1)
        ok = ps.notna() & gs.notna()
        if ok.sum() > 2 and ps[ok].nunique() > 1:
            m2 = float(stats.spearmanr(ps[ok], gs[ok])[0])
        else:
            m2 = float("nan")
    rmse = float(np.sqrt(((p - g) ** 2).mean()))
    return m1, m2, rmse, int((ps[ok] == 0).sum())


def compute_compartment_metrics(pred, gold, group_index, subset_ids, eval_cols):
    pred = np.asarray(pred, dtype=float).reshape(len(pred), -1)
    gold = np.asarray(gold, dtype=float).reshape(len(gold), -1)
    group = np.asarray(group_index)
    subset_ids = np.asarray(subset_ids)
    out = {}
    _m1s, _m2s, _rmses, _ndegs = [], [], [], []
    for name, cname, j in eval_cols:
        m = subset_ids == name
        if m.sum() < 3:
            continue
        m1, m2, rmse, ndeg = _three_metrics(pred[m, j], gold[m, j], group[m])
        out[f"{name}|{cname}_指标1_ρ"] = m1
        out[f"{name}|{cname}_指标2_ρ"] = m2
        out[f"{name}|{cname}_指标3_RMSE"] = rmse
        out[f"{name}|{cname}_组内std退化目标数"] = ndeg
        out[f"{name}|{cname}_样例数"] = int(m.sum())
        _m1s.append(m1); _m2s.append(m2); _rmses.append(rmse); _ndegs.append(ndeg)

    def _nanmean(vals):
        v = np.asarray([x for x in vals if np.isfinite(x)], dtype=float)
        return float(v.mean()) if len(v) else float("nan")

    out["合计|_指标1_ρ"] = _nanmean(_m1s)
    out["合计|_指标2_ρ"] = _nanmean(_m2s)
    out["合计|_指标3_RMSE"] = _nanmean(_rmses)
    out["合计|_组内std退化目标数"] = int(np.sum(_ndegs)) if _ndegs else 0
    out["合计|_评测设置数"] = len(_m1s)
    out["合计|_样例数"] = int(len(pred))
    out["合计|_目标数"] = int(len(np.unique(group)))









    _m1 = out["合计|_指标1_ρ"]
    _m2 = out["合计|_指标2_ρ"]
    _m3 = out["合计|_指标3_RMSE"]
    _c1 = 0.0 if not np.isfinite(_m1) else float(_m1)
    _c2 = float(_m2) if np.isfinite(_m2) else -1.0
    _c3 = float(_m3) if np.isfinite(_m3) else float("nan")
    out[BEST_METRIC_KEY] = (_c1 / Z_M1) + (_c2 / Z_M2) - (_c3 / Z_M3 if np.isfinite(_c3) else 0.0)
    return out


def make_compute_metrics(eval_group_index, eval_subset_index, eval_cols, subset_names):
    groups = np.asarray(eval_group_index)






    _max_id = int(np.max(eval_subset_index)) if len(eval_subset_index) else -1
    assert _max_id < len(subset_names), (
        f"subset_index 最大值 {_max_id} 超出 subset_names 长度 {len(subset_names)}："
        "subset_index 是全局编号，subset_names 必须传全局的 SUBSET_ORDER")
    subsets = np.asarray([subset_names[i] for i in eval_subset_index])

    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        preds = np.asarray(preds)
        labels = np.asarray(labels)
        if preds.ndim == 1:
            preds = preds.reshape(-1, 1)
        if labels.ndim == 1:
            labels = labels.reshape(-1, 1)
        return compute_compartment_metrics(preds, labels, groups, subsets, eval_cols)

    return compute_metrics

In [ ]:




_gold = ds_labels(_demo_run["val_ds"])
_grp = np.asarray(_demo_run["val_ds"]["group_index"])
_sub = np.asarray(["en-nn"] * len(_grp))
_cols = _demo_run["eval_cols"]


_m = compute_compartment_metrics(_gold, _gold, _grp, _sub, _cols)
for _c in _cols:
    _k1 = f"{_c[0]}|{_c[1]}_指标1_ρ"
    _k2 = f"{_c[0]}|{_c[1]}_指标2_ρ"
    _k3 = f"{_c[0]}|{_c[1]}_指标3_RMSE"
    assert abs(_m[_k1] - 1.0) < 1e-9, f"完美预测的指标1 应为 1，实际 {_m[_k1]}"
    assert abs(_m[_k2] - 1.0) < 1e-9, f"完美预测的指标2 应为 1，实际 {_m[_k2]}"
    assert _m[_k3] < 1e-9, f"完美预测的 RMSE 应为 0，实际 {_m[_k3]}"
print("① 完美预测      : 指标1=1.000 指标2=1.000 RMSE=0.000  ✓")


_m = compute_compartment_metrics(np.full_like(_gold, 3.0), _gold, _grp, _sub, _cols)
for _c in _cols:
    _k1 = f"{_c[0]}|{_c[1]}_指标1_ρ"
    _k2 = f"{_c[0]}|{_c[1]}_指标2_ρ"
    assert np.isnan(_m[_k1]), f"常数预测的指标1 应为 nan，实际 {_m[_k1]}"
    assert np.isnan(_m[_k2]), f"常数预测的指标2 应为 nan，实际 {_m[_k2]}"
print("② 常数预测      : 指标1=nan 指标2=nan（未定义）              ✓")


_m = compute_compartment_metrics(-_gold, _gold, _grp, _sub, _cols)
for _c in _cols:
    _k1 = f"{_c[0]}|{_c[1]}_指标1_ρ"
    assert abs(_m[_k1] + 1.0) < 1e-9, f"反向预测的指标1 应为 -1，实际 {_m[_k1]}"
print("③ 反向预测      : 指标1=-1.000                              ✓")


_flat = np.full_like(_gold, 2.5)
_m = compute_compartment_metrics(_gold, _flat, _grp, _sub, _cols)
for _c in _cols:
    _k2 = f"{_c[0]}|{_c[1]}_指标2_ρ"
    assert np.isnan(_m[_k2]), f"人工 std 恒为 0 时指标2 应为 nan，实际 {_m[_k2]}"
print("④ 人工 std 恒为 0: 指标2=nan（被解释变量无方差）           ✓")
print("→ 指标实现通过四组断言，接下来所有结果都由它产出。")

## Step6 配置训练参数

**这一节要解决什么**：把超参组装成一个**函数**，而不是散落的字面量。

**为什么必须是函数**：transformers 的 `Trainer` 会**就地改写** `TrainingArguments` 对象
（`per_device_train_batch_size`、`max_steps`、`label_names`、`_n_gpu` 都会被动过）。
如果两个 Trainer 共用同一个 args 对象，第二个 Trainer 拿到的就是「已经被训练过」的脏 args，
它的 `num_batches` 与实际 dataloader 长度对不上，直接抛
`ValueError: zip() argument 2 is longer than argument 1`。
所以「超参相同」的正确实现是：**每次调用 `build_training_args()` 重新构造一个对象**，
字段默认值相同即等价，而不是复用同一个变量。

**超参取值与理由**：

| 超参 | 值 | 为什么 |
|---|---|---|
| `learning_rate` | 2e-5 | 微调大模型的常规档位。两个模型**用同一个值**，对照实验里 lr 不成为混淆变量 |
| `per_device_train_batch_size` | 8 | 序列长 160、模型 900M～1.5B；显存不够时用 `gradient_accumulation_steps` 把有效 batch 补回去（数学等价、几乎无损，第一选择） |
| `num_train_epochs` | 3 | 本项目每个目标只有 5–10 条例句，全量数据仅 1.5k–3.5k 条；轮数太多会过拟合到「记住句子」 |
| `warmup_steps` | 总步数 × 10% | transformers 5 **删除了 `warmup_ratio`**，只保留绝对步数，所以这里自己算 |
| `lr_scheduler_type` | linear | 线性衰减是回归任务的稳定默认；余弦/常数留到消融时再比 |
| `weight_decay` | 0.01 | 常规正则量级 |
| `eval_strategy` / `save_strategy` | epoch | transformers 5 把 `evaluation_strategy` 改名为 `eval_strategy` |
| `load_best_model_at_end` | True | 必须与 `save_strategy="epoch"` 同时满足，否则初始化阶段直接抛 ValueError |

In [ ]:






_TA_PARAMS = set(inspect.signature(TrainingArguments.__init__).parameters)
_EVAL_STRATEGY_KEY = "eval_strategy" if "eval_strategy" in _TA_PARAMS else "evaluation_strategy"
_TRAINER_PARAMS = set(inspect.signature(Trainer.__init__).parameters)
_TOKENIZER_KW = "processing_class" if "processing_class" in _TRAINER_PARAMS else "tokenizer"
_HAS_SAVE_ONLY_MODEL = "save_only_model" in _TA_PARAMS
print("transformers 兼容层:", {"eval_key": _EVAL_STRATEGY_KEY, "tokenizer_kw": _TOKENIZER_KW,
                                "save_only_model": _HAS_SAVE_ONLY_MODEL})



def _bnb_available():
    try:
        import bitsandbytes
        return True
    except Exception:
        return False


_HAS_BNB = _bnb_available()


def _use_8bit_allowed():
    mode = USE_8BIT_OPT
    if isinstance(mode, str):
        low = mode.strip().lower()
        if low in ("true", "1", "yes", "8bit"):
            return True
        if low in ("false", "0", "no", "fp32"):
            return False
        return _HAS_BNB
    return bool(mode)






























FIT_ACT_MIN_GB = 0.3

FIT_TIERS = [
    {"name": "t0", "label": "全参微调 · fp32 AdamW · AMP", "optim": "adamw_torch",
     "amp": True, "scope": 1.00, "ckpt": False, "act_gb": 2.0},
    {"name": "t1", "label": "全参微调 · 8bit AdamW · AMP", "optim": "adamw_bnb_8bit",
     "amp": True, "scope": 1.00, "ckpt": False, "act_gb": 2.0},
    {"name": "t2", "label": "全参微调 · Adafactor · AMP", "optim": "adafactor",
     "amp": True, "scope": 1.00, "ckpt": True, "act_gb": 1.5},
    {"name": "t3", "label": "全参微调 · Adafactor · 无 AMP", "optim": "adafactor",
     "amp": False, "scope": 1.00, "ckpt": True, "act_gb": 1.5},
    {"name": "t4", "label": "顶部 50% 层 · Adafactor · AMP", "optim": "adafactor",
     "amp": True, "scope": 0.50, "ckpt": True, "act_gb": 1.5},
    {"name": "t5", "label": "顶部 50% 层 · Adafactor · 无 AMP", "optim": "adafactor",
     "amp": False, "scope": 0.50, "ckpt": True, "act_gb": 1.5},
    {"name": "t6", "label": "顶部 25% 层 · Adafactor · 无 AMP", "optim": "adafactor",
     "amp": False, "scope": 0.25, "ckpt": True, "act_gb": 1.5},
]

_OPTIM_STATE_BYTES = {"adamw_torch": 8.0, "adamw_bnb_8bit": 2.0, "adafactor": 0.2}


def tier_fixed_bytes(tier):
    w = 4.0
    amp = 2.0 if tier["amp"] else 0.0
    st = _OPTIM_STATE_BYTES[tier["optim"]]
    return w + amp + tier["scope"] * (4.0 + st)


def tier_estimate_gb(n_params, tier, act_gb=None):
    fixed = n_params * tier_fixed_bytes(tier) / (1024 ** 3)
    act = tier["act_gb"] if act_gb is None else act_gb
    return fixed, fixed + act


def _tier_available(tier):
    if tier["optim"] == "adamw_bnb_8bit":
        return _use_8bit_allowed() and _HAS_BNB
    return True


def _vram_total_gb():
    if not torch.cuda.is_available():
        return None
    return torch.cuda.get_device_properties(torch.cuda.current_device()).total_memory / (1024 ** 3)


def resolve_fit_tier(vram_gb, ref_params):
    if vram_gb is None:
        return FIT_TIERS[0], []
    budget = vram_gb * FIT_SAFETY
    rows, chosen = [], None
    for t in FIT_TIERS:
        if not _tier_available(t):
            rows.append((t["name"], t["label"], None, None, None,
                         "不可用（需要 bitsandbytes）" if t["optim"] == "adamw_bnb_8bit" else "不可用"))
            continue

        fixed, need_min = tier_estimate_gb(ref_params, t, act_gb=FIT_ACT_MIN_GB)
        _fixed8, need8 = tier_estimate_gb(ref_params, t)
        ok = need_min <= budget
        rows.append((t["name"], t["label"], fixed, need_min, need8,
                     "✓ 装得下" if ok else f"✗ 超出预算 {budget:.1f} GiB"))
        if ok and chosen is None:
            chosen = t

    if FIT_MODE != "auto":
        forced = next((t for t in FIT_TIERS if t["name"] == str(FIT_MODE).lower()), None)
        if forced is not None and _tier_available(forced):
            chosen = forced
    if chosen is None:

        avail = [t for t in FIT_TIERS if _tier_available(t)]
        chosen = avail[-1] if avail else FIT_TIERS[0]
    return chosen, rows


FIT_TIER_NAME = FIT_TIERS[0]["name"]
OPTIM = FIT_TIERS[0]["optim"]
USE_GRAD_CKPT = False
FIT_SCOPE_FRAC = 1.0


def apply_tier(tier):
    global OPTIM, USE_GRAD_CKPT, FIT_SCOPE_FRAC, FIT_TIER_NAME, USE_BF16, USE_FP16
    OPTIM = tier["optim"]
    USE_GRAD_CKPT = tier["ckpt"]
    FIT_SCOPE_FRAC = tier["scope"]
    FIT_TIER_NAME = tier["name"]
    if tier["amp"]:
        USE_BF16, USE_FP16 = HW_BF16, HW_FP16
    else:
        USE_BF16, USE_FP16 = False, False


def check_vram_budget(n_params, model_name, optim=None):
    if not torch.cuda.is_available():
        print("非 CUDA 环境：跳过显存预算检查（按标准全参微调跑）")
        return None
    n_gpu = torch.cuda.device_count()
    for _i in range(n_gpu):
        _p = torch.cuda.get_device_properties(_i)
        print(f"  GPU{_i}: {_p.name} | {_p.total_memory / 1024 ** 3:.1f} GiB | "
              f"compute capability {_p.major}.{_p.minor}")
    total_gb = _vram_total_gb()
    print(f"显存预算：{model_name} | {n_params / 1e6:.0f}M 参数（实际） | "
          f"选档参考 {FIT_REF_PARAMS / 1e6:.0f}M（仅 auto 模式用；当前 FIT_MODE={FIT_MODE}） | "
          f"单卡容量 {total_gb:.1f} GiB | 安全系数 {FIT_SAFETY:.0%}")
    tier, rows = resolve_fit_tier(total_gb, FIT_REF_PARAMS)
    print("  档位表（固定开销与 batch 无关，降 batch 救不了）：")
    print(f"    {'档':4s}{'含义':34s}{'固定':>9s}{'bs=1 需':>10s}{'bs=8 需':>10s}  判定")
    for name, label, fixed, need_min, need8, verdict in rows:
        if fixed is None:
            print(f"    {name:4s}{label:34s}{'-':>9s}{'-':>10s}{'-':>10s}  {verdict}")
        else:
            print(f"    {name:4s}{label:34s}{fixed:8.1f}G{need_min:9.1f}G{need8:9.1f}G  {verdict}")
    fixed_act, need_act = tier_estimate_gb(n_params, tier)
    print(f"  → 采用档位 {tier['name']}（{tier['label']}）；"
          f"本模型固定开销 {fixed_act:.1f} GiB"
          f"（+ 激活：bs=1 约 {tier['act_gb'] * 0.2:.1f} / bs=8 约 {tier['act_gb']:.1f} GiB，"
          f"实际能跑多大由探测决定）")
    if n_gpu > 1:
        print(f"  ⚠️ 本机有 {n_gpu} 张卡，但**单进程默认只用 GPU0**。多卡要用对方式才有效：")
        print("     · DDP（数据并行）：每张卡各存**一整份**模型 → **不降低单卡显存需求**，只是把数据分半")
        print("     · ZeRO-3 / FSDP（参数分片）：把参数+梯度+优化器状态切到多卡 → "
              "这才能真正用满多卡显存（需装 deepspeed）")
        print("     · 最省事：两条基线**各占一张卡并行跑**（两个独立实验，互不干扰）")
    _fixed_m, need_min = tier_estimate_gb(n_params, tier, act_gb=FIT_ACT_MIN_GB)
    if need_min <= total_gb:
        return True
    print("=" * 78)
    print(f"⚠️ 警告：档位 {tier['name']} 即使按 micro-batch=1 也要约 {need_min:.1f} GiB，"
          f"本卡只有 {total_gb:.1f} GiB。")
    print("   Step8 会用真实模型逐档试探；若仍 OOM 会自动再降一档，"
          "并在全部档位都失败时给出精确诊断（而不是一句 out of memory）。")
    print("=" * 78)
    return False


def ds_int_col(ds, name):
    arr = np.asarray(ds[name])
    if arr.ndim > 1:
        arr = arr.reshape(-1)
    return arr.astype(np.int64)


def _assert_args_consistent(args, n_train):
    _bs = int(args.per_device_train_batch_size)
    _ga = int(args.gradient_accumulation_steps)
    _ep = int(args.num_train_epochs)
    assert _bs == BATCH_SIZE, (
        f"per_device_train_batch_size={_bs} ≠ 配置 {BATCH_SIZE}：口径漂移，拒绝开训")
    assert _ga == GRAD_ACCUM, (
        f"gradient_accumulation_steps={_ga} ≠ 配置 {GRAD_ACCUM}：口径漂移，拒绝开训")
    _loader = math.ceil(n_train / _bs)
    _max_steps = math.ceil(_loader / _ga) * _ep
    _expected_warmup = max(1, int(math.ceil(n_train / (_bs * _ga)) * _ep * WARMUP_RATIO))
    assert int(args.warmup_steps) == _expected_warmup, (
        f"warmup_steps={args.warmup_steps} ≠ 按同一口径算出的 {_expected_warmup}：口径漂移，拒绝开训")
    _wf = float(args.warmup_steps) / max(1, _max_steps)


    _tol = max(0.02, 2.0 / max(1, _max_steps))
    assert abs(_wf - WARMUP_RATIO) <= _tol, (
        f"warmup 实际占比 {_wf:.3f} 与设定 {WARMUP_RATIO} 相差过大"
        f"（warmup_steps={args.warmup_steps} / max_steps={_max_steps}）：口径漂移，拒绝开训")
    print(f"   口径自检 ✓ 有效 batch = {_bs}×{_ga} = {_bs * _ga} | "
          f"每轮批数 {_loader} | 总更新步 {_max_steps} | "
          f"warmup {args.warmup_steps} 步（占 {_wf:.1%}）")


def build_training_args(output_dir, n_train, epochs=None, **overrides):
    _epochs = int(epochs) if epochs else int(EPOCHS)
    steps_per_epoch = math.ceil(n_train / (BATCH_SIZE * GRAD_ACCUM))
    total_steps = steps_per_epoch * _epochs
    base = dict(
        output_dir=str(output_dir),
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        num_train_epochs=_epochs,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=max(1, int(total_steps * WARMUP_RATIO)),
        lr_scheduler_type="linear",






        logging_steps=max(1, steps_per_epoch // 8),
        save_strategy="epoch",
        save_total_limit=1,




        load_best_model_at_end=True,




        metric_for_best_model=BEST_METRIC_KEY,
        greater_is_better=True,
        remove_unused_columns=False,
        dataloader_pin_memory=(DEVICE == "cuda"),
        dataloader_num_workers=NUM_WORKERS,
        dataloader_persistent_workers=DATALOADER_PERSISTENT,
        dataloader_multiprocessing_context=None if NUM_WORKERS == 0 else "fork",
        dataloader_prefetch_factor=None if NUM_WORKERS == 0 else 4,
        seed=SEED,
        data_seed=SEED,
        report_to=[],
        disable_tqdm=True,
        bf16=USE_BF16,
        fp16=USE_FP16,
        optim=OPTIM,
        gradient_checkpointing=USE_GRAD_CKPT,
    )
    base[_EVAL_STRATEGY_KEY] = "epoch"
    if _HAS_SAVE_ONLY_MODEL:
        base["save_only_model"] = True
    base.update(overrides)






    if base.get("gradient_checkpointing") and "gradient_checkpointing_kwargs" in _TA_PARAMS:
        base["gradient_checkpointing_kwargs"] = {"use_reentrant": False}
    args = TrainingArguments(**base)












    try:
        args._n_gpu = 1
    except Exception:
        pass
    return args

## Step7 创建训练器

**这一节要解决什么**：写出损失函数，并把它接进 `Trainer`。

### ★ v3.1 的核心：三个官方指标各自都有一个对应的损失项

官方评测有三项指标。v2 的损失只正经对着其中两项（而且第二项方向是错的），
v3.1 把三项**逐一**落到具体的损失项上：

| 官方指标 | 对应的损失项 | 项的形式 | 为什么这样对应 |
|---|---|---|---|
| **指标 1** 组合性相关 ρ（**主指标**） | `L_huber`（绝对值锚）+ `L_order`（组内成对排序） | 绝对误差 / 组内成对符号 | ρ 是**秩**相关：跨目标的相对大小由 Huber 承担（目标间差异 ≫ 标签噪声），组内的相对顺序由**只取符号**的排序项承担（排序对独立噪声不敏感 ⇒ 是「去噪后的信号」） |
| **指标 2** 语境方差相关 ρ（次要指标 / **非 nan 红线**） | `L_guard`（反收缩 hinge）+ `L_xrank`（跨目标 std 的**序**） | 下界 hinge / 序比较 | 指标 1 与指标 3 的最优解都是「收缩到条件均值」⇒ 会**系统性地压平组内离散度**、伤指标 2 ⇒ 必须有一项**反向**力量。`L_xrank` 直接优化「跨目标 std 的大小关系」，正是 Spearman 的可微形式 |
| **指标 3** RMSE | `L_mse`（显式二次项） | 平方误差 | Huber(δ=1) 在大误差处走线性段（抗噪），而 RMSE 是纯二次 ⇒ 指标 3 只认二次，给它一个专属项（权重小，不压过指标 1/2） |

### ★ v4 的两个新机制项（各挂在一个基线上）

v4 不再是「同配置换模型」的对照实验，而是**两个独立的机制实验**，
所以损失项里多了两项，且**互斥**地只在一个基线上开启：

| 基线 | 新项 | 形式 | 作用范围 | 为什么 |
|---|---|---|---|---|
| **DeBERTa** | `L_softcl`（软标签对比学习） | `(cos(z_i,z_j) − w_ij)²`，`w_ij = cos(π·|Δy|/5)` | **仅 pv** | `L_order` 只用**符号**（丢掉了「差多少」）、`L_huber` 被标注噪声主导；`L_softcl` 用**标签之间的距离**做**成对**监督 ⇒ 对噪声更稳、且保留「差多少」。而 nn 的组内差异 **93%–129% 是标注噪声**（实测噪声分解）⇒ 在 nn 上做这件事等于拟合噪声 ⇒ 关掉 |
| **Qwen** | `L_ce`（数字 token 软目标交叉熵） | `−Σ_k q_k·log softmax(logits)_k`，`q_k ∝ exp(−(k−y)²/2σ²)` | 四个子集 | 无回归头时模型唯一的输出通路是 LM 头给出的「下一个 token 是数字」的分布。**期望值层**（huber）管「分数对不对」，**分布层**（本项）管「分布形状对不对」——只有前者时，模型可以用尖锐的双峰去凑期望值 |

**为什么 `L_softcl` 不是 InfoNCE（SupCon 软标签版）**：那个形式有两个反向的坑 ——
① 按锚点归一化 `q = w/Σw` 会抹掉「确信度」，让**离群样本被主动拉向簇中心**；
② batch 内 softmax 对「把正例概率压到 0」的惩罚是无界的（`log p → −∞`），
而软标签权重只有 0.02–0.13 ⇒ 数值上「全挤在一起」反而更划算。
⇒ 改成**有界的相似度回归**：目标 `w_ij = cos(π·|Δy|/5) ∈ [−1,1]`
（**必须带符号**，否则 `cos=+1` 与 `cos=−1` 的代价几乎一样，
分不出「正交」与「重合」，「标签相远就分开」这条语义直接失效）。

**为什么 `L_ce` 的最小值不是 0**：软目标交叉熵的最小值是**目标分布的熵 H(q)**，
只有硬标签且完全预测正确时才是 0 ⇒ 单测里断言的是 `CE == H(q)`（实测 1.0592）。

### ★ 为什么按子集差异化（`SUBSET_LOSS_PROFILE`）

指标 2 的**现实天花板**由蒙特卡洛给出（`analysis/mc_metric2_ceiling.py`）：

| 设置 | m2_perfect | m2_realistic | v2 Qwen joint 实测 |
|---|---|---|---|
| en-nn/ModAvg | 0.176 | 0.052 | 0.149 |
| en-nn/HeadAvg | 0.235 | 0.081 | 0.271 |
| de-nn/ModAvg | 0.002 | 0.001 | 0.363 |
| de-nn/HeadAvg | 0.001 | 0.002 | 0.317 |
| en-pv/Avg | 0.573 | 0.413 | 0.589 |
| de-pv/Avg | 0.437 | 0.257 | 0.500 |

机制（`analysis/check_m2_mechanism.py`）：gold 的组内 std 有 **40%–74% 由「目标级歧义度」决定**
（这个目标本身让评分者分歧），而不是语境效应；组内 std 的 split-half 可复现性只有 **21%–39%**。

⇒ **nn 上追指标 2 基本无望**（连完美模型只有 0.00–0.24）⇒ 那里只用很松的 κ 做「防塌陷」；
⇒ **pv 上天花板 0.26–0.57，可达** ⇒ 值得用更紧的 κ + 跨目标 std 序；
⇒ 指标 2 的红线含义是「**非 nan**」，不是「数值高」（v2 实测 5/6 设置超过 realistic 天花板
⇒ 那是对噪声结构的偶然对齐，不可复现、不能当优化目标）。

### ★ 为什么删掉 v2 的「组内 std 等值匹配」（`MU_STD`）

- 标签构造已被精确反解 ⇒ 人工的目标级 std 里**掺着标注噪声**：
  `Var(gold_std) = true_var + mean(Std²/10)/k`，实测噪声把人工 std **膨胀 2.7%–13.6%**。
  等值匹配 = 要求模型去补上那部分**不可复现**的噪声 ⇒ **方向错**：
  预测被过度离散化 ⇒ 指标 1 与 RMSE 一起变差。
- v2 实测完全吻合：`MU_STD=0.05` **指标 2 零收益**（Qwen joint −0.022，落在噪声内），
  却让 DeBERTa **慢了 21–40%** —— 典型的「方向错 + 代价真实」。
- **更根本的原因：batch 内根本凑不出跨目标对**。有效 batch 只有 8，
  而 grouped sampler 让同一目标的例句连着排 ⇒ **一个 batch 里通常只有 1~2 个目标**，
  「组内 std 失配」这个量在 batch 内既稀疏又高噪声 ⇒ 它不是一个可优化的目标。
  v3.1 的 `L_xrank` 用「跨 batch 累积统计」解决了这个约束。

### ★ 一次自我更正：不要用「人工 std 掺噪」论证指标 2 不可追

此前依据「人工 std = sqrt(真实² + 噪声²/k) ⇒ 模型无法复现噪声 ⇒ 指标 2 不可追」
把指标 2 降级成「只保底」。**这个论证是错的**，两处：

1. 「去噪 std 与人工 std 的秩相关 ≈ 0.97–0.9996」是**循环论证**（「去噪 std」本来就是从人工 std 导出的）；
2. 真正决定天花板的是**目标级歧义度**（40%–74% 的组内 std 由它决定）
   + 极低的 split-half 可复现性（21%–39%）
   ⇒ 现实天花板是 **nn 0.00–0.08、pv 0.26–0.41**。

⇒ 结论不是「指标 2 完全不可追」，而是「**nn 上不可追、pv 上可追**」
⇒ 这正是 `SUBSET_LOSS_PROFILE` 要按子集差异化的原因。
⇒ 标签噪声确实把人工 std 抬高了 2.7%–13.6%（该结论有效），
  但它决定的是「**κ 该定多少**」，不是「能不能追」。

### ★ 为什么权重要「按占比自动标定」

v2 的 `MU_STD=0.05` 是**猜**出来的：太小没效果、太大压过主项，
而且同一个数在不同 batch / 子集 / 阶段对应的实际占比完全不同。
v3.1 用 `SubsetLossBalancer` 把权重定义成「**该项应占总损失的百分之几**」，
再用各项的实时量级（EMA）在线反解实际权重 ⇒ 没有「猜」的成分。
（比 v3 更进一步：**每个子集、每个标签列各有自己的 EMA**，
否则联合运行时「nn 的项」与「pv 的项」会互相污染量级估计，实际占比彻底失控。）

### ★ 为什么实现要「少同步」

v2 的 `group_std_loss` 里 `torch.unique`（变长输出）+ 布尔掩码索引 + `.item()`，
每组约 3 次 GPU→CPU 同步、每步 4–6 组，打断了 GPU 流水 —— 这是 DeBERTa 慢 21–40% 的直接原因。
v3.1 改用 `index_add_`（输出长度固定）+ 张量化的加权求和
⇒ 热路径只剩 **2 次同步**（`torch.unique` 取当前组、`any()` 取历史是否非空）。

### ★ 打开组内项之前必须有的护栏

小品词动词的第 2 列是**占位的常数 0**（`loss_mask=0`），它的「人工组内 std」恒为 0。
若不跳过这一列，组内项会强行把模型第 2 列的组内离散度**压成 0**，
而且这个梯度会经共享主干回流、干扰真正要学的第 1 列。
所以损失项一律接收 `valid`（这一列有没有真标签），无效的组整体跳过。

**为什么只覆写 `compute_loss` 这一个方法**：`Trainer` 把「损失怎么算」和
「怎么优化、怎么调度、怎么保存、怎么评估」分得很干净。
前向、反向、梯度累积、日志、评估、保存全由框架负责，我们只替换损失这一环。

In [ ]:

N_SUB = len(SUBSET_ORDER)


def profile_vectors(device, dtype=torch.float32):
    def vec(key):
        return torch.tensor([float(SUBSET_LOSS_PROFILE[n].get(key, 0.0)) for n in SUBSET_ORDER],
                            device=device, dtype=dtype)



    return dict(order=vec("order"), mse=vec("mse"), guard=vec("guard"),
                xrank=vec("xrank"), kappa=vec("kappa"),
                ce=vec("ce"), softcl=vec("softcl"))


def _agg_by_subset(values, index, n_sub):
    return torch.zeros(n_sub, device=values.device, dtype=values.dtype).index_add_(0, index, values)


def order_parts(pred, gold, group_index, sidx, tau, valid, order_vec, n_sub):
    B = pred.shape[0]
    iu = torch.triu(torch.ones(B, B, dtype=torch.bool, device=pred.device), diagonal=1)
    mf = iu & (group_index[:, None] == group_index[None, :])
    if valid is not None:
        ok = valid > 0
        mf = mf & ok[:, None] & ok[None, :]
    diff_p = pred[:, None] - pred[None, :]
    diff_y = gold[:, None] - gold[None, :]
    sign = torch.sign(diff_y)
    mf = mf & (sign != 0)
    mff = mf.to(pred.dtype)
    w = order_vec[sidx]
    num = F.softplus(-sign * diff_p / tau) * mff * w[:, None]
    den = mff * w[:, None]
    idx = sidx[:, None].expand(B, B).reshape(-1)
    return _agg_by_subset(num.reshape(-1), idx, n_sub), _agg_by_subset(den.reshape(-1), idx, n_sub)


def mse_parts(pred, gold, valid, sidx, mse_vec, n_sub):
    w = mse_vec[sidx]
    if valid is not None:
        w = w * valid.to(pred.dtype)
    num = (pred - gold) ** 2 * w
    return _agg_by_subset(num, sidx, n_sub), _agg_by_subset(w, sidx, n_sub)


def guard_parts(std_pred, std_gold, cnt, group_subset, kappa_vec, n_sub):
    kg = kappa_vec[group_subset]
    ok = (cnt >= 2) & (kg > 0)
    okf = ok.to(std_pred.dtype)
    gap = (kg * std_gold - std_pred).clamp(min=0.0)
    num = gap * gap * okf
    return _agg_by_subset(num, group_subset, n_sub), _agg_by_subset(okf, group_subset, n_sub)


def ce_parts(digit_logits, gold, valid, sidx, ce_vec, n_sub, sigma=None):
    if digit_logits.dim() == 2:
        digit_logits = digit_logits.unsqueeze(1)
    if gold.dim() == 1:
        gold = gold.unsqueeze(1)
    if sigma is None:
        sigma = CE_SOFT_SIGMA
    B, C, K = digit_logits.shape
    k = torch.arange(K, device=digit_logits.device, dtype=digit_logits.dtype).view(1, 1, K)
    y = gold.unsqueeze(-1)
    q = torch.exp(-0.5 * ((k - y) ** 2) / (sigma ** 2))
    q = q / q.sum(dim=-1, keepdim=True).clamp(min=1e-9)
    logp = F.log_softmax(digit_logits, dim=-1)
    ce = -(q * logp).sum(dim=-1)
    w = ce_vec[sidx].unsqueeze(1)
    if valid is not None:
        v = valid.to(ce.dtype)
        if v.dim() == 1:
            v = v.unsqueeze(1)
        w = w * v

    idx = sidx.unsqueeze(1).expand(B, C).reshape(-1)
    num = _agg_by_subset((ce * w).reshape(-1), idx, n_sub)
    den = _agg_by_subset(w.reshape(-1), idx, n_sub)
    return num, den


def softcl_parts(z, sample_label, sidx, label_span, softcl_vec, n_sub):
    B = z.shape[0]
    eye = torch.eye(B, dtype=torch.bool, device=z.device)
    same_sub = sidx[:, None] == sidx[None, :]
    m = (~eye) & same_sub
    diff = (sample_label[:, None] - sample_label[None, :]).abs()
    w = torch.cos(math.pi * diff / float(label_span))
    sim = z @ z.t()
    err = (sim - w) ** 2
    wf = softcl_vec[sidx].unsqueeze(1)
    fm = m.to(z.dtype)
    num = _agg_by_subset((err * fm * wf).reshape(-1),
                         sidx.unsqueeze(1).expand(B, B).reshape(-1), n_sub)
    den = _agg_by_subset((fm * wf).reshape(-1),
                         sidx.unsqueeze(1).expand(B, B).reshape(-1), n_sub)
    return num, den


def xrank_parts(std_pred, std_gold, cur_ids, group_subset, xrank_vec, margin, tau, hist_mask, n_sub):
    N = torch.zeros(n_sub, device=std_pred.device, dtype=std_pred.dtype)
    D = torch.zeros(n_sub, device=std_pred.device, dtype=std_pred.dtype)
    G = std_pred.numel()
    ar = torch.arange(G, device=std_pred.device)
    for g in cur_ids:
        sg = std_gold[g]
        sp = std_pred[g]
        sub_g = group_subset[g]
        allowed = (group_subset == sub_g) & hist_mask & (ar != g) & ((std_gold - sg).abs() > margin)
        af = allowed.to(std_pred.dtype)
        sign = torch.sign(sg - std_gold)
        pen = F.softplus(-sign * (sp - std_pred) / tau)
        w = xrank_vec[sub_g].to(std_pred.dtype)
        idx = sub_g.repeat(G)
        N = N.index_add_(0, idx, pen * af * w)
        D = D.index_add_(0, idx, af * w)
    return N, D

In [ ]:





toyl = torch.tensor([[1.0], [4.0], [2.0], [5.0], [1.0], [3.0]])
toyp = torch.tensor([[1.5], [3.0], [2.2], [4.0], [1.3], [1.7]])
toyg = torch.tensor([0, 0, 1, 1, 2, 2])
toys = torch.tensor([0, 0, 2, 2, 2, 2])
toyv = torch.ones(6)

_prof = profile_vectors(torch.device("cpu"), torch.float32)
print("各子集的损失配置（占比 / κ）：")
for _s in range(len(SUBSET_ORDER)):
    _p = SUBSET_LOSS_PROFILE[SUBSET_ORDER[_s]]
    print(f"  {SUBSET_ORDER[_s]:6s} order={_p['order']:.2f}  mse={_p['mse']:.2f}  "
          f"guard={_p['guard']:.2f}  xrank={_p['xrank']:.2f}  kappa={_p['kappa']:.2f}")

_N, _D = order_parts(toyp[:, 0], toyl[:, 0], toyg, toys, ORDER_TAU, toyv, _prof["order"], 4)
_l_order = _N / _D.clamp(min=1.0)
_N, _D = mse_parts(toyp[:, 0], toyl[:, 0], toyv, toys, _prof["mse"], 4)
_l_mse = _N / _D.clamp(min=1.0)

_gold_g = [[1.0, 4.0], [2.0, 5.0], [1.0, 3.0]]
_pred_g = [[1.5, 3.0], [2.2, 4.0], [1.3, 1.7]]
_std_g = torch.tensor([float(torch.std(torch.tensor(g), unbiased=True)) for g in _gold_g])
_std_p = torch.tensor([float(torch.std(torch.tensor(g), unbiased=True)) for g in _pred_g])
_cnt = torch.tensor([2.0, 2.0, 2.0])
_gsub = torch.tensor([0, 2, 2])
_N, _D = guard_parts(_std_p, _std_g, _cnt, _gsub, _prof["kappa"], 4)
_l_guard = _N / _D.clamp(min=1.0)
_N, _D = xrank_parts(_std_p, _std_g, torch.tensor([0, 1, 2]), _gsub, _prof["xrank"],
                      XRANK_MARGIN, XRANK_TAU, torch.ones(3, dtype=torch.bool), 4)
_l_xrank = _N / _D.clamp(min=1.0)

print()
print(f"L_huber = {F.huber_loss(toyp, toyl, delta=HUBER_DELTA).item():.4f}   "
      f"绝对值误差（指标 1 的锚 / 指标 3 的主体）")
print(f"L_order  (en-nn / en-pv) = {_l_order[0].item():.4f} / {_l_order[2].item():.4f}   "
      f"组内相对顺序（目标0真实 1<4，预测 1.5>3.0 → 被罚）")
print(f"L_mse    (en-nn / en-pv) = {_l_mse[0].item():.4f} / {_l_mse[2].item():.4f}   "
      f"指标 3（RMSE）的显式二次项")
print(f"L_guard  (en-nn / en-pv) = {_l_guard[0].item():.4f} / {_l_guard[2].item():.4f}   "
      f"反收缩下界（en-nn κ=0.15 → 下界 0.32 < 预测 1.06，不触发；"
      f"en-pv κ=0.50 → 下界 0.71 > 预测 0.28 → 被罚）")
print(f"L_xrank  (en-nn / en-pv) = {_l_xrank[0].item():.4f} / {_l_xrank[2].item():.4f}   "
      f"跨目标 std 的序（en-nn 的 xrank 占比=0 ⇒ 该子集关闭；"
      f"en-pv 在组 1/2 之间比较 ⇒ 有值）")
print()
print("→ 绝对值项是锚；排序项纠正组内顺序；MSE 项对齐 RMSE；")
print("   guard 项**反向**对抗「指标 1/3 把组内方差压平」—— 这正是指标 2 被伤害的根源；")
print("   xrank 项只在 pv 上开（nn 的指标 2 天花板只有 0.00–0.08，强行优化只会拟合噪声结构）。")
print("→ v2 的「组内 std 等值匹配」已被删除：人工 std 里掺着不可复现的标注噪声，")
print("   等值匹配 = 要求模型去拟合噪声（实测零收益且拖慢 21–40%）。")


print()
print("--- v4 新增的两个损失项 ---")
_k = torch.arange(6, dtype=torch.float32)

_dl_good = -((_k.view(1, -1) - toyl.view(-1, 1)) ** 2) / (2 * CE_SOFT_SIGMA ** 2)
_dl_bad = -((_k.view(1, -1) - (toyl.view(-1, 1) + 2.0)) ** 2) / (2 * CE_SOFT_SIGMA ** 2)
_N, _D = ce_parts(_dl_good.unsqueeze(1), toyl, toyv, toys, _prof["ce"], 4)
_ce_good = (_N / _D.clamp(min=1.0))[2].item()
_N, _D = ce_parts(_dl_bad.unsqueeze(1), toyl, toyv, toys, _prof["ce"], 4)
_ce_bad = (_N / _D.clamp(min=1.0))[2].item()
print(f"L_ce    (en-pv) = {_ce_good:.4f}（分布正好落在标签上） / "
      f"{_ce_bad:.4f}（分布整体偏 2 分）   数字 token 上的软目标交叉熵")

_ang = toyl[:, 0] / 5.0 * torch.pi
_z_good = F.normalize(torch.stack([torch.cos(_ang), torch.sin(_ang)], dim=1), dim=1)
_z_bad = F.normalize(torch.stack([torch.cos(toys.float()), torch.sin(toys.float())], dim=1), dim=1)
_N, _D = softcl_parts(_z_good, toyl[:, 0], toys, SOFTCL_LABEL_SPAN, _prof["softcl"], 4)
_l_sc_good = (_N / _D.clamp(min=1.0))[2].item()
_N, _D = softcl_parts(_z_bad, toyl[:, 0], toys, SOFTCL_LABEL_SPAN, _prof["softcl"], 4)
_l_sc_bad = (_N / _D.clamp(min=1.0))[2].item()
print(f"L_softcl(en-pv) = {_l_sc_good:.4f}（表征按标签排布） / "
      f"{_l_sc_bad:.4f}（表征与标签无关）   软标签相似度回归")
print("→ ce 管「分数分布的形状」，softcl 管「表征之间的相似度」；")
print("   softcl 的目标很直白：cos(z_i, z_j) 应当等于 exp(-|y_i-y_j|/τ_label)；")
print("   它只在 pv 上开（nn 的组内差异九成以上是标注噪声，在那里做等于拟合噪声）。")

In [ ]:
def stats_dtype_for(device):
    try:
        torch.zeros(1, dtype=torch.float64, device=device)
        return torch.float64
    except Exception:
        return torch.float32


class GroupStats:

    def __init__(self, n_groups, device, dtype=None, decay=0.8):
        self.n = int(n_groups)
        self.decay = float(decay)
        self.device = device
        if dtype is None:
            dtype = stats_dtype_for(device)
        self.dtype = dtype
        self.p_cnt = torch.zeros(self.n, device=device, dtype=dtype)
        self.p_s1 = torch.zeros(self.n, device=device, dtype=dtype)
        self.p_s2 = torch.zeros(self.n, device=device, dtype=dtype)
        self.g_cnt = torch.zeros(self.n, device=device, dtype=dtype)
        self.g_s1 = torch.zeros(self.n, device=device, dtype=dtype)
        self.g_s2 = torch.zeros(self.n, device=device, dtype=dtype)

    def _batch_stats(self, x, gidx, valid):
        xd = x.to(self.dtype)
        vd = valid.to(self.dtype)
        cnt = torch.zeros(self.n, device=x.device, dtype=self.dtype)
        s1 = torch.zeros(self.n, device=x.device, dtype=self.dtype)
        s2 = torch.zeros(self.n, device=x.device, dtype=self.dtype)
        cnt.index_add_(0, gidx, vd)
        s1.index_add_(0, gidx, vd * xd)
        s2.index_add_(0, gidx, vd * xd * xd)
        return cnt, s1, s2

    def stats(self, pred, gold, gidx, valid):
        pc, ps1, ps2 = self._batch_stats(pred, gidx, valid)
        gc, gs1, gs2 = self._batch_stats(gold.detach(), gidx, valid)
        return (self.p_cnt + pc, self.p_s1 + ps1, self.p_s2 + ps2,
                self.g_cnt + gc, self.g_s1 + gs1, self.g_s2 + gs2)

    def commit(self, pred, gold, gidx, valid):
        pc, ps1, ps2 = self._batch_stats(pred, gidx, valid)
        gc, gs1, gs2 = self._batch_stats(gold.detach(), gidx, valid)
        d = self.decay
        self.p_cnt = d * self.p_cnt + pc.detach()
        self.p_s1 = d * self.p_s1 + ps1.detach()
        self.p_s2 = d * self.p_s2 + ps2.detach()
        self.g_cnt = d * self.g_cnt + gc.detach()
        self.g_s1 = d * self.g_s1 + gs1.detach()
        self.g_s2 = d * self.g_s2 + gs2.detach()


def std_from_moments(cnt, s1, s2, out_dtype=None):
    n = cnt.clamp(min=1.0)
    mean = s1 / n
    var = (s2 / n - mean * mean) * (cnt / (cnt - 1.0).clamp(min=1.0))
    var = var.clamp(min=0.0)
    safe = torch.sqrt(var + 1e-12)
    out = safe * (cnt >= 2).to(var.dtype)
    out = torch.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)
    if out_dtype is not None:
        out = out.to(out_dtype)
    return out


class SubsetLossBalancer:

    MAX_WEIGHT = 5.0

    def __init__(self, n_sub, mix, beta=BALANCE_EMA_BETA, device="cpu"):
        self.n = int(n_sub)
        self.beta = float(beta)
        self.mix = {k: v.to(device) for k, v in mix.items()}
        self.main_ema = torch.zeros(self.n, device=device)
        self.term_ema = {k: torch.zeros(self.n, device=device) for k in mix}
        self.weights = {}





        self.w_ema = {k: torch.zeros(self.n, device=device) for k in mix}

    def update_main(self, main_vec, active):
        new = self.beta * self.main_ema + (1.0 - self.beta) * main_vec * active
        self.main_ema = torch.where(active > 0, new, self.main_ema)
        return self.main_ema

    def weight(self, name, term_vec, active):
        frac = self.mix[name]
        ema = self.term_ema[name]
        new_ema = self.beta * ema + (1.0 - self.beta) * term_vec * active
        ema = torch.where(active > 0, new_ema, ema)
        self.term_ema[name] = ema
        w = (frac / (1.0 - frac).clamp(min=1e-3)) * self.main_ema / (ema + 1e-6)
        w = w.clamp(max=self.MAX_WEIGHT).detach()
        w = torch.where(active > 0, w, torch.zeros_like(w))
        self.weights[name] = w

        _we = self.w_ema[name]
        _we = torch.where(active > 0, self.beta * _we + (1.0 - self.beta) * w, _we)
        self.w_ema[name] = _we
        return w


class CompartmentTrainer(Trainer):

    def __init__(self, *args, grouped_batching=True, n_groups=None, group_subset=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.grouped_batching = grouped_batching
        self.n_groups = n_groups



        self._gs = [GroupStats(n_groups or 1, torch.device("cpu"))
                    if n_groups else None for _ in range(N_OUTPUTS)]
        self._gs_ready = False

        self._group_subset = group_subset
        self._gsub_t = None
        self._prof = None
        self._bals = None

        self._step = 0
        self._finite_check_every = int(LOSS_FINITE_CHECK_EVERY)

    def _ensure_prof(self, device, dtype):
        if self._gs_ready:
            return

        _sdtype = stats_dtype_for(device)
        for st in self._gs:
            if st is not None:
                st.dtype = _sdtype
                st.p_cnt = st.p_cnt.to(device=device, dtype=_sdtype)
                st.p_s1 = st.p_s1.to(device=device, dtype=_sdtype)
                st.p_s2 = st.p_s2.to(device=device, dtype=_sdtype)
                st.g_cnt = st.g_cnt.to(device=device, dtype=_sdtype)
                st.g_s1 = st.g_s1.to(device=device, dtype=_sdtype)
                st.g_s2 = st.g_s2.to(device=device, dtype=_sdtype)
        self._prof = profile_vectors(device, dtype)
        mix = {k: self._prof[k] for k in ("order", "mse", "guard", "xrank", "ce")}

        self._bals = [SubsetLossBalancer(N_SUB, mix, BALANCE_EMA_BETA, device)
                      for _ in range(N_OUTPUTS)]


        self._bal_softcl = SubsetLossBalancer(N_SUB, {"softcl": self._prof["softcl"]},
                                               BALANCE_EMA_BETA, device)
        self._main_vec_col0 = None
        if self._group_subset is not None:
            self._gsub_t = torch.as_tensor(self._group_subset, device=device, dtype=torch.long)
        self._gs_ready = True


    def training_step(self, model, inputs, num_items_in_batch=None):
        loss = super().training_step(model, inputs, num_items_in_batch=num_items_in_batch)
        _m = model if hasattr(model, "flush_readout_grad") else self.model
        if hasattr(_m, "flush_readout_grad"):
            _m.flush_readout_grad()
        return loss

    def _get_train_sampler(self, train_dataset=None):
        if not self.grouped_batching:
            return super()._get_train_sampler(train_dataset)
        ds = train_dataset if train_dataset is not None else self.train_dataset
        return TargetGroupedSampler(ds["group_index"], seed=self.args.seed)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs["labels"]
        mask = inputs.get("loss_mask")
        gidx = inputs.get("group_index")
        sidx = inputs.get("subset_index")


        _need2 = None
        if mask is not None:
            _need2 = (mask[:, 1] > 0)
        outputs = model(input_ids=inputs["input_ids"],
                        attention_mask=inputs["attention_mask"],
                        input_ids2=inputs.get("input_ids2"),
                        attention_mask2=inputs.get("attention_mask2"),
                        need2=_need2)
        logits = outputs.logits



        per = F.huber_loss(logits, labels, delta=HUBER_DELTA, reduction="none")
        if mask is not None:
            main = (per * mask).sum() / mask.sum().clamp(min=1.0)
        else:
            main = per.mean()
        loss = main
        if gidx is None or sidx is None or not self.n_groups:
            return (loss, {"logits": logits}) if return_outputs else loss
        self._ensure_prof(logits.device, logits.dtype)
        gsub = self._gsub_t

        cur_ids = torch.unique(gidx)
        hist_any = torch.stack([(st.g_cnt > 0).any() for st in self._gs]).any()
        for j in range(logits.shape[1]):



            _valid = (mask[:, j] if mask is not None else
                      torch.ones_like(logits[:, j]))
            _valid = _valid.to(logits.dtype)

            _mnum = _agg_by_subset(per[:, j] * _valid, sidx, N_SUB)
            _mden = _agg_by_subset(_valid, sidx, N_SUB)
            _main_vec = _mnum / _mden.clamp(min=1.0)
            _act = (_mden > 0).to(logits.dtype)
            bal = self._bals[j]
            bal.update_main(_main_vec, _act)
            if j == 0:
                self._main_vec_col0 = (_main_vec, _act)

            _share = _mden / _mden.sum().clamp(min=1.0)

            st = self._gs[j]
            cnt, s1, s2, gcnt, gs1, gs2 = st.stats(logits[:, j], labels[:, j], gidx, _valid)
            std_p = std_from_moments(cnt, s1, s2, out_dtype=logits.dtype)
            std_g = std_from_moments(gcnt, gs1, gs2, out_dtype=logits.dtype)
            for name in ("order", "mse", "guard", "xrank", "ce"):
                _frac = self._prof[name]
                if name == "order":
                    _N, _D = order_parts(logits[:, j], labels[:, j], gidx, sidx,
                                         ORDER_TAU, _valid, _frac, N_SUB)
                elif name == "mse":
                    _N, _D = mse_parts(logits[:, j], labels[:, j], _valid, sidx, _frac, N_SUB)
                elif name == "guard":
                    _N, _D = guard_parts(std_p, std_g, cnt, gsub, self._prof["kappa"], N_SUB)
                elif name == "ce":

                    _dl = getattr(outputs, "digit_logits", None)
                    if _dl is None:
                        continue
                    _N, _D = ce_parts(_dl[:, j, :], labels[:, j], _valid, sidx, _frac, N_SUB)
                else:
                    if cur_ids.numel() == 0 or not bool(hist_any):
                        continue



                    _hist_ok = (st.g_cnt > 0) & (gcnt >= 2)
                    _N, _D = xrank_parts(std_p, std_g, cur_ids, gsub, _frac,
                                         XRANK_MARGIN, XRANK_TAU, _hist_ok, N_SUB)
                _tv = _N / _D.clamp(min=1.0)
                _tact = (_D > 0).to(logits.dtype)
                if AUTO_BALANCE:
                    _w = bal.weight(name, _tv, _tact)
                else:
                    _w = torch.ones(N_SUB, device=logits.device)






                _cap = (_frac / (1.0 - _frac).clamp(min=1e-3)) * _main_vec * _share
                _contrib = torch.minimum(_w * _tv * _tact, _cap)
                loss = loss + _contrib.sum()

            st.commit(logits[:, j], labels[:, j], gidx, _valid)






        _z = getattr(outputs, "proj", None)
        if _z is not None and self._bal_softcl is not None:
            _frac_sc = self._prof["softcl"]
            if float(_frac_sc.max()) > 0:

                if mask is not None:
                    _mlab = (labels * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)
                else:
                    _mlab = labels.mean(dim=1)
                _N, _D = softcl_parts(_z, _mlab, sidx, SOFTCL_LABEL_SPAN,
                                      _frac_sc, N_SUB)
                _tv = _N / _D.clamp(min=1.0)
                _tact = (_D > 0).to(logits.dtype)
                if self._main_vec_col0 is not None:
                    _mv0, _act0 = self._main_vec_col0
                    self._bal_softcl.update_main(_mv0, _act0)
                    if AUTO_BALANCE:
                        _w = self._bal_softcl.weight("softcl", _tv, _tact)
                    else:
                        _w = torch.ones(N_SUB, device=logits.device)
                    _share0 = _act0 / _act0.sum().clamp(min=1.0)
                    _cap = (_frac_sc / (1.0 - _frac_sc).clamp(min=1e-3)) * _mv0 * _share0
                    loss = loss + torch.minimum(_w * _tv * _tact, _cap).sum()








        self._step += 1
        if self._finite_check_every > 0 and self._step % self._finite_check_every == 0:
            if not bool(torch.isfinite(loss.detach())):
                raise RuntimeError(
                    f"训练第 {self._step} 步的 loss 变成 NaN/Inf（当前档位无 AMP ⇒ 没有 GradScaler "
                    "兜底，坏梯度会直接写进权重）。请检查：该步的组内 std 是否退化"
                    "（analysis/diagnose_std_gradient.py 可复现 fp32 抵消误差）、"
                    "以及是否有目标只带 1 条例句（en-nn 实测存在 1 个这样的目标）。")
        return (loss, {"logits": logits}) if return_outputs else loss

    def loss_weight_snapshot(self):
        out = {}
        for j, bal in enumerate(self._bals or []):
            for k, v in bal.w_ema.items():
                for s in range(v.numel()):
                    out[f"w_{k}[col{j}·{SUBSET_ORDER[s]}]"] = round(float(v[s]), 5)
        if self._bal_softcl is not None:
            for k, v in self._bal_softcl.w_ema.items():
                for s in range(v.numel()):
                    out[f"w_{k}[sample·{SUBSET_ORDER[s]}]"] = round(float(v[s]), 5)
        return out


class TargetGroupedSampler(Sampler):
    def __init__(self, group_index, seed=SEED, shuffle=True):
        self.groups = np.asarray(group_index)
        self.seed = seed
        self.shuffle = shuffle
        self.epoch = 0
        self._indices = self._build()

    def _build(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        uniq = np.unique(self.groups)
        if self.shuffle:
            rng.shuffle(uniq)
        order = []
        for g in uniq:
            idx = np.where(self.groups == g)[0]
            if self.shuffle:
                rng.shuffle(idx)
            order.extend(idx.tolist())
        return order

    def __iter__(self):
        return iter(self._indices)

    def __len__(self):
        return len(self._indices)

    def set_epoch(self, epoch):
        self.epoch = epoch
        self._indices = self._build()


print("CompartmentTrainer / TargetGroupedSampler 已定义")

## Step8 模型训练（自动跑完全部运行）

**这一节不需要任何手动切换**：它按 Step1 的 `RUN_PLAN` 自动依次执行
「四个子集各单独训练一次」+「四个子集联合训练一次」，共 5 次。

**每一次运行都做四件事**（顺序不能换）：

1. **准备数据**：按本次参与的子集切分、构造 Dataset（`prepare_run`）。
2. **训练前评估随机初始化模型** —— 给 Step9 的「参考对照」区提供一条基线。
   少了它，「训练后 ρ=0.6」无法回答「它到底学到了什么」。
3. **正式训练**。
4. **训练后评估**：与随机初始化**完全相同的口径**（同一份验证集、同一个
   `compute_metrics`），唯一区别是模型权重。口径不对齐的对照没有意义。

**第一轮额外做三项开训前检查**（只做一次，因为它们与数据规模无关）：

- **显存探测**：逐档真跑 2 步，量峰值显存，取能跑通的最大 micro-batch，
  再用梯度累积把有效 batch 补回 8。
- **梯度流验证**：手动 `backward()` 一次，打印「有多少参数真的拿到了梯度」。
  能抓到的典型 bug 是「主干被冻结 / 回归头没接上」——这些**不会报错**，
  只会让模型训半天纹丝不动。
- **冒烟测试**：`max_steps=5` 跑通「前向 → 损失 → 反向 → 参数更新 → 评估」
  整条链路。直接开长训，一旦中途报错，要么白等几十分钟，要么分不清是 bug 还是超时。

**为什么每次运行都要重建模型**：冒烟测试那 5 步、显存探测那 2 步探针都会改动权重。
正式训练必须从**干净的随机初始化**开始，否则基线不干净、不可复现。

In [ ]:

def should_check_consistency(overrides, check_consistency=None):
    if check_consistency is not None:
        return bool(check_consistency)
    return not (("max_steps" in overrides) or ("num_train_epochs" in overrides))


def make_trainer(m, out_dir, data, epochs=None, check_consistency=None, **overrides):

    _probed = ("max_steps" in overrides) or ("num_train_epochs" in overrides)
    check_consistency = should_check_consistency(overrides, check_consistency)
    if _probed and check_consistency is not True:
        print(f"   （体检用途的 Trainer：覆盖了 {sorted(set(overrides) & {'max_steps', 'num_train_epochs'})} "
              "⇒ 自动关闭口径断言；正式训练路径仍保留）")
    _args = build_training_args(out_dir, len(data["train_ds"]), epochs=epochs, **overrides)






    _n_gpu = int(getattr(_args, "n_gpu", 1) or 1)
    assert _n_gpu == 1, (
        f"args.n_gpu={_n_gpu} > 1：Trainer 会自动套 nn.DataParallel，"
        "其反向传播会复制一整份梯度导致 OOM。请用 COMPARTMENT_GPU=<卡号> 只暴露一张卡。")







    if check_consistency:
        _assert_args_consistent(_args, n_train=len(data["train_ds"]))



    _n_groups = int(np.max(ds_int_col(data["train_ds"], "group_index"))) + 1



    _gs_map = np.zeros(_n_groups, dtype=np.int64)
    _gs_map[ds_int_col(data["train_ds"], "group_index")] = ds_int_col(data["train_ds"], "subset_index")
    assert _gs_map.max() < len(SUBSET_ORDER), (
        f"目标组的子集编号最大 {_gs_map.max()}，超出 SUBSET_ORDER 长度 {len(SUBSET_ORDER)}")

    return CompartmentTrainer(
        model=m,
        args=_args,
        train_dataset=data["train_ds"],
        eval_dataset=data["val_ds"],
        data_collator=collator,


        **{_TOKENIZER_KW: tokenizer},
        compute_metrics=make_compute_metrics(
            data["val_ds"]["group_index"], data["val_ds"]["subset_index"],
            data["eval_cols"], SUBSET_ORDER),







        grouped_batching=GROUPED_BATCHING,
        n_groups=_n_groups,
        group_subset=_gs_map,
    )
















class FitOOMError(RuntimeError):
    pass


class FitNumericError(RuntimeError):
    pass


def _is_oom(exc):
    return "out of memory" in str(exc).lower()


def release_gpu_memory(tag=""):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        _a = torch.cuda.memory_allocated() / (1024 ** 3)
        _r = torch.cuda.memory_reserved() / (1024 ** 3)
        _c = _vram_total_gb() or 0.0
        print(f"[显存·{tag or '释放'}] 已分配 {_a:.2f} GiB | 已保留 {_r:.2f} GiB"
              f" | 卡 {_c:.2f} GiB")
    else:
        print(f"[显存·{tag or '释放'}] 非 CUDA 环境，跳过")


def _probe_reserve_gb():
    try:
        return float(FIT_PROBE_RESERVE_GB)
    except Exception:
        return 0.8


def probe_fit_tier(model, data, tier, candidates=(8, 4, 2, 1), steps=2):
    if not torch.cuda.is_available():
        print("非 CUDA 环境（CPU/MPS）：跳过显存探测，沿用配置中的 batch size")
        return None
    _card = _vram_total_gb() or 0.0
    _reserve = _probe_reserve_gb()
    _fallback = None
    for bs in candidates:
        _tr = None
        try:
            _tr = make_trainer(model, OUTPUT_ROOT / "_probe", data,
                               per_device_train_batch_size=bs,






                               gradient_accumulation_steps=GRAD_ACCUM,
                               max_steps=steps,
                               save_strategy="no",
                               load_best_model_at_end=False,
                               logging_steps=steps,


                               check_consistency=False,
                               optim=tier["optim"],
                               gradient_checkpointing=tier["ckpt"],
                               dataloader_num_workers=0, dataloader_persistent_workers=False,
                               dataloader_multiprocessing_context=None,
                               dataloader_prefetch_factor=None,
                               **{_EVAL_STRATEGY_KEY: "no"})
            torch.cuda.reset_peak_memory_stats()
            _t0 = time.time()
            _tr.train()
            _dt = time.time() - _t0
            peak = torch.cuda.max_memory_allocated() / (1024 ** 3)







            _gn = [h.get("grad_norm") for h in _tr.state.log_history if h.get("grad_norm") is not None]
            _bad_gn = []
            for _g in _gn:
                try:
                    _gv = float(_g)
                except (TypeError, ValueError):
                    continue
                if not math.isfinite(_gv):
                    _bad_gn.append(_gv)
            if _bad_gn:
                raise FitNumericError(
                    f"探针阶段的梯度已经是 NaN/Inf（grad_norm={_bad_gn[:3]}），"
                    f"但 loss 与数据都正常 ⇒ 这是**档位的数值精度问题**"
                    f"（{'AMP/fp16 溢出' if tier.get('amp') else 'fp32 精度问题'}）而不是代码 bug。"
                    "处理方式：换下一个档位重试。"
                    "若全部档位都这样，再查：① 组内 std 的 fp32 抵消误差"
                    "（analysis/diagnose_std_gradient.py）；② 只有 1 条例句的目标"
                    "（en-nn 实测存在 1 个）。")


            sps = _dt / max(1, steps * bs)

            _need = peak + _reserve
            _fits = (_need <= _card)
            print(f"  {tier['name']} micro-batch={bs}: 通过，峰值显存 {peak:.2f} GiB，"
                  f"每样本 {sps * 1000:.2f} ms"
                  f"（+余量 {_reserve:.1f} GiB = {_need:.2f} GiB / 卡 {_card:.2f} GiB"
                  f" ⇒ {'余量充足 ✓' if _fits else '余量不足 ✗（但仍保留本档位）'}）")
            if _fits:
                return bs, peak, sps








            if _fallback is None:
                _fallback = (bs, peak, sps)
        except FitNumericError:
            raise
        except RuntimeError as e:
            if not _is_oom(e):
                raise


            _lines = [l for l in str(e).strip().splitlines() if l.strip()]
            _head = _lines[0][:150] if _lines else ""
            _detail = next((l for l in _lines if "Tried to allocate" in l or "GiB is free" in l), "")
            print(f"  {tier['name']} micro-batch={bs}: OOM | {_head}")
            if _detail:
                print(f"      {_detail.strip()[:220]}")
        finally:
            if _tr is not None:
                del _tr
            gc.collect()
            torch.cuda.empty_cache()

    if _fallback is not None:
        _b, _p, _s = _fallback
        print(f"  ⚠️ 档位 {tier['name']}：没有哪个 batch 能同时满足「余量 {_reserve:.1f} GiB」，"
              f"但 micro-batch={_b} 跑得通（峰值 {_p:.2f} GiB / 卡 {_card:.2f} GiB）")
        print("     ⇒ **保留本档位**（余量不足不是降档的理由：Qwen 的峰值由固定开销主导，"
              "降 micro-batch 降不下峰值；降档会让两条基线不同档 ⇒ 对照结论作废）")
        print(f"     ⇒ 采用**能跑通的最大 batch**={_b}（峰值由固定开销主导，降 batch 换不来多少余量，"
              "却要赔上好几倍速度；真 OOM 了有「正式训练自动重试」兜底）")
        print(f"     ⇒ 后续的 preflight 已做三项降峰值处理（开 checkpointing / 样本数减半 / "
              f"释放计算图），并使用探测出的 micro-batch={_b}")
        return _fallback
    return None



def _total_train_rows():
    tot = 0
    for _name in SUBSET_ORDER:
        _p = DATA_ROOT / f"{_name}-train.tsv"
        with open(_p, "r", encoding="utf-8") as _f:
            tot += max(0, sum(1 for _ in _f) - 1)
    return tot


def project_train_time(sec_per_sample):
    n_rows = _total_train_rows()
    per_epoch = 0.8 * n_rows
    total = per_epoch * (EPOCHS + JOINT_EPOCHS)
    hours = total * sec_per_sample * 1.3 / 3600
    return n_rows, total, hours


def prepare_for_training(build_fn, data):
    vram = _vram_total_gb()
    tier0, _rows = resolve_fit_tier(vram, FIT_REF_PARAMS)
    start = FIT_TIERS.index(tier0)




    _locked = ""
    if FIT_MODE != "auto" and str(FIT_MODE).lower() == tier0["name"]:
        _locked = tier0["name"]
    if _locked:
        print(f"★ 档位已锁定为 {_locked}（FIT_MODE 显式指定）⇒ **不自动降档**")
        print("   （降档 = 冻住一半主干 = 换了实验；本版机制要求全参微调。"
              "若这一档真跑不通，请先查「有效 batch / micro-batch / 读出结构」是否与设计一致，"
              "再考虑改档位，并把改动写进口径清单）")
    last_err = ""
    for i in range(start, len(FIT_TIERS)):
        t = FIT_TIERS[i]
        if _locked and t["name"] != _locked:
            break
        if not _tier_available(t):
            print(f"跳过档位 {t['name']}：当前环境不可用")
            continue
        apply_tier(t)
        print("=" * 78)
        print(f"试探档位 {t['name']}：{t['label']} | optim={t['optim']} | "
              f"可训层比例={t['scope']:.0%} | 混合精度={'开' if t['amp'] else '关'} | "
              f"gradient_checkpointing={t['ckpt']} | "
              f"固定开销 {tier_estimate_gb(FIT_REF_PARAMS, t)[0]:.1f} GiB（按参考 {FIT_REF_PARAMS / 1e6:.0f}M 算）")
        print("=" * 78)
        model = None
        try:
            model = build_fn()
            if PROBE_MEMORY and torch.cuda.is_available():
                res = probe_fit_tier(model, data, t)
                if res is None:
                    last_err = f"档位 {t['name']}：所有 batch 档位都 OOM 或余量不足"
                    print(f"→ {last_err}，自动降一档重试")
                    continue
                bs, peak, sps = res
            else:
                print("非 CUDA 环境或 PROBE_MEMORY=False：跳过显存探测，沿用配置中的 batch size")
                bs, peak, sps = BATCH_SIZE, float("nan"), float("nan")








            del model
            model = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            model = build_fn()
            try:
                run_preflight_checks(model, data, OUTPUT_ROOT / "_preflight", micro_bs=bs)
            except FitOOMError as e:
                last_err = f"档位 {t['name']}：开训前检查阶段仍 OOM（{str(e)[:160]}）"
                print(f"→ {last_err}，自动降一档重试")
                print("   ⚠️ 若最终落在与另一条基线**不同**的档位 ⇒ 训练范围不同，结论不可混谈！")
                print("      ⇒ 本版机制要求**全参微调**（t3）；先查有效 batch 与 micro-batch 是否符合设计，"
                      "再决定要不要改档。")
                continue
            return t, bs, peak, sps
        except FitNumericError as e:




            last_err = f"档位 {t['name']}：数值不稳（{str(e)[:200]}）"
            print(f"→ {last_err}，自动降一档重试")
            continue
        finally:
            if model is not None:
                del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    n_params = FIT_REF_PARAMS
    _lines = [f"档位 {t['name']}：{t['label']} → 固定开销 "
              f"{tier_estimate_gb(n_params, t)[0]:.1f} GiB" for t in FIT_TIERS]
    _lock_txt = (f"\n  ⚠️ 档位已锁定为 {FIT_MODE}（FIT_MODE 显式指定）⇒ 不会自动降档。\n"
                 f"    ★ 若要换档，请记住：本版机制要求**全参微调**（t3）；"
                 f"换档后训练范围改变，这一版的数字就不能和别的档位的结果放在一起说。"
                 if _locked else "")
    raise RuntimeError(
        "所有训练档位都跑不通。\n"
        f"  本卡容量          : {vram:.1f} GiB\n"
        f"  参考参数量        : {n_params / 1e6:.0f}M（两条基线中较大者）\n"
        f"  最后一次失败原因  : {last_err}\n" + _lock_txt + "\n"
        "  各档位固定开销    : " + " | ".join(_lines) + "\n"
        "  可选方案：\n"
        "    ① 选更大显存的实例（如 L4 24GB / A100 40GB）—— 档位会自动回到 t0/t1\n"
        "    ② 换更小的同族模型（deberta-v3-base / Qwen2.5-0.5B-Instruct），管线一行不用改\n"
        "    ③ ZeRO-3 / FSDP 分片（deepspeed）\n"
        "    ④ 手动指定档位：把 FIT_MODE 写成 't0'~'t6'"
        "（本版要求 t3 = 全参微调；换档会改变训练范围，结果不可跨档比较）")


print("Trainer 工厂与档位探测就绪")

In [ ]:

def run_preflight_checks(model, data, out_dir, micro_bs=None):
    try:
        _preflight_body(model, data, out_dir, micro_bs=micro_bs)
    except RuntimeError as e:
        if _is_oom(e):
            raise FitOOMError(str(e)[:300]) from e
        raise


def _preflight_body(model, data, out_dir, micro_bs=None):



    if USE_GRAD_CKPT and hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        print(f"preflight：已按当前档位开启 gradient checkpointing"
              f"（档位 {FIT_TIER_NAME}，关掉它手工反向会多占 ~1 GiB 激活）")
    _mb = int(micro_bs) if micro_bs else int(BATCH_SIZE)
    _mb = max(1, _mb)


    model.train()
    _n_fwd = max(1, min(2, _mb))
    _gb = collator([data["train_ds"][i] for i in range(_n_fwd)])
    _gb = {k: (v.to(DEVICE) if isinstance(v, torch.Tensor) else v) for k, v in _gb.items()}
    model.zero_grad(set_to_none=True)
    _go = model(input_ids=_gb["input_ids"], attention_mask=_gb["attention_mask"],
                input_ids2=_gb.get("input_ids2"), attention_mask2=_gb.get("attention_mask2"),
                need2=((_gb["loss_mask"][:, 1] > 0) if "loss_mask" in _gb else None))
    _gl = F.huber_loss(_go.logits, _gb["labels"], delta=HUBER_DELTA)
    _gl.backward()



    if hasattr(model, "flush_readout_grad"):
        model.flush_readout_grad()
    _named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    _with_grad = [(n, p) for n, p in _named if p.grad is not None and float(p.grad.abs().sum()) > 0]


    if getattr(model, "head", None) is not None:
        _head_g = float(model.head.weight.grad.abs().sum()) if model.head.weight.grad is not None else 0.0
        _head_name = "回归头权重梯度绝对值和"
    else:
        _rw = model.readout_weight()
        _head_g = float(_rw.grad.abs().sum()) if _rw.grad is not None else 0.0
        _head_name = "LM 头权重梯度绝对值和"
    _bb_p = next(model.backbone.parameters())
    _bb_g = float(_bb_p.grad.abs().sum()) if _bb_p.grad is not None else 0.0
    print(f"loss                   : {_gl.item():.4f}   （NaN? {bool(torch.isnan(_gl))}）")
    print(f"可训练参数             : {len(_named)}")
    print(f"拿到非零梯度的参数     : {len(_with_grad)}  ({len(_with_grad) / max(1, len(_named)):.1%})")
    print(f"{_head_name:22s}: {_head_g:.6g}   （必须 > 0，否则读出通路没接上）")
    print(f"主干首参数梯度绝对值和 : {_bb_g:.6g}   （必须 > 0，否则主干被冻结）")
    assert _head_g > 0, "读出通路没有梯度：回归头/lm_head 没接进计算图"
    assert _bb_g > 0, "主干没有梯度：主干被冻结了"
    assert not torch.isnan(_gl), "loss 出现 NaN：多半是 fp16 加载或标签里有脏值"





    try:
        _gidx = _gb.get("group_index")
        _sidx = _gb.get("subset_index")
        if _gidx is not None and _sidx is not None:
            _v = _gb["loss_mask"][:, 0] if "loss_mask" in _gb else None
            _p0 = _go.logits[:, 0].detach()
            _y0 = _gb["labels"][:, 0].detach()
            _vf = _v if _v is not None else torch.ones_like(_p0)
            _n_g = int(np.max(ds_int_col(data["train_ds"], "group_index"))) + 1
            _gsub_np = np.zeros(_n_g, dtype=np.int64)
            _gsub_np[ds_int_col(data["train_ds"], "group_index")] = ds_int_col(data["train_ds"], "subset_index")
            _gsub = torch.tensor(_gsub_np, device=_p0.device, dtype=torch.long)
            _prof = profile_vectors(_p0.device, _p0.dtype)
            _n, _d = order_parts(_p0, _y0, _gidx, _sidx, ORDER_TAU, _v, _prof["order"], len(SUBSET_ORDER))
            _r = _n / _d.clamp(min=1.0)
            _n, _d = mse_parts(_p0, _y0, _vf, _sidx, _prof["mse"], len(SUBSET_ORDER))
            _ms = _n / _d.clamp(min=1.0)
            _gs0 = GroupStats(_n_g, _p0.device)
            _c, _s1, _s2, _gc, _gs1, _gs2 = _gs0.stats(_p0, _y0, _gidx, _vf)
            _sp = std_from_moments(_c, _s1, _s2, out_dtype=_p0.dtype)
            _sg = std_from_moments(_gc, _gs1, _gs2, out_dtype=_p0.dtype)
            _n, _d = guard_parts(_sp, _sg, _c, _gsub, _prof["kappa"], len(SUBSET_ORDER))
            _gd = _n / _d.clamp(min=1.0)
            _n, _d = xrank_parts(_sp, _sg, torch.unique(_gidx), _gsub, _prof["xrank"],
                                 XRANK_MARGIN, XRANK_TAU,
                                 torch.ones(_n_g, dtype=torch.bool, device=_p0.device),
                                 len(SUBSET_ORDER))
            _xr = _n / _d.clamp(min=1.0)

            _ce_txt = ""
            _dl = getattr(_go, "digit_logits", None)
            if _dl is not None:


                _n, _d = ce_parts(_dl[:, 0:1, :], _gb["labels"][:, 0:1], _vf, _sidx,
                                  _prof["ce"], len(SUBSET_ORDER))
                _ce = _n / _d.clamp(min=1.0)
                assert torch.isfinite(_ce).all(), "L_ce 出现 NaN/Inf：检查数字分布与标签的形状"
                _ce_txt = f"  L_ce={_ce.tolist()}"
            _sc_txt = ""
            _z = getattr(_go, "proj", None)
            if _z is not None:
                _mlab = (_gb["labels"] * _gb["loss_mask"]).sum(1) / _gb["loss_mask"].sum(1).clamp(min=1.0)
                _n, _d = softcl_parts(_z, _mlab, _sidx, SOFTCL_LABEL_SPAN,
                                      _prof["softcl"], len(SUBSET_ORDER))
                _sc = _n / _d.clamp(min=1.0)
                assert torch.isfinite(_sc).all(), "L_softcl 出现 NaN/Inf：检查投影向量是否全零"
                _sc_txt = f"  L_softcl={_sc.tolist()}"
            print(f"损失项体检（按子集聚合）: L_order={_r.tolist()}  L_mse={_ms.tolist()}")
            print(f"                        L_guard={_gd.tolist()}  L_xrank={_xr.tolist()}"
                  f"{_ce_txt}{_sc_txt}")
            assert torch.isfinite(_r).all() and torch.isfinite(_ms).all() \
                and torch.isfinite(_gd).all() and torch.isfinite(_xr).all(), \
                "损失项出现 NaN/Inf：先查 group_index / subset_index 的 dtype 与形状"
    except AssertionError:
        raise
    except Exception as _e:
        raise AssertionError(f"损失项体检失败（{type(_e).__name__}: {_e}）：检查 group_index / subset_index / loss_mask 的形状与 dtype") from _e
    model.zero_grad(set_to_none=True)







    del _go
    _go = None
    _gb = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()














    _gc_trainer = make_trainer(model, out_dir / "_gradcheck", data,
                                max_steps=1, num_train_epochs=1,
                                per_device_train_batch_size=_mb,
                                gradient_accumulation_steps=1,
                                save_strategy="no", load_best_model_at_end=False,
                                logging_steps=1, **{_EVAL_STRATEGY_KEY: "no"})
    _n_real = max(1, min(int(_mb), len(data["train_ds"])))
    _probe_batch = collator([data["train_ds"][i] for i in range(_n_real)])
    _probe_batch = {k: (v.to(DEVICE) if isinstance(v, torch.Tensor) else v)
                    for k, v in _probe_batch.items()}
    model.zero_grad(set_to_none=True)
    _real_loss = _gc_trainer.compute_loss(model, _probe_batch)
    _real_loss.backward()


    if hasattr(model, "flush_readout_grad"):
        model.flush_readout_grad()
    _bad = []
    _gmax = 0.0
    for _n, _p in model.named_parameters():
        if _p.grad is None:
            continue
        if torch.isnan(_p.grad).any() or torch.isinf(_p.grad).any():
            _bad.append(_n)
        _gmax = max(_gmax, float(_p.grad.abs().max()))
    print(f"真实损失反向          : loss={float(_real_loss):.4f} | 最大梯度={_gmax:.3e} | "
          f"非有限梯度的参数数={len(_bad)}")
    assert torch.isfinite(_real_loss), f"真实损失不是有限值（{float(_real_loss)}）"
    assert not _bad, (
        f"真实损失的反向产生了非有限梯度，涉及 {len(_bad)} 个参数（前 5: {_bad[:5]}）。"
        "这正是云端崩溃的根因类别：坏梯度被写进权重 ⇒ 之后 loss 全是 NaN。"
        "排查顺序：① 上面各分项的数值；② analysis/diagnose_std_gradient.py 的抵消误差；"
        "③ 是否有目标只带 1 条例句（en-nn 实测存在 1 个这样的目标）。")
    model.zero_grad(set_to_none=True)
    del _gc_trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("→ 梯度流验证通过（含真实损失的四项反向）。")









    _st = make_trainer(model, out_dir / "_smoke", data,
                       max_steps=5, num_train_epochs=1,
                       per_device_train_batch_size=_mb,
                       gradient_accumulation_steps=1,
                       save_strategy="no", load_best_model_at_end=False,
                       logging_steps=1, **{_EVAL_STRATEGY_KEY: "no"})
    _so = _st.train()
    del _st
    gc.collect()
    assert np.isfinite(_so.training_loss), "冒烟训练 loss 非有限值"
    print("冒烟训练 loss 轨迹:", _so.training_loss)
    print("→ 冒烟测试通过（每个 Trainer 都是独立构造 args，不复用对象）。")

In [ ]:





ALL_RUNS = []

for _i, (_mode, _subset) in enumerate(RUN_PLAN):
    _subsets = SUBSET_ORDER if _mode == "joint" else [_subset]
    _run_name = f"{_mode}_{_subset}" if _subset else _mode
    _out_dir = OUTPUT_ROOT / MODEL_KEY / _run_name
    print("=" * 78)
    print(f"运行 {_i + 1}/{len(RUN_PLAN)}：{_run_name}  |  参与子集 {_subsets}")
    print("=" * 78)


    _data = prepare_run(_subsets, tokenizer)
    print(f"训练 {len(_data['train_ds'])} 条 | 验证 {len(_data['val_ds'])} 条 | "
          f"评测设置 {len(_data['eval_cols'])} 个")





    _run_epochs = JOINT_EPOCHS if _mode == "joint" else EPOCHS
    print(f"本次运行轮数 = {_run_epochs}"
          f"（{'联合运行，提交用' if _mode == 'joint' else '单子集诊断'}）")


    model = build_model()
    n_params = sum(p.numel() for p in model.parameters())
    check_vram_budget(n_params, MODEL_REGISTRY[MODEL_KEY]["model_id"])









    if _i == 0:
        model = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


        _tier, _bs, _peak, _sps = prepare_for_training(build_model, _data)
        apply_tier(_tier)
        BATCH_SIZE = _bs



        GRAD_ACCUM = max(1, int(round(TARGET_EFFECTIVE_BATCH / _bs)))
        print(f"→ 采用档位 {_tier['name']}（{_tier['label']}）| micro-batch={BATCH_SIZE} | "
              f"梯度累积={GRAD_ACCUM} | 有效 batch={BATCH_SIZE * GRAD_ACCUM}"
              f"（目标 {TARGET_EFFECTIVE_BATCH}） | 峰值显存约 {_peak:.2f} GiB")
        print("   （★ v4.4：档位**写死为 t3（全参微调）** —— "
              "档位不再由显存探测决定，所以不可能再悄悄漂到 t5 把主干冻掉一半）")

        if _sps == _sps:
            _n_rows, _tot, _hours = project_train_time(_sps)
            print(f"训练时长投影：5 次运行共约 {_tot:,} 个样本·轮 → 约 {_hours:.1f} 小时"
                  f"（Kaggle 上限 12 小时，预算 {TIME_BUDGET_H} 小时）")
            if _hours > TIME_BUDGET_H:
                print("=" * 78)
                print(f"⚠️ 投影时长 {_hours:.1f} 小时 > 预算 {TIME_BUDGET_H} 小时！")
                print("   ① 调小 EPOCHS（本条基线单独改即可，但必须记进口径清单）")
                print("   ② 把 RUN_PLAN 改成只跑联合那一次（先拿到一个能提交的模型）")
                print("   ③ 换更快的卡 / 更小的同族模型")
                print("=" * 78)












    _max_oom_retry = 3
    _attempt = 0
    _rand_metrics = None
    _rand_eval_loss = float("nan")
    while True:
        release_gpu_memory("正式训练前")
        model = build_model()
        trainer = make_trainer(model, _out_dir, _data, epochs=_run_epochs)



        if _rand_metrics is None:
            _t0 = time.time()
            _rand_metrics = strip_eval_prefix(trainer.evaluate(_data["val_ds"]))
            print(f"随机初始化模型基线（{time.time() - _t0:.1f}s）: "
                  f"指标1 ρ={_rand_metrics.get('合计|_指标1_ρ', float('nan')):.4f} | "
                  f"指标2 ρ={_rand_metrics.get('合计|_指标2_ρ', float('nan')):.4f} | "
                  f"RMSE={_rand_metrics.get('合计|_指标3_RMSE', float('nan')):.4f}")





            _rand_eval_loss = float(_rand_metrics.get("loss", float("nan")))
            print(f"随机初始化时的评估 loss = {_rand_eval_loss:.4f}（图 F 的 epoch 0 起点）")
            release_gpu_memory("随机基线评估后")


        _t0 = time.time()
        try:
            _res = trainer.train()
        except RuntimeError as _e:
            if not _is_oom(_e) or _attempt >= _max_oom_retry:
                raise
            _attempt += 1
            _new_bs = max(1, BATCH_SIZE // 2)
            if _new_bs == BATCH_SIZE:
                raise
            _new_ga = max(1, int(round(TARGET_EFFECTIVE_BATCH / _new_bs)))
            print("=" * 78)
            print(f"⚠️ 正式训练 OOM（第 {_attempt}/{_max_oom_retry} 次重试）: {str(_e)[:220]}")
            print(f"   micro-batch {BATCH_SIZE} → {_new_bs}，梯度累积 {GRAD_ACCUM} → {_new_ga}"
                  f"（**有效 batch 仍为 {TARGET_EFFECTIVE_BATCH}**，口径不变）")
            print("   ⇒ 重建模型与 Trainer 后重试（权重会被重新初始化，保持可比）")
            print("=" * 78)
            BATCH_SIZE = _new_bs
            GRAD_ACCUM = _new_ga
            del trainer, model
            release_gpu_memory("OOM 重试前")
            continue
        _dt = time.time() - _t0
        print(f"训练完成，用时 {_dt / 60:.1f} 分钟，loss={_res.training_loss:.4f}")
        break





    _log_hist = [dict(h) for h in trainer.state.log_history]
    _n_train_logs = sum(1 for h in _log_hist if "loss" in h and "eval_loss" not in h)
    _n_eval_logs = sum(1 for h in _log_hist if "eval_loss" in h)
    print(f"日志快照：训练点 {_n_train_logs} 个 | 评估点 {_n_eval_logs} 个")




    _actual_bs = int(trainer.args.per_device_train_batch_size)
    _actual_ga = int(trainer.args.gradient_accumulation_steps)
    _dl_len = math.ceil(len(_data["train_ds"]) / _actual_bs)
    _actual_max_steps = int(trainer.state.max_steps)
    _expected_max_steps = math.ceil(_dl_len / _actual_ga) * int(trainer.args.num_train_epochs)
    _actual_warmup = int(trainer.args.warmup_steps)
    _actual_eff_batch = _actual_bs * _actual_ga
    _actual_wf = _actual_warmup / max(1, _actual_max_steps)
    assert _actual_max_steps == _expected_max_steps, (
        f"Trainer 的总更新步 {_actual_max_steps} 与按口径算出的 {_expected_max_steps} 不一致")
    assert _actual_eff_batch == BATCH_SIZE * GRAD_ACCUM, (
        f"实际有效 batch = {_actual_eff_batch} ≠ 配置 {BATCH_SIZE * GRAD_ACCUM}")
    _wf_tol = max(0.02, 2.0 / max(1, _actual_max_steps))
    assert abs(_actual_wf - WARMUP_RATIO) <= _wf_tol, (
        f"实际 warmup 占比 {_actual_wf:.3f} ≠ 设定 {WARMUP_RATIO}")
    _loss_terms = trainer.loss_weight_snapshot()
    print(f"口径回读 ✓ 有效 batch={_actual_eff_batch} | 每轮批数={_dl_len} | "
          f"总更新步={_actual_max_steps} | warmup={_actual_warmup} 步（占 {_actual_wf:.1%}）")
    print(f"损失分项权重（自适应标定后）: {_loss_terms}")


    _pred_out = trainer.predict(_data["val_ds"])
    _trained_metrics = strip_eval_prefix(trainer.evaluate(_data["val_ds"]))
    print("--- 训练后指标 ---")
    for k, v in _trained_metrics.items():
        if isinstance(v, float) and ("指标" in k or "目标数" in k or "综合分" in k):
            print(f"   {k:34s} {v:.4f}")


    _val_df = _data["val_df"]
    _gold = ds_labels(_data["val_ds"])
    _type_level = np.zeros((len(_val_df), N_OUTPUTS))
    for _name, _cname, _j in _data["eval_cols"]:
        _src = "y0" if _j == 0 else "y1"
        _m = (_val_df["subset_name"] == _name).to_numpy()
        _type_level[_m, _j] = _val_df.groupby("group_key")[_src].transform("mean").to_numpy()[_m]
    _oracle_metrics = compute_compartment_metrics(
        _type_level, _gold, np.asarray(_data["val_ds"]["group_index"]),



        np.asarray([SUBSET_ORDER[i] for i in _data["val_ds"]["subset_index"]]),
        _data["eval_cols"])





    if _run_name in SAVE_FINAL_FOR:
        trainer.save_model(str(_out_dir / "final"))
        tokenizer.save_pretrained(str(_out_dir / "final"))
        print(f"模型已保存到: {_out_dir / 'final'}")



    for _ck in sorted(_out_dir.glob("checkpoint-*")):
        shutil.rmtree(_ck, ignore_errors=True)





    _settings = dict(
        model_key=MODEL_KEY, model_id=MODEL_REGISTRY[MODEL_KEY]["model_id"],



        head_mode=HEAD_MODE, mech=MECH_NAME,
        optim=OPTIM, gradient_checkpointing=USE_GRAD_CKPT,


        fit_tier=FIT_TIER_NAME, trainable_scope=FIT_SCOPE_FRAC,
        mixed_precision=("amp" if (USE_BF16 or USE_FP16) else "fp32"),
        gpu_index=GPU_INDEX,
        effective_batch=BATCH_SIZE * GRAD_ACCUM, micro_batch=BATCH_SIZE,
        grad_accum=GRAD_ACCUM, lr=LR, epochs=EPOCHS, max_length=MAX_LENGTH,


        run_epochs=int(_run_epochs), joint_epochs=int(JOINT_EPOCHS),



        metric_for_best_model=BEST_METRIC_KEY,
        weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO,
        huber_delta=HUBER_DELTA,


        loss_profile={k: dict(v) for k, v in SUBSET_LOSS_PROFILE.items()},
        auto_balance=bool(AUTO_BALANCE),
        order_tau=ORDER_TAU, xrank_margin=XRANK_MARGIN, xrank_tau=XRANK_TAU,
        balance_ema_beta=BALANCE_EMA_BETA,

        softcl_label_span=SOFTCL_LABEL_SPAN,
        softcl_proj_dim=(SOFTCL_PROJ_DIM if getattr(model, "proj", None) is not None else None),
        ce_soft_sigma=CE_SOFT_SIGMA, readout_cues=READOUT_CUES,
        z_m1=Z_M1, z_m2=Z_M2, z_m3=Z_M3,
        best_metric_formula="m1/Z_M1 + m2/Z_M2 - m3/Z_M3",
        loss_finite_check_every=LOSS_FINITE_CHECK_EVERY,

        stats_dtype=(str(trainer._gs[0].dtype).replace("torch.", "")
                     if trainer._gs and trainer._gs[0] is not None else "n/a"),



        actual_effective_batch=_actual_eff_batch,
        actual_max_steps=_actual_max_steps,
        actual_warmup_steps=_actual_warmup,
        actual_warmup_frac=round(_actual_wf, 4),
        dataloader_len=_dl_len,
        loss_term_weights={k: round(v, 4) for k, v in _loss_terms.items()},
        seed=SEED, n_outputs=N_OUTPUTS, pooling=MODEL_REGISTRY[MODEL_KEY]["pooling"],
        train_subset=_subsets,


        logging_steps=int(trainer.args.logging_steps),
    )
    _set_dir = OUTPUT_ROOT / "summary"
    _set_dir.mkdir(parents=True, exist_ok=True)
    with open(_set_dir / f"run_settings_{_run_name}.json", "w", encoding="utf-8") as _f:
        json.dump(_settings, _f, ensure_ascii=False, indent=2)
    ALL_RUNS.append(dict(
        run_name=_run_name, mode=_mode, subset=_subset, subsets=list(_subsets),
        eval_cols=_data["eval_cols"], out_dir=_out_dir,
        metrics=_trained_metrics, random_metrics=_rand_metrics, oracle_metrics=_oracle_metrics,
        settings=_settings,
        preds=np.asarray(_pred_out.predictions).reshape(len(_gold), -1),
        gold=_gold,
        groups=np.asarray(_data["val_ds"]["group_index"]),
        subset_ids=np.asarray([SUBSET_ORDER[i] for i in _data["val_ds"]["subset_index"]]),
        val_df=_val_df,
        train_minutes=_dt / 60,
        log_history=_log_hist,
        random_eval_loss=_rand_eval_loss,
    ))
    print(f"→ 运行 {_run_name} 完成，产物在 {_out_dir}")


    del trainer, model, _pred_out
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 78)
print(f"全部 {len(ALL_RUNS)} 次运行完成")
print("=" * 78)

## Step9 模型评估、对照图与结果分析

这一节分成**两个互不混淆的区块**：

### 区块一：评估标准（模型本身的结果）

这就是**官方评测会看到的数字**——模型在 6 个评测设置上的三项指标。
表 1 与图 A / 图 B / 图 C 全部属于这一块。

### 区块二：参考对照（不是评估标准）

两个**人造系统**，用来回答「这个分数算好吗」：

| 对照系统 | 它是什么 | 用来回答什么问题 |
|---|---|---|
| **随机初始化** | 同一个模型、同一份数据，但**没训练过** | 「模型到底学到了什么」——训练后必须显著优于它，否则等于没训 |
| **type-level oracle** | 给每个目标赋它自己的**真实均值**，完全不看语境 | 「天花板在哪」——它在指标 1 / RMSE 上能拿满分，但指标 2 必然是 `nan`（组内 std 恒为 0）。我们的模型如果指标 2 也接近 nan，说明还没真正做到「因句而异」 |

**这两个都不是提交系统的成绩**，所以它们被单独放在区块二，
不与评估标准混在同一张表、同一根柱子里。

**六张图各管一件事**：

| 图 | 内容 | 看什么 |
|---|---|---|
| 图 A | 预测 vs 真值散点（每个「子集×列」一个面板） | 点是否落在对角线上；ρ 是整体排序能力，RMSE 是数值校准 |
| 图 B | 6 个评测设置 × 三项指标 | **评估标准**本身：哪个设置强、哪个设置弱、指标 2 有没有塌成 nan |
| 图 C | 组内 std 对照散点（人工 std vs 预测 std） | 这就是指标 2 的可视化：横轴是金标准，纵轴是系统输出，ρ 即指标 2 |
| 图 D | 五次运行的合计指标横向对照 | 「联合训练」到底有没有比「单子集训练」更好 |
| 图 E | 训练后 / 随机初始化 / type-level oracle | 参照系：这个分数算好吗（**不是**评估标准） |
| 图 F | 训练损失曲线（训练集与测试集同图、不同颜色） | 训练过程健不健康：有没有收敛、有没有过拟合（泛化差距） |

**为什么图 C 最关键**：指标 2 直接决定是否出局。
如果图 C 的点挤成一条水平线（预测的组内 std 几乎不变），
说明模型没有真正对语境敏感——哪怕指标 1 很高，这份提交也会在指标 2 上拿 `nan`。

In [ ]:



def style_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, alpha=0.22, linewidth=0.6)
    ax.set_axisbelow(True)


def annotate_box(ax, x, y, text, color="#2C3E50", ha="left", va="top"):
    ax.text(x, y, text, ha=ha, va=va, fontsize=8.4, color=color,
            bbox=dict(boxstyle="round,pad=0.32", facecolor="white",
                      edgecolor=color, linewidth=0.8, alpha=1.0))


def label_bars(ax, items, ylim, fontsize=7.0):
    lo, hi = ylim
    gap = (hi - lo) * 0.058
    placed = []
    for x, y, t in sorted(items, key=lambda z: (z[0], -z[1])):
        yy = y
        for px, py in placed:
            if abs(px - x) < 0.32 and abs(py - yy) < gap:
                yy = py + gap
        placed.append((x, yy))
        va = "bottom" if y >= 0 else "top"
        ax.text(x, yy, t, ha="center", va=va, fontsize=fontsize, color="#2C3E50",
                bbox=dict(boxstyle="round,pad=0.16", facecolor="white",
                          edgecolor="none", alpha=1.0))


BLUE, ORANGE, GREY, GREEN, RED = "#3C6E9E", "#C2703D", "#8A8F98", "#2E7D5B", "#B03A2E"


MAIN_RUN = next((r for r in ALL_RUNS if r["mode"] == "joint"), ALL_RUNS[-1])
print("主图使用的运行:", MAIN_RUN["run_name"], "| 评测设置:", len(MAIN_RUN["eval_cols"]))

In [ ]:



_rows = []
for _r in ALL_RUNS:
    for _n, _cn, _j in _r["eval_cols"]:
        _rows.append({
            "运行": _r["run_name"],
            "子集": _n,
            "标签列": _cn,
            "指标1 ρ": _r["metrics"].get(f"{_n}|{_cn}_指标1_ρ", float("nan")),
            "指标2 ρ": _r["metrics"].get(f"{_n}|{_cn}_指标2_ρ", float("nan")),
            "指标3 RMSE": _r["metrics"].get(f"{_n}|{_cn}_指标3_RMSE", float("nan")),
            "组内std退化目标数": _r["metrics"].get(f"{_n}|{_cn}_组内std退化目标数", 0),
            "样例数": _r["metrics"].get(f"{_n}|{_cn}_样例数", 0),
        })
RESULTS = pd.DataFrame(_rows)
MAIN_RESULTS = RESULTS[RESULTS["运行"] == MAIN_RUN["run_name"]].reset_index(drop=True)
print("=" * 96)
print(f"表 1  评估标准 —— {MAIN_RUN['run_name']}（联合运行覆盖全部评测设置）")
print("=" * 96)
print(MAIN_RESULTS.round(4).to_string(index=False))
print(f"\n合计（各评测设置取平均）：指标1 ρ={MAIN_RESULTS['指标1 ρ'].mean():.4f} | "
      f"指标2 ρ={MAIN_RESULTS['指标2 ρ'].mean():.4f} | "
      f"RMSE={MAIN_RESULTS['指标3 RMSE'].mean():.4f}")
print(f"best-checkpoint 选择指标（{BEST_METRIC_KEY}，= 指标1/{Z_M1} + 指标2/{Z_M2} − RMSE/{Z_M3}，"
      f"即在 z 分数空间里等权）= "
      f"{MAIN_RUN['metrics'].get(BEST_METRIC_KEY, float('nan')):.4f}")
print("→ 指标 2 若是 nan，说明该设置下模型对同一目标的所有例句给出了相同分数，"
      "会在官方评测中直接出局。")


_summary_dir = OUTPUT_ROOT / "summary"
_summary_dir.mkdir(parents=True, exist_ok=True)
RESULTS.to_csv(_summary_dir / "results_all_runs.csv", index=False)
print("\n已保存:", _summary_dir / "results_all_runs.csv")



















_DECLARED = [
    "fit_tier", "trainable_scope", "mixed_precision", "optim",
    "effective_batch", "actual_effective_batch",
    "lr", "epochs", "run_epochs", "joint_epochs",
    "warmup_ratio", "actual_warmup_frac",
    "huber_delta", "seed", "n_outputs", "max_length", "weight_decay",
    "metric_for_best_model", "auto_balance", "order_tau", "xrank_margin",
    "xrank_tau", "balance_ema_beta", "loss_finite_check_every",
]
_MECH_ITEMS = {
    "head_mode": "★ v4 机制项：读出方式（regression / lm_head）",
    "mech": "★ v4 机制项：本实验验证的机制名",
    "loss_profile": "★ v4 机制项：损失配方按机制分叉（DeBERTa 开 softcl、Qwen 开 ce）",
    "softcl_label_span": "★ v4 机制项：softcl 的标签量程",
    "ce_soft_sigma": "★ v4 机制项：软目标分布带宽",
    "readout_cues": "★ v4 机制项：lm_head 的读出锚点",
    "z_m1": "★ v4 新增：best 准则的 z 分数常数（上一版没有这个准则）",
    "z_m2": "★ v4 新增：best 准则的 z 分数常数",
    "z_m3": "★ v4 新增：best 准则的 z 分数常数",
}
_FREE_ITEMS = {
    "model_key": "模型身份", "model_id": "模型身份",
    "pooling": "架构决定的池化方式（mean / last）",
    "micro_batch": "batch 分解方式（乘积 = effective_batch 即等价）",
    "grad_accum": "batch 分解方式（乘积 = effective_batch 即等价）",
    "gradient_checkpointing": "只影响速度与显存，数学上等价",
    "gpu_index": "用哪一张**同型号**卡（只影响并行调度）",
    "train_subset": "本次运行的子集（single vs joint 本就不同）",
    "actual_max_steps": "本次运行的总更新步数（随子集大小/轮数变）",
    "actual_warmup_steps": "本次运行的 warmup 绝对步数",
    "dataloader_len": "本次运行每轮的批数",
    "loss_term_weights": "自适应标定出的实际权重（随训练进程变，只作记录）",
    "stats_dtype": "组统计矩的累加精度（CUDA/CPU=float64；MPS 不支持 float64）",
}
print("\n" + "=" * 78)
print("本实验的口径（独立记录；A 组里「声明值」与「实际值」必须一致）")
print("=" * 78)
_s = ALL_RUNS[0]["settings"]
print(f"\n【A. 本实验口径】{len(_DECLARED)} 项")
for _k in _DECLARED:
    print(f"  {_k:24s} = {_s.get(_k)}")
print("\n  ★ v1 事故核对（声明 vs 事实，必须一致）：")
for _d, _a, _why in [
    ("effective_batch", "actual_effective_batch", "有效 batch：声明值 = 实际生效值"),
    ("warmup_ratio", "actual_warmup_frac", "warmup 占比：声明值 = 实际生效值"),
    ("fit_tier", "trainable_scope", "档位 = 可训层比例（t3 ⇒ 1.0 全参）"),
]:
    print(f"    {_d}={_s.get(_d)} | {_a}={_s.get(_a)}    ← {_why}")
print(f"\n【B. 机制项】{len(_MECH_ITEMS)} 项 —— 与上一版不同是**设计如此**")
for _k, _why in _MECH_ITEMS.items():
    print(f"  {_k:24s} = {_s.get(_k)}      ← {_why}")
print(f"\n【C. 实现细节 / 环境】{len(_FREE_ITEMS)} 项 —— 允许不同，不影响数值")
for _k, _why in _FREE_ITEMS.items():
    print(f"  {_k:24s} = {_s.get(_k)}      ← {_why}")
print("\n口径文件:", _summary_dir / f"run_settings_{ALL_RUNS[0]['run_name']}.json")
print("★ 判读规则（v4.4）：")
print("   ① 本实验**独立成立** —— 不再要求与另一个实验口径一致；")
print("   ② A 组的「声明值」必须等于「实际生效值」（v1 事故就是这么发生的）；")
print("   ③ 要与其它实验/版本比较时，用这张表去核对差异，**不要假设两边一定一样**。")

In [ ]:



_panels = MAIN_RUN["eval_cols"]
_pred = MAIN_RUN["preds"]
_gold = MAIN_RUN["gold"]
_groups = MAIN_RUN["groups"]
_subsets = MAIN_RUN["subset_ids"]
_nc = min(3, len(_panels))
_nr = math.ceil(len(_panels) / _nc)
fig, axes = plt.subplots(_nr, _nc, figsize=(4.8 * _nc, 4.5 * _nr), squeeze=False)
for _k, (_name, _cname, _j) in enumerate(_panels):
    ax = axes[_k // _nc][_k % _nc]
    _m = _subsets == _name
    ax.scatter(_gold[_m, _j], _pred[_m, _j], s=14, alpha=0.42, color=BLUE, edgecolors="none")
    _lo = max(0.0, float(min(_gold[_m, _j].min(), _pred[_m, _j].min())) - 0.3)
    _hi = float(max(_gold[_m, _j].max(), _pred[_m, _j].max())) + 0.3
    ax.plot([_lo, _hi], [_lo, _hi], "--", color=GREY, linewidth=1.1, zorder=1)
    ax.set_xlim(_lo, _hi); ax.set_ylim(_lo, _hi)
    ax.set_xlabel(T("gold")); ax.set_ylabel(T("pred"))
    ax.set_title(f"{_name}·{_cname}", fontsize=10.5)
    _r = stats.spearmanr(_pred[_m, _j], _gold[_m, _j])[0]
    _e = float(np.sqrt(((_pred[_m, _j] - _gold[_m, _j]) ** 2).mean()))
    annotate_box(ax, _lo + 0.04 * (_hi - _lo), _hi - 0.04 * (_hi - _lo),
                 f"ρ = {_r:.3f}\nRMSE = {_e:.3f}")
    style_axes(ax)
for _k in range(len(_panels), _nr * _nc):
    axes[_k // _nc][_k % _nc].axis("off")
fig.suptitle(f"{T('figA')} ({MAIN_RUN['run_name']})", fontsize=12.5,
             fontweight="bold", y=1.02)
fig.tight_layout()
_figA = _summary_dir / "figA_pred_vs_gold.png"
fig.savefig(_figA, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("已保存:", _figA)

In [ ]:



_metric_defs = [("指标1_ρ", T("rho_up"), T("m1")),
                ("指标2_ρ", T("rho_up"), T("m2")),
                ("指标3_RMSE", T("rmse_down"), T("m3"))]
_xlabels = [f"{n}\n{c}" for n, c, _j in _panels]

fig, axes = plt.subplots(1, 3, figsize=(2.0 * len(_panels) + 3.6, 5.2))
for _mi, (_suffix, _ylabel, _mtitle) in enumerate(_metric_defs):
    ax = axes[_mi]
    _vals = [MAIN_RUN["metrics"].get(f"{n}|{c}_{_suffix}", float("nan")) for n, c, _j in _panels]
    _finite = [v for v in _vals if np.isfinite(v)]
    _heights = [v if np.isfinite(v) else 0.0 for v in _vals]
    _colors = [BLUE if np.isfinite(v) else "#D8DBE0" for v in _vals]
    ax.bar(np.arange(len(_panels)), _heights, 0.6, color=_colors, edgecolor="white")


    _lo = min([0.0] + _finite) if _finite else 0.0
    _hi = max([0.0] + _finite) if _finite else 1.0
    _span = max(_hi - _lo, 1e-6)
    ax.set_ylim(_lo - 0.08 * _span, _hi + 0.22 * _span)
    _items = [(k, v, f"{v:.3f}") for k, v in enumerate(_vals) if np.isfinite(v)]
    label_bars(ax, _items, ax.get_ylim(), fontsize=7.4)
    for k, v in enumerate(_vals):
        if not np.isfinite(v):

            ax.text(k, _lo + 0.02 * _span, "nan", ha="center", va="bottom",
                    fontsize=8.0, color=RED, fontweight="bold")
    ax.axhline(0, color="#B0B4BA", linewidth=0.8, zorder=0)
    ax.set_xticks(np.arange(len(_panels)))
    ax.set_xticklabels(_xlabels, fontsize=7.2)
    ax.set_ylabel(_ylabel)
    ax.set_title(f"({chr(97 + _mi)}) {_mtitle}", fontsize=10.5)
    style_axes(ax)
fig.suptitle(f"{T('figB')}: {MAIN_RUN['run_name']} on {len(_panels)} {T('n_settings')}",
             fontsize=12.5, fontweight="bold", y=1.02)
fig.tight_layout()
_figB = _summary_dir / "figB_evaluation_standard.png"
fig.savefig(_figB, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("已保存:", _figB)

In [ ]:







_c_sub = list(_panels)
_nc = min(3, len(_c_sub))
_nr = math.ceil(len(_c_sub) / _nc)
fig, axes = plt.subplots(_nr, _nc, figsize=(4.6 * _nc, 4.3 * _nr), squeeze=False)
for _k, (_name, _cname, _j) in enumerate(_c_sub):
    ax = axes[_k // _nc][_k % _nc]
    _m = _subsets == _name
    _gs = pd.DataFrame({"g": _groups[_m], "p": _pred[_m, _j], "y": _gold[_m, _j]}).groupby("g")
    _ps = _gs["p"].std(ddof=1)
    _ys = _gs["y"].std(ddof=1)
    _ok = _ps.notna() & _ys.notna()
    _r2 = stats.spearmanr(_ps[_ok], _ys[_ok])[0] if _ok.sum() > 2 else float("nan")
    ax.scatter(_ys[_ok], _ps[_ok], s=20, alpha=0.55, color=ORANGE, edgecolors="none")
    _mx = float(max(_ys[_ok].max(), _ps[_ok].max())) * 1.10 if _ok.sum() else 1.0
    ax.plot([0, _mx], [0, _mx], "--", color=GREY, linewidth=1.1, zorder=1)
    ax.set_xlim(0, _mx); ax.set_ylim(0, _mx)
    ax.set_xlabel(T("gold_std"))
    ax.set_ylabel(T("pred_std"))
    ax.set_title(f"{_name}·{_cname}", fontsize=10.5)
    annotate_box(ax, _mx * 0.03, _mx * 0.97,
                 f"{T('m2rho')} {_r2:.3f}\n{T('n_targets')} {int(_ok.sum())}\n"
                 f"{T('const_std')} {int((_ps[_ok] == 0).sum())}")
    style_axes(ax)
for _k in range(len(_c_sub), _nr * _nc):
    axes[_k // _nc][_k % _nc].axis("off")
fig.suptitle(f"{T('figC')}: {T('gold_axis')} ({MAIN_RUN['run_name']})",
             fontsize=12.5, fontweight="bold", y=1.02)
fig.tight_layout()
_figC = _summary_dir / "figC_group_std.png"
fig.savefig(_figC, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("已保存:", _figC)

In [ ]:




_run_names = [r["run_name"] for r in ALL_RUNS]
_run_labels = []
for _r in ALL_RUNS:
    if _r["mode"] == "joint":
        _run_labels.append(T("joint_short"))
    else:
        _run_labels.append(T("single_short", short=short_of(_r["subset"])))

fig, axes = plt.subplots(1, 3, figsize=(2.1 * len(ALL_RUNS) + 3.2, 4.8))
for _mi, (_suffix, _ylabel, _mtitle) in enumerate(_metric_defs):
    ax = axes[_mi]
    _vals = [_r["metrics"].get(f"合计|_{_suffix}", float("nan")) for _r in ALL_RUNS]
    _finite = [v for v in _vals if np.isfinite(v)]
    _heights = [v if np.isfinite(v) else 0.0 for v in _vals]
    _colors = [ORANGE if _r["mode"] == "joint" else BLUE for _r in ALL_RUNS]
    ax.bar(np.arange(len(ALL_RUNS)), _heights, 0.62, color=_colors, edgecolor="white")
    _lo = min([0.0] + _finite) if _finite else 0.0
    _hi = max([0.0] + _finite) if _finite else 1.0
    _span = max(_hi - _lo, 1e-6)
    ax.set_ylim(_lo - 0.08 * _span, _hi + 0.24 * _span)
    label_bars(ax, [(k, v, f"{v:.3f}") for k, v in enumerate(_vals) if np.isfinite(v)],
               ax.get_ylim(), fontsize=7.6)
    for k, v in enumerate(_vals):
        if not np.isfinite(v):
            ax.text(k, _lo + 0.02 * _span, "nan", ha="center", va="bottom",
                    fontsize=8.0, color=RED, fontweight="bold")
    ax.axhline(0, color="#B0B4BA", linewidth=0.8, zorder=0)
    ax.set_xticks(np.arange(len(ALL_RUNS)))
    ax.set_xticklabels(_run_labels, fontsize=7.4)
    ax.set_ylabel(_ylabel)
    ax.set_title(f"({chr(97 + _mi)}) {_mtitle}", fontsize=10.5)
    style_axes(ax)
_handles = [plt.Rectangle((0, 0), 1, 1, color=BLUE), plt.Rectangle((0, 0), 1, 1, color=ORANGE)]
fig.legend(_handles, [T("single"), T("joint")],
           loc="upper center", bbox_to_anchor=(0.5, 0.99), ncol=2, fontsize=9, frameon=False)
fig.suptitle(T("figD"), fontsize=12.5, fontweight="bold", y=1.06)
fig.tight_layout()
_figD = _summary_dir / "figD_run_comparison.png"
fig.savefig(_figD, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("已保存:", _figD)

In [ ]:






print("=" * 96)
print("区块二  参考对照（不是评估标准，仅用于解读上面的数字）")
print("=" * 96)
_cmp_rows = []
for _r in ALL_RUNS:
    for _sys, _m in [("训练后模型", _r["metrics"]),
                     ("随机初始化", _r["random_metrics"]),
                     ("type-level oracle", _r["oracle_metrics"])]:
        _cmp_rows.append({
            "运行": _r["run_name"], "对照系统": _sys,
            "指标1 ρ": _m.get("合计|_指标1_ρ", float("nan")),
            "指标2 ρ": _m.get("合计|_指标2_ρ", float("nan")),
            "指标3 RMSE": _m.get("合计|_指标3_RMSE", float("nan")),
        })
COMPARE = pd.DataFrame(_cmp_rows)
print(COMPARE.round(4).to_string(index=False))
COMPARE.to_csv(_summary_dir / "reference_comparison.csv", index=False)

print("\n--- 三个系统分别是什么 ---")
print("随机初始化      ：同一模型、同一份数据，但一步都没训练。"
      "它是「瞎猜」的下界；训练后模型必须显著高于它，否则等于没训。")
print("type-level oracle：给每个目标赋它自己的真实均值（不看语境）。"
      "它在指标 1 / RMSE 上接近满分，但指标 2 必然是 nan"
      "（同一目标的所有例句拿到同一个分数 → 组内 std 恒为 0 → 相关系数未定义）。")
print("→ 我们的模型如果指标 2 也接近 nan，说明还没有真正做到「因句而异」。")

In [ ]:


_cmp_systems = [(T("trained"), "metrics", BLUE),
                (T("random_init"), "random_metrics", GREY),
                (T("oracle"), "oracle_metrics", ORANGE)]
fig, axes = plt.subplots(1, 3, figsize=(11.0, 4.6))
for _mi, (_suffix, _ylabel, _mtitle) in enumerate(_metric_defs):
    ax = axes[_mi]
    _items = []
    for _si, (_sys, _key, _c) in enumerate(_cmp_systems):
        _v = MAIN_RUN[_key].get(f"合计|_{_suffix}", float("nan"))
        _x = _si
        if np.isfinite(_v):
            ax.bar(_x, _v, 0.55, color=_c, edgecolor="white")
            _items.append((_x, _v, f"{_v:.3f}"))
        else:
            ax.bar(_x, 0.0, 0.55, color="#D8DBE0", edgecolor="white")
    _finite = [y for _x, y, _t in _items]
    _lo = min([0.0] + _finite) if _finite else 0.0
    _hi = max([0.0] + _finite) if _finite else 1.0
    _span = max(_hi - _lo, 1e-6)
    ax.set_ylim(_lo - 0.08 * _span, _hi + 0.24 * _span)
    label_bars(ax, _items, ax.get_ylim(), fontsize=8.4)

    if _suffix == "指标2_ρ":
        ax.text(2, _lo + 0.02 * _span, "nan", ha="center", va="bottom",
                fontsize=9.0, color=RED, fontweight="bold")
    ax.axhline(0, color="#B0B4BA", linewidth=0.8, zorder=0)
    ax.set_xticks(range(len(_cmp_systems)))
    ax.set_xticklabels([s for s, _k, _c in _cmp_systems], fontsize=8.2)
    ax.set_ylabel(_ylabel)
    ax.set_title(f"({chr(97 + _mi)}) {_mtitle}", fontsize=10.5)
    style_axes(ax)
fig.suptitle(f"{T('figE')} ({T('not_standard')}): {MAIN_RUN['run_name']}",
             fontsize=12.0, fontweight="bold", y=1.03)
fig.tight_layout()
_figE = _summary_dir / "figE_reference_comparison.png"
fig.savefig(_figE, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("已保存:", _figE)

In [ ]:










def _loss_series(hist):
    tr = [(float(h.get("epoch", 0.0)), float(h["loss"]))
          for h in hist if "loss" in h and "eval_loss" not in h]
    ev = [(float(h.get("epoch", 0.0)), float(h["eval_loss"]))
          for h in hist if "eval_loss" in h]
    tr.sort()
    ev.sort()
    return tr, ev


_RUN_COLORS = [BLUE, GREEN, ORANGE, "#7A5195", "#3F7F93"]
fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.7))


ax = axes[0]
for _ri, _r in enumerate(ALL_RUNS):
    _tr, _ev = _loss_series(_r.get("log_history", []))
    if not _tr:
        continue
    _lbl = (T("joint_short").replace("\n", " ") if _r["mode"] == "joint"
            else short_of(_r["subset"]))
    ax.plot([p[0] for p in _tr], [p[1] for p in _tr], marker="o", markersize=3.0,
            linewidth=1.5, color=_RUN_COLORS[_ri % len(_RUN_COLORS)], label=_lbl)
ax.set_xlabel(T("epoch"))
ax.set_ylabel(T("loss"))
ax.set_title(f"(a) {T('loss_all_runs')}", fontsize=10.5)
ax.legend(fontsize=8.2, frameon=False)
style_axes(ax)


ax = axes[1]
_tr, _ev = _loss_series(MAIN_RUN.get("log_history", []))


_rl = MAIN_RUN.get("random_eval_loss", float("nan"))
if np.isfinite(_rl):
    _ev = [(0.0, float(_rl))] + _ev
if _tr:
    ax.plot([p[0] for p in _tr], [p[1] for p in _tr], color=BLUE, linewidth=1.6,
            marker="o", markersize=3.2, label=T("train_loss"))
if _ev:
    _ex = [p[0] for p in _ev]
    _ey = [p[1] for p in _ev]
    ax.plot(_ex, _ey, color=RED, linewidth=1.8, marker="s", markersize=6.0,
            label=T("eval_loss"))
    for _i0, (_x0, _y0) in enumerate(zip(_ex, _ey)):


        _va = "bottom" if _i0 == 0 else "top"
        annotate_box(ax, _x0, _y0, f"{_y0:.3f}", color=RED, ha="center", va=_va)



    if _ex and abs(_ex[0]) < 1e-9:
        ax.annotate(T("untrained"), xy=(_ex[0], _ey[0]), xytext=(0.30, 0.52),
                    textcoords="axes fraction", fontsize=8.2, color=RED,
                    arrowprops=dict(arrowstyle="->", color=RED, linewidth=0.9))
ax.set_xlabel(T("epoch"))
ax.set_ylabel(T("loss"))
ax.set_title(f"(b) {T('loss_main')}", fontsize=10.5)
ax.legend(fontsize=8.6, frameon=False)
style_axes(ax)

fig.suptitle(f"{T('figF')}: {MAIN_RUN['run_name']}", fontsize=12.5,
             fontweight="bold", y=1.02)
fig.tight_layout()
_figF = _summary_dir / "figF_loss_curves.png"
fig.savefig(_figF, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("已保存:", _figF)

## 附录：推理与提交（`*-pred.tsv` 导出）

**为什么把导出放在最后**：提交格式是本任务**最容易在最后一天翻车**的环节。
下面这份导出函数已经把所有已知的坑写成了断言——任何一条不满足都直接 `raise`，
而不是让它静默走到提交那天。

**格式规范（官方）**：

- 提交物是**一个 zip**，内含 4 个文件：`en-nn-pred.tsv` / `en-pv-pred.tsv` /
  `de-nn-pred.tsv` / `de-pv-pred.tsv`
- **无表头**，制表符分隔
- 名词复合词每行 **3** 列：`ContextID` ⇥ `修饰语预测` ⇥ `中心语预测`（**修饰语在前**）
- 小品词动词每行 **2** 列：`ContextID` ⇥ `单一预测`
- 测试集**不含** `*Avg` / `*Std` 列，载入逻辑必须容忍缺列
- 预测值必须落在 **[0, 5]**；输出**不得是常数**（否则指标 2 = nan）

In [ ]:

def export_pred(tsv_path, preds, lang, task):
    test = pd.read_csv(tsv_path, sep="\t")
    ids = test["ContextID"].astype(str)
    preds = np.asarray(preds, dtype=float)
    if preds.ndim == 1:
        preds = preds.reshape(-1, 1)

    if task == "nn":
        assert preds.shape[1] == 2, f"名词复合词需要 2 列（修饰语、中心语），收到 {preds.shape[1]}"
        body = pd.DataFrame({"ContextID": ids, "mod": preds[:, 0], "head": preds[:, 1]})
    else:
        assert preds.shape[1] == 1, f"小品词动词需要 1 列，收到 {preds.shape[1]}"
        body = pd.DataFrame({"ContextID": ids, "pred": preds[:, 0]})

    assert len(body) == len(test), "行数与测试集不一致"
    assert set(body["ContextID"]) == set(ids), "ContextID 集合与测试集不一致"
    _v = body.select_dtypes("number").to_numpy()
    assert np.isfinite(_v).all(), "预测里出现 NaN/inf"
    assert _v.min() >= 0.0 and _v.max() <= 5.0, f"预测超出 [0,5]：[{_v.min():.3f}, {_v.max():.3f}]"

    out = OUTPUT_ROOT / f"{lang}-{task}-pred.tsv"
    body.to_csv(out, sep="\t", index=False, header=False)
    return out






_exported = []
for _r in ALL_RUNS:
    for _n, _cn, _j in _r["eval_cols"]:
        if _cn != SUBSET_SPECS[_n]["cols"][0]:
            continue
        _sp = SUBSET_SPECS[_n]
        _m = _r["subset_ids"] == _n
        _cols = [_r["preds"][_m, jj] for jj in range(len(_sp["cols"]))]
        _tsv = OUTPUT_ROOT / f"_demo_test_{_n}.tsv"
        pd.DataFrame({"ContextID": [f"{i:05d}" for i in range(int(_m.sum()))],
                      "Context": _r["val_df"].loc[_m, "Context"].to_numpy()
                      }).to_csv(_tsv, sep="\t", index=False)
        _out = export_pred(_tsv, np.stack(_cols, axis=1), _sp["lang"], _sp["task"])
        _exported.append((_r["run_name"], _n, _out))
        print(f"[{_r['run_name']} / {_n}] -> {_out.name}")
print("\n→ 无表头、列数正确、值域 [0,5]、ContextID 原样透传。")
print("→ 正式提交时把 4 个 *-train.tsv 换成官方测试 TSV，文件名保持 en-nn-pred.tsv 等 4 个。")

## 结论与延伸

### 这份实验的运行前置条件

> 本文件是 **DeBERTa-v2-xlarge** 的专属**机制实验**，与另一个实验（DeBERTa / Qwen）
> 共用**同一份源码**（`tools/nbgen/cells_a/b/c.py` → `build_two.py` 生成），
> 数据处理、切分、指标、绘图、显存档位全部逐字相同，只靠 `MODEL_KEY` 分叉。
>
> **★ v4 的定位变了**：v1–v3 是「同配置换模型」的**对照实验**；
> v4 是**机制实验** —— DeBERTa 测「软标签对比学习」，Qwen2.5-1.5B-Instruct 测「不加回归头、
> 让模型自己预测分数」。两个实验**故意**在「读出方式 / 机制项 / 损失配方」上不同，
> 但所有**非机制项**（数据、切分、lr、轮数、有效 batch、warmup、Huber δ、指标实现、绘图）
> 仍然逐字相同 —— 这样「v3 → v4」的差异才能干净地归因到机制本身。
>
> **本文件验证的机制**：**软标签对比学习（表征的相似度对齐标签的相似度）**。

| 项 | 要求 | 说明 |
|---|---|---|
| Python | ≥ 3.10 | 本机实测 3.12.7 |
| torch | ≥ 2.4 | 本机实测 2.14.0 |
| transformers | ≥ 5.0 | **5.x 的 API 与 4.x 不同**：`eval_strategy`、`processing_class`、`warmup_steps`、`dtype=` 都是 5.x 写法；代码已做双兼容（签名探测自动切换） |
| datasets | ≥ 3.0 | 本机实测 3.6.0 |
| 网络 | 云端 GPU 可直连 `huggingface.co`；本机通常不通 | Step1 会**自动探测**官方源，不通就换 `hf-mirror.com`，下载时再兜一层重试。也可设 `COMPARTMENT_HF_MODE=official\|mirror\|cache` 强制指定 |
| **显存** | **单卡 ≥ 12GiB 可用**（T4 14.56GiB 即可） | 训练「档位」按 **参考参数量 1.55e9**（两条基线中较大者）+ 本机显存自动选，共 7 档且固定开销单调递减：t0 全参·fp32 AdamW·AMP(18 B/param) → t1 全参·8bit AdamW·AMP(12) → t2 全参·Adafactor·AMP(10.2) → **t3 全参·Adafactor·关 AMP(8.2)** → t4 顶部50%层·Adafactor·AMP(8.1) → t5 顶部50%层·无AMP(6.1) → t6 顶部25%层·无AMP(5.05)。T4 上落在 **t3，仍是全参微调（一个参数都没冻）**。Step8 会用真实模型真跑 2 步验证，OOM 自动降一档 |
| 内存 | ≥ 16GB | 主干 fp32 权重约 3.6GB / 6.2GB |
| 磁盘 | ≥ 10GB 可用 | 5 次运行只保留**联合运行**的权重（~3.6GB），训练用 checkpoint 训完即删 |
| 时间 | 5 次运行（4 个单子集 + 1 个联合） | **v3：全部 5 次运行都是 3 轮**（`EPOCHS=JOINT_EPOCHS=3`）；总样本·轮 = 0.8×总行数×(3+3)。**v1 实测**：DeBERTa 约 2.2 h、Qwen 约 4.6 h（v2 的 5 轮 joint 版本约 8.0–8.4 h，但没换来可测收益）。Step8 开跑前会打印**时长投影**，超预算会报警 |

### 显存够不够：先看这张表

全参微调 + AdamW 的**固定开销**（权重 fp32 4 B/param + 梯度 fp32 4 B/param
+ AdamW 的 m/v 8 B/param = **16 B/param**，再加上开混合精度时 AMP 的 fp16 权重副本 2 B/param
→ **18 B/param**）**与 batch 大小无关**，所以「降 batch / 加梯度累积」救不了它。
真正能降下来的只有三样：**优化器状态**、**AMP 的 fp16 副本**、**可训练参数的范围**。

| 模型 | 参数量 | fp32 AdamW+AMP | 8bit+AMP | Adafactor+AMP | Adafactor 无 AMP（t3） |
|---|---|---|---|---|---|
| `microsoft/deberta-v2-xlarge` | 885M | 14.8 GiB | 9.9 GiB | 8.4 GiB | 6.8 GiB ✓ |
| `Qwen/Qwen2.5-1.5B-Instruct` | 1544M | **25.9 GiB** | 17.3 GiB | 14.7 GiB ✗ | **11.8 GiB ✓** |

（表中为「固定开销」，不含激活；实测 DeBERTa 在 **t3** 档的**峰值 9.93 GiB**，
比固定开销 6.8 GiB 多出的 ~3.1 GiB 是激活 + 显存碎片 + CUDA context。）

**T4 只有 14.56 GiB**，所以在 T4 上能保住「全参微调」语义的档位是
**t3（Adafactor + 关 AMP，8.2 B/param ≈ 11.8 GiB）** —— 没有任何参数被冻结。

### 为什么「9B 都能全量微调，1.5B 却超了」

这两句话说的**不是同一件事**，差别在三个地方：

| | 9B 全参微调（业界常见） | 本项目 1.5B 全参微调 |
|---|---|---|
| 硬件 | 2 × A100 **80GB**（合计 160GB） | 2 × T4 **14.56GB**（合计 29.1GB） |
| 固定开销 | 9B × 18 B = **162GB**，靠 ZeRO-3 分片塞进 160GB（还要再省） | 1.5B × 18 B = **25.9GB**，单卡 14.56GB 装不下 |
| 是否分片 | 用 ZeRO-3 / FSDP 把参数切到多卡 | notebook 是**单进程**，每卡都要放完整一份 |

关键在于：**DDP（数据并行）不省显存** —— 每张卡仍要放一整份模型+梯度+优化器状态，
只是把数据分半（提速用）。只有 **ZeRO-3 / FSDP 参数分片**才会把 25.9GB 摊到两张卡
（13GB/卡），而那需要 `accelerate launch --num_processes 2`，
**在 papermill 的单进程里跑不了**；而且本 notebook 自定义了分组采样器
（让同一目标的例句落在同一个 batch），DDP 下若不改成分布式采样器，
两个进程会取到**同一批样本** → 训练结果悄悄错掉。

所以本 notebook 的选择是：**单卡 + 把固定开销从 18 B/param 降到 8.2 B/param
（Adafactor + 关 AMP）**，这样 1.5B 也能在 T4 上做**完整的全参微调**。

### 训练档位（tier）：★ v4.4 起**写死 t3（全参微调）**

> **v4.4 的变更**：档位不再由「显存探测自动选」决定，而是**写死 `FIT_MODE = "t3"`**。
> 起因：v4.3 的云端实跑里，自动降档把 Qwen 悄悄降到了 t5（只训顶部 50% 层），
> 而 DeBERTa 留在 t3 ⇒ 两条基线的**训练范围**一个全参、一个半参。
> ⇒ 降档不是「显存优化」，是**换实验**：t4/t5 会冻住一半主干，
>   而本版两个机制（DeBERTa 的软标签对比学习、Qwen 的无回归头读出）都要求主干整体可学。
> ⇒ 现在的规则：**每条基线各自写死 t3（全参）**；两条基线之间不再强制口径一致
>   （它们是两个独立的机制实验），但每条基线自己的口径全部记录在「口径清单」里。
> 下面这套档位表**仍然保留**：它是 `FIT_MODE="auto"` 时的选档逻辑，也是降档链的依据。

**显存模型（云端实跑校准过，务必照抄这份口径）**：

```
每参数开销 = 权重 fp32 4
           + （开 AMP 时）fp16 权重副本 2   ← 反向传播要用它算 grad_input，
                                               被 autograd 存到 backward 结束
           + 梯度 fp32 4 × 可训比例
           + 优化器状态 × 可训比例   AdamW 8 / 8bit AdamW 2 / Adafactor ≈0.2
```

> 实测校验：DeBERTa-xlarge（885M）在 t2 档的**实测峰值 9.93 GiB**，
> 按上式算 10.2 B/param × 885M = 8.4 GiB + 激活/碎片 ≈ 1.5 GiB → 9.9 GiB ✓ 对得上。
> 第一版估算漏了 AMP 的 2 B/param → Qwen-1.5B 少算 3.1 GiB → 以为装得下 → 实测 bs=1 也 OOM。

| 档位 | 含义 | optim | AMP | 可训层 | 固定开销 | 什么时候选它 |
|---|---|---|---|---|---|---|
| t0 | 全参微调 · fp32 AdamW | `adamw_torch` | 开 | 100% | 18 B/param | 40GB+ 的卡（A100 等） |
| t1 | 全参微调 · 8bit AdamW | `adamw_bnb_8bit` | 开 | 100% | 12 B/param | 装了 bitsandbytes 且 ≥24GB 卡 |
| t2 | 全参微调 · Adafactor | `adafactor` | 开 | 100% | 10.2 B/param | 16GB+ 的卡（L4 24GB 会选 t1） |
| **t3** | **全参微调 · Adafactor · 关 AMP** | `adafactor` | 关 | 100% | **8.2 B/param** | **T4 14.56GB 就是这一档**（仍全参微调，只是不用混合精度） |
| t4 | 顶部 50% 层 · Adafactor | `adafactor` | 开 | 50% | 8.1 B/param | 想要 AMP 速度、可以接受冻一半层时 |
| t5 | 顶部 50% 层 · Adafactor | `adafactor` | 关 | 50% | 6.1 B/param | 更省显存 |
| t6 | 顶部 25% 层 · Adafactor | `adafactor` | 关 | 25% | 5.05 B/param | 兜底 |

> ⚠️ **上表的前提是「单卡模式」**。若进程能看见 2 张卡，transformers 的 Trainer 会自动套
> `nn.DataParallel`，其反向传播要**再复制一整份梯度**（Qwen-1.5B 多 5.75 GiB）→ 必然 OOM。
> Step1 用 `CUDA_VISIBLE_DEVICES` 把进程限制成单卡，`build_training_args` 里再钉 `args._n_gpu = 1` 兜底。

**选档规则（仅 `FIT_MODE="auto"` 时启用）**：用**两条基线中较大者的参数量**
（`FIT_REF_PARAMS = 1.55e9`）作参考，配合本机显存选第一个装得下的档位
（判据用「micro-batch=1 的最小激活」，因为激活是 batch 的事、由探测阶段逐档降）。
若估算仍偏乐观，Step8 会用**真实模型真跑 2 步**验证，OOM 就自动降一档重来；
全部档位都失败时给出精确诊断（而不是一句 out of memory）。

> ★ **v4.4：默认 `FIT_MODE="t3"`，这条自动选档路径默认不启用。**
> 写死的好处是**档位不可能再悄悄漂移**；真跑不通时代码直接报错并给出诊断，
> 而不是偷偷降一档、让口径在无人察觉的情况下变掉。
>
> ★ 另一个同样关键的 v4.4 改动：**Qwen 的有效 batch 回到 8**（micro 8 × accum 1，与 v3 逐字相同）。
> 原因：梯度累积 ≥2 会让绑定 embedding 的稠密梯度在反向时**同时存在两份**
> （已有的 `.grad` + 新算出的临时缓冲）⇒ 实机 **+890.25 MiB**
> ⇒ t3 装不下（探针 8/4/2/1 全部 OOM 且内存不随 batch 下降）⇒ 被迫降到 t5。
> accum=1 时 autograd 直接「窃取」那块临时缓冲（全程只 1 份）⇒ 与 v3 一样装得下。

> t3 档为什么「关 AMP」：开 AMP 时 PyTorch 会把权重的 fp16 副本留到反向传播用，
> 多占 2 B/param。Qwen-1.5B 正是差这 2 B/param（3.1 GiB）而装不下 14.56 GiB 的卡。
> 关掉 AMP 就省出这 3.1 GiB，代价是纯 fp32 训练慢一些（T4 没有 TF32）——
> 但 T4 上 batch 很小、属于「发射受限」而非「算力受限」，实测慢得不多。

### 「两张 T4 能不能凑成 30G 用？」——不能，而且云端实跑发现：**两张卡反而是致命的**

**结论先说**：CUDA 显存**不能跨卡合并**。一个张量只能待在一张卡上，
「2 × 14.56 GiB」不会变成「一张 30 GiB 的卡」。

**而且本项目在云端实跑里发现了一件更糟的事**：只要进程能看见 2 张卡，
transformers 的 Trainer 就会**自动**把模型套上 `nn.DataParallel`：

```python
# transformers/trainer.py :: _wrap_model()
if self.args.n_gpu > 1 and not is_loaded_in_8bit and not is_loaded_in_4bit:
    model = nn.DataParallel(model)
```

`nn.DataParallel` 的反向传播要经过 `Broadcast` → `ReduceAddCoalesced`
→ `comm.reduce_add_coalesced(...)`，**额外分配并复制一整份梯度** →
**梯度显存直接翻倍**。Qwen-1.5B 就是多 5.75 GiB → 14.56 GiB 的卡必然 OOM。

云端日志里的证据非常干净：**t3 / t4 / t5 三个档位 × 8/4/2/1 四个 batch 全部 OOM**，
每次都死在同一个地方：

```
Tried to allocate 54.00 MiB ... Of the allocated memory 14.26 GiB is allocated by PyTorch
  堆栈: torch/nn/parallel/_functions.py → ReduceAddCoalesced → nccl.reduce
```

（14.26 GiB ≈ 权重 5.75 + 梯度 5.75 + 正在复制的第二份梯度 2.76 ✓ 数字闭环）

所以「用两张卡」在单进程里不但没省显存，反而**多花了 5.75 GiB**。
本项目的做法是：**用 `CUDA_VISIBLE_DEVICES` 把进程限制成只看见 1 张卡**
（Step1 已设，`build_training_args` 里再钉一次 `args._n_gpu = 1` 兜底），
让 Trainer 走单卡路径，梯度显存不再翻倍。

| 想做的事 | 能不能把两张卡的显存合起来用 | 说明 |
|---|---|---|
| **DDP（数据并行）** | **不能** | 每张卡各存**一整份**模型+梯度+优化器状态，只是把 batch 分半 → 提速，不省显存 |
| **nn.DataParallel（Trainer 自动套的）** | **不能，而且更糟** | 反向传播还要**再复制一份梯度** → 显存 ×2 |
| **ZeRO-3 / FSDP（参数分片）** | **能** | 每卡只放 1/2 → 这才是「29 GiB 一起用」 |
| **两条基线各占一张卡** | 不需要合并 | 两个独立实验各跑各的，墙钟时间直接减半 |

**为什么本 notebook 不直接上 ZeRO-3/FSDP**（三个实打实的原因）：

1. 需要**多进程**启动：`accelerate launch --num_processes 2` 或 `torchrun`。
   papermill / `jupyter nbconvert` 起的是**普通单进程**，不会自动开 DDP。
2. 本 notebook 自定义了 `TargetGroupedSampler`（让同一目标的例句落在同一个 batch，
   否则组内排序损失与组内 std 损失形同虚设）。DDP 下若不改成分布式采样器，
   **两个进程会取到同一批样本** → 训练结果悄悄错掉，而且**不报错**。
3. 改造要动的地方多（采样器、Trainer 子类、跨进程结果汇总），属于大改，
   而当前阻塞点（Qwen 装不下）用单卡 + 档位机制已经能解。

**所以现在用的是最省事的方案**：

```bash
COMPARTMENT_GPU=0   # DeBERTa 版（单卡，梯度显存不翻倍）
COMPARTMENT_GPU=1   # Qwen 版（同一台机器、两条基线各占一张卡，各自独立运行）
```

> ★ **v4.4 补充**：两张 T4 的**正确用法是「两条基线并行，各占一张」** ——
> 墙钟从 8.8 h 串行降到 5.74 h 并行，数值完全不受影响（同型号卡）。
> 而「让单次训练吃满两卡的显存」只有 ZeRO-3/FSDP 能做到，
> 它需要多进程（papermill 单进程跑不了）且会改变浮点求和顺序 ⇒ 与历史结果不再逐字可比。

两条基线各占一张卡同时跑，**两张卡的算力都用满了**，墙钟时间减半；
而「29 GiB 合成一张卡」这件事，等以后真的要训 7B/9B 时再用 FSDP 脚本形态来做。

### 多卡怎么用：先分清「提速」和「省显存」

| 用法 | 每卡上放什么 | 能降低单卡显存需求吗 | 能提速吗 |
|---|---|---|---|
| 单进程 | 全部在一张卡 | — | 基准 |
| **DDP（数据并行）** | **每卡各存一整份**模型+梯度+优化器状态，只是数据分半 | **不能**（每卡仍是完整那一份） | 能（约 1.8×） |
| **ZeRO-3 / FSDP（参数分片）** | 参数/梯度/优化器状态**切分**到多卡 | **能**（这才是用满多卡显存的办法） | 能 |
| 两条基线各占一张卡 | 两条基线是**互不相干的实验**，各占一卡同时跑 | 不需要（各跑各的） | 能（墙钟减半） |

**对 Qwen2.5-1.5B 具体意味着什么**：DDP **救不了**它 —— 每卡仍要装 23.0 GiB 的固定开销。
本 notebook 的做法是走 **t3 档**（全参微调 + Adafactor + 关 AMP = 8.2 B/param ≈ 11.8 GiB，单卡就够）；
真想用满 29 GiB 训更大的模型，则要上 ZeRO-3/FSDP 脚本形态（见上一节的三条理由）。

**注意**：papermill / `jupyter nbconvert` 起的是**普通单进程**，**不会自动开 DDP**。
真要用 DDP 得显式 `accelerate launch --num_processes 2 notebook.ipynb`，
而且必须把自定义采样器改成分布式采样器，否则两个进程会取到同一批样本。

### 训练时间够不够：开跑前就有投影

Step8 探测时会量出「每样本耗时」，据此投影 **5 次运行**（4 个单子集 + 1 个联合）
的总时长并打印；若超过 `TIME_BUDGET_H`（默认 11 小时，给评估与出图留 1 小时余量），
会给出三条可选做法。**它只报警、不自动改 EPOCHS** —— 自动改会让两条基线的
训练轮数不同，对照口径就废了。


### 图上的中文：云端可能没有中文字体

Kaggle 等云端镜像的默认字体（DejaVu Sans）**不含中文字形**，中文会静默变成方块「口口口」。
本 notebook 的处理是：先找系统字体 → 找不到就下载 Noto Sans CJK → 仍失败则**图注自动切英文**。
也可以用环境变量强制：

```bash
export COMPARTMENT_NO_CJK=1     # 强制英文图注（不下载 16MB 字体）
```

### 自动跑了哪些运行

`RUN_PLAN`（Step1）是唯一的开关：

| 运行 | 训练数据 | 覆盖的评测设置 | 用途 |
|---|---|---|---|
| `single / en-nn` | 只有 en-nn | en-nn·ModAvg、en-nn·HeadAvg | 单子集分析 |
| `single / de-nn` | 只有 de-nn | de-nn·ModAvg、de-nn·HeadAvg | 单子集分析 |
| `single / en-pv` | 只有 en-pv | en-pv·Avg | 单子集分析 |
| `single / de-pv` | 只有 de-pv | de-pv·Avg | 单子集分析 |
| `joint` | 四个子集拼起来 | 全部 6 个 | **一个模型覆盖 6 个评测设置**，正式提交用 |

**联合模式的三处技术处理**（这也是 single 与 joint 能共用一套代码的原因）：

1. **统一输出维度**：四个子集都用 2 列输出（复合词：修饰语/中心语；小品词动词：只有第 1 列有标签，
   第 2 列用 `loss_mask` 屏蔽）。模型结构、损失、指标、绘图因此完全不用判断子集类型。
2. **分组键带子集**：`group_index` 编的是 **(子集, 目标)** 而不是单纯的目标。
   指标 2 的组内 std 必须按这个分组算，否则不同子集里同名目标会被串成一组。
3. **切分在子集内部做**：每个子集各自按「目标级互斥 + 先分箱后分层抽样」切分，
   再把四个子集的 train/val 拼起来。这样每个子集的验证集难度分布都保持正确。

**想少跑几次**：直接注释掉 `RUN_PLAN` 里对应的行，其余代码一行不用动。

### 云端跑法（推荐命令）

```bash
# 1) 拉代码与数据（数据放在仓库的 train/ 下）
git clone <your-repo> ComPartMENT && cd ComPartMENT

# 2) 环境变量（都可省略，notebook 会自动探测）
export COMPARTMENT_TRAIN_DIR=/kaggle/input/datasets/kintsugi0v0/compartment-data/train
export COMPARTMENT_OUTPUT=/kaggle/working/outputs
export COMPARTMENT_HF_MODE=auto        # auto=先探测官方源，不通换镜像
export TOKENIZERS_PARALLELISM=false

# 3) 依赖
pip install "transformers>=5" "huggingface-hub<2.0" "datasets<4" "pyarrow<18" \
            scikit-learn pandas matplotlib seaborn scipy jupyter nbconvert

# 4) 先用 30 秒跑通整条链路（2 层随机主干，不下权重）
#    把 Step1 的 SMOKE 改成 True
jupyter nbconvert --to notebook --execute \
  "v4/ComPartMENT(DeBERTa-v2-xlarge-软标签对比学习)/ComPartMENT_DeBERTa-v2-xlarge_软标签对比学习.ipynb" \
  --ExecutePreprocessor.timeout=1800

# 5) 正式跑（把 Step1 的 SMOKE 改回 False）
nohup jupyter nbconvert --to notebook --execute \
  "v4/ComPartMENT(DeBERTa-v2-xlarge-软标签对比学习)/ComPartMENT_DeBERTa-v2-xlarge_软标签对比学习.ipynb" \
  --ExecutePreprocessor.timeout=7200 > train.log 2>&1 &
```

**数据路径不用改代码**：notebook 会按「环境变量 `COMPARTMENT_TRAIN_DIR` → Kaggle 挂载目录
`/kaggle/input/datasets/kintsugi0v0/compartment-data/train` → 仓库内 `train/`」的顺序自动探测，
命中即用。Kaggle 上直接跑就是挂载目录，不用改任何一行。

**云端唯一必须改的地方**（Step1）：

| 位置 | 改什么 | 为什么 |
|---|---|---|
| `SMOKE` | `False` | `True` 是 30 秒冒烟模式，不产出可用模型 |
| `HF_MODE` | 一般不用改 | `auto` 会先探测官方源；海外云端直通用它最稳 |

### 显存不够时的降级顺序（代价从小到大）

Step8 的**训练档位机制**已经把这套顺序自动化了（见上文的档位表），
这里只列「为什么是这个顺序」：

1. **降 `per_device_train_batch_size`，用 `gradient_accumulation_steps` 把有效 batch 补回去**
   —— 数学等价、几乎无损，第一选择（显存探测会自动做这件事）
   —— ⚠️ 但只省**激活**显存，救不了「权重+梯度+优化器状态」这块固定开销
2. **开 `gradient_checkpointing`**（记得同时 `model.config.use_cache = False`）
   —— 激活降到约 1/N，慢约 30%
3. **换优化器**（`adafactor` 把 m/v 从 8 B/param 压到 ≈0.2；`adamw_bnb_8bit` 压到 2）
   —— 唯一能显著降低**固定开销**的办法
4. **关掉混合精度**（AMP 的 fp16 权重副本占 2 B/param，Qwen-1.5B 就是差这 3.1 GiB）
   —— 代价是纯 fp32 训练慢一些
5. **只训顶部若干层**（`apply_fit_scope`）—— 会冻结底层与 embedding，是**方法层面的改变**，
   必须记进口径清单（本版两条基线各自独立，改动只影响本条基线）
6. **降 `MAX_LENGTH`** —— 最有效，但要改全局配置并**从头重跑**（分词在 Step3 已固化）
7. **换更小的同族模型** —— `microsoft/deberta-v3-base` / `Qwen2.5-0.5B-Instruct`，
   管线一行不用改

### 三个判断「实验有没有做对」的指标

1. **梯度流验证必须通过**：回归头与主干的梯度绝对值和都 > 0，且 loss 不是 NaN。
   否则模型可能训了十几分钟而参数纹丝不动。
2. **指标 2 不能是 `nan`**：组内预测 std 恒为 0 会让相关系数未定义。
   图 C 上如果点全挤在水平线上，就说明模型没有真正对语境敏感。
3. **训练后必须显著优于随机初始化**（参考对照区）：「训练后模型」的指标 1 要明显高于
   「随机初始化」，否则模型没有学到任何东西（此时优先查：标签列是否对齐、
   输入模板是否恒定、学习率是否过大）。

### 后续可以做什么（对应本项目的建模决策）

| 方向 | 做法 | 开关位置 |
|---|---|---|
| 调组内排序的比重（指标 1） | v3.1 已开启（占比 8%，四个子集都有）；调大更重「组内顺序」、调小更重「绝对值」 | `SUBSET_LOSS_PROFILE[子集]["order"]` |
| 调指标 3 的比重 | v3.1 已开启（占比 5%，四个子集都有） | `SUBSET_LOSS_PROFILE[子集]["mse"]` |
| 调防塌陷的比重（指标 2） | v3.1 已开启（占比 6%），只在下界被突破时发力 | `SUBSET_LOSS_PROFILE[子集]["guard"]` |
| 调指标 2 的下界 κ | nn 用 0.15（只挡真塌陷）、pv 用 0.50（主动顶住离散度） | `SUBSET_LOSS_PROFILE[子集]["kappa"]` |
| 开关「跨目标 std 序」 | v3.1 **只在 pv 开**（nn 的指标 2 天花板只有 0.00–0.08） | `SUBSET_LOSS_PROFILE[子集]["xrank"]` |
| 关掉自适应标定 | 改为固定权重（用于消融「自动标定」本身有没有用） | `AUTO_BALANCE=False` |
| **恢复「组内 std 等值匹配」** | ⚠️ **不建议**：人工 std 掺着不可复现的标注噪声（v2 实测零收益且慢 21–40%） | 见 Step7 的说明 |
| 换模型规模 | `deberta-v2-xlarge` → `deberta-v3-xlarge`；`Qwen2.5-1.5B` → `7B` | `MODEL_REGISTRY` |
| 减少运行次数 | 只跑 joint 一次即可覆盖全部 6 个评测设置 | `RUN_PLAN` |
| 改进定位 | 德语复合词必须预切分（不能字符串切片还原），切分版与未切分版 ensemble | Step3 的 `locate_target` |
| 帕累托前沿 | 把输出缩放系数当旋钮，在 dev 上画「指标1 ρ vs 指标2 ρ」的前沿，而不是用单一指标选点 | Step9 之后 |

### 复现命令

```bash
cd ComPartMENT
pip install "transformers>=5" "huggingface-hub<2.0" "datasets<4" "pyarrow<18" scikit-learn pandas matplotlib seaborn scipy
jupyter nbconvert --to notebook --execute \
  "v4/ComPartMENT(DeBERTa-v2-xlarge-软标签对比学习)/ComPartMENT_DeBERTa-v2-xlarge_软标签对比学习.ipynb"
```